# CSC 4792 Data Mining and Warehousing

## Kabwe Municipal Council Dataset

### Project Information

- **Course:** CSC 4792 Data Mining and Warehousing
- **Academic Year:** 2025/26
- **Council:** Kabwe Municipal Council
- **Country:** Zambia
- **Official Website:** https://www.kabwecouncil.gov.zm

### Objective

The objective of this project is to collect, extract, clean,
transform, and curate information associated with Kabwe Municipal
Council from its official digital sources.

The resulting datasets will be stored as pipe-delimited CSV files
and published on Kaggle.

## 1. Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
from urllib.parse import urljoin
import fitz
import pdfplumber

In [4]:
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Data Source

In [49]:
BASE_URL = "https://www.kabwecouncil.gov.zm"

print(BASE_URL)

https://www.kabwecouncil.gov.zm


In [50]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

response = requests.get(
    BASE_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

Status code: 200


In [51]:
soup = BeautifulSoup(response.text, "html.parser")

In [8]:
links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(BASE_URL, link["href"])

    links.append({
        "text": text,
        "url": url
    })

links_df = pd.DataFrame(links)

links_df.head(20)

,text,url
0,,https://www.kabwecouncil.gov.zm#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169
5,Who we are,https://www.kabwecouncil.gov.zm/?page_id=118
6,Departments,https://www.kabwecouncil.gov.zm/?page_id=770
7,office of the the town clerk,https://www.kabwecouncil.gov.zm/?page_id=2634
8,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
9,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640


In [9]:
print("Number of links:", len(links_df))

Number of links: 122


In [48]:
keywords = [
    "cdf",
    "project",
    "budget",
    "financial",
    "report",
    "development",
    "idp",
    "ward",
    "revenue",
    "procurement",
    "minutes"
]

pattern = "|".join(keywords)

relevant_links = links_df[
    links_df["text"].str.contains(
        pattern,
        case=False,
        na=False
    )
]

relevant_links

,text,url
27,CDF,https://www.kabwecouncil.gov.zm/?page_id=2542
28,CDF GUIDLINES,https://www.kabwecouncil.gov.zm/?page_id=2579
29,CDF branding guidelines,https://www.kabwecouncil.gov.zm/?page_id=3674
66,CDF,https://www.kabwecouncil.gov.zm/?page_id=2542
67,CDF GUIDLINES,https://www.kabwecouncil.gov.zm/?page_id=2579
68,CDF branding guidelines,https://www.kabwecouncil.gov.zm/?page_id=3674
91,CDF Skills Bursaries Applicants,https://www.katetecouncil.gov.zm/wp-content/uploads/2023/11/SKILLS-DEVELOPME...
108,CDF PROJECT MONITORING BY KABWE MUNICIPAL COUNCIL MANAGEMENT TEAM,https://www.kabwecouncil.gov.zm/?p=4386
115,Ministry of Local Government and Rural Development,https://www.mlgrd.gov.zm/


In [11]:
cdf_columns = [
    "project_id",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "sector",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url"
]

cdf_projects = pd.DataFrame(columns=cdf_columns)

cdf_projects

,project_id,year,constituency,project_name,project_description,project_type,ward,project_site,sector,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url


In [13]:
import requests

pdf_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

pdf_response = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", pdf_response.status_code)
print("Content-Type:", pdf_response.headers.get("Content-Type"))
print("File size:", len(pdf_response.content), "bytes")
print("First 20 bytes:", pdf_response.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 93259 bytes
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


In [14]:
pdf_path = "../data/raw/2024_bwacha_cdf_projects.pdf"

with open(pdf_path, "wb") as file:
    file.write(pdf_response.content)

print("PDF downloaded successfully.")

PDF downloaded successfully.


In [16]:
import requests
import pdfplumber
import pandas as pd

pdf_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

response = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")
print("Content-Type:", response.headers.get("Content-Type"))
print("First 20 bytes:", response.content[:20])

Status code: 200
File size: 93259 bytes
Content-Type: application/pdf
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


In [17]:
pdf_path = "../data/raw/2024_bwacha_cdf_projects.pdf"

with open(pdf_path, "wb") as file:
    file.write(response.content)

print("PDF saved to:", pdf_path)

PDF saved to: ../data/raw/2024_bwacha_cdf_projects.pdf


In [18]:
with pdfplumber.open(pdf_path) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages):
        tables = page.extract_tables()

        print(
            f"Page {page_number + 1}: "
            f"{len(tables)} table(s) found"
        )

Number of pages: 5
Page 1: 1 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 1 table(s) found


In [19]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()

    table = tables[0]

    for row in table[:10]:
        print(row)

['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', None, None, None, None, None, None, None, None, None]
['No.', 'Project Name', 'Project Description', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None]
['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', 

In [20]:
df = pd.DataFrame(
    table[1:],
    columns=table[0]
)

df.head()

,2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,No.,Project Name,Project Description,Ward,Project\nSite/Location,Application\nAmount,Engineers'\nEstimates,Approved\nAmount,Contract\nAmount,Status
1,Education,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
3,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
4,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,


In [21]:
print(df.shape)
print(df.columns.tolist())

(8, 10)
['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', nan, nan, nan, nan, nan, nan, nan, nan, nan]


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 10 columns):
 #   Column                                                                                Non-Null Count  Dtype
---  ------                                                                                --------------  -----
 0   2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY
KABWE MUNICIPAL COUNCIL  8 non-null      str  
 1   nan                                                                                   7 non-null      str  
 2   nan                                                                                   7 non-null      str  
 3   nan                                                                                   7 non-null      str  
 4   nan                                                                                   7 non-null      str  
 5   nan                                                                                   7 non-null      str  
 6   n

In [23]:
for row in table[:10]:
    print(row)

['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', None, None, None, None, None, None, None, None, None]
['No.', 'Project Name', 'Project Description', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None]
['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', 

In [24]:
all_rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    all_rows.append(row)

print("Total rows extracted:", len(all_rows))

Total rows extracted: 47


In [25]:
project_rows = []

for row in all_rows:
    if row[0] is not None and str(row[0]).strip().isdigit():
        project_rows.append(row)

print("Number of project rows:", len(project_rows))

Number of project rows: 36


In [26]:
for row in project_rows[:5]:
    print(row)

['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['4', 'Construction of 1X3 Classroom\nBlock at Mine Primary School', 'Construction of 1X3\nClassroom Block at Mine\nPrimary School -\nMutwewansofu zone', 'Kangomba', 'Mine Primary\nSchool', '', '', '', '', '']
['5', 'Repairing of the Mono-Pump,\nConstruction of Toilets for Pre-\nSchool Pupils, Procurement of\nDesks at Mary Chidgey\nCommunity Primary School', 'Repairing

In [27]:
columns = [
    "project_number",
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status"
]

df = pd.DataFrame(project_rows, columns=columns)

df.head()

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,


In [28]:
print(df.shape)
print(df.columns.tolist())

(36, 10)
['project_number', 'project_name', 'project_description', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']


In [29]:
for column in df.columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df.head()

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,,,,,
1,2,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
2,3,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
3,4,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,,,,,
4,5,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,,,,,


In [30]:
print(df.loc[0, "project_name"])
print(df.loc[0, "project_description"])

Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward
Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone


In [31]:
df.insert(1, "year", 2024)
df.insert(2, "constituency", "Bwacha")

In [32]:
df.head()

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,,,,,
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,,,,,
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,,,,,


In [33]:
df["source_url"] = pdf_url
df["source_document"] = "2024 Bwacha CDF Community Projects Submission"

In [34]:
pd.set_option("display.max_colwidth", 100)

df.head(10)

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Muwowo East,Mukobeko Correctional Day Secondary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   project_number       36 non-null     str  
 1   year                 36 non-null     int64
 2   constituency         36 non-null     str  
 3   project_name         36 non-null     str  
 4   project_description  36 non-null     str  
 5   ward                 36 non-null     str  
 6   project_site         36 non-null     str  
 7   application_amount   36 non-null     str  
 8   engineers_estimate   36 non-null     str  
 9   approved_amount      36 non-null     str  
 10  contract_amount      36 non-null     str  
 11  status               36 non-null     str  
 12  source_url           36 non-null     str  
 13  source_document      36 non-null     str  
dtypes: int64(1), str(13)
memory usage: 4.1 KB


In [36]:
df["ward"].value_counts()

ward
Kangomba        8
Kawama          8
Chimaniman i    6
Bwacha          4
Muwowo East     2
Munyama         2
Chililalila     2
Chinyama        2
Ngungu          2
Name: count, dtype: int64

In [37]:
df["status"].value_counts(dropna=False)

status
    36
Name: count, dtype: int64

In [38]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    print(f"\n--- {column} ---")
    print(df[column].unique()[:20])


--- application_amount ---
<StringArray>
['']
Length: 1, dtype: str

--- engineers_estimate ---
<StringArray>
['']
Length: 1, dtype: str

--- approved_amount ---
<StringArray>
['']
Length: 1, dtype: str

--- contract_amount ---
<StringArray>
['']
Length: 1, dtype: str


In [39]:
def clean_amount(value):
    value = str(value).strip()

    # Treat blank values as missing
    if value == "" or value.lower() in ["nan", "none", "n/a", "na", "-"]:
        return pd.NA

    # Remove currency symbols, commas and other non-numeric characters
    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "":
        return pd.NA

    return float(value)

In [40]:
for column in financial_columns:
    df[column] = df[column].apply(clean_amount)

In [41]:
df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


In [42]:
df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


In [43]:
df[financial_columns].dtypes

application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object

In [44]:
df[financial_columns].isna().sum()

application_amount    36
engineers_estimate    36
approved_amount       36
contract_amount       36
dtype: int64

In [45]:
text_columns = [
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "status"
]

for column in text_columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [46]:
df.head(10)

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [47]:
duplicates = df.duplicated(
    subset=[
        "project_name",
        "ward",
        "project_site"
    ]
)

print("Duplicate rows:", duplicates.sum())

Duplicate rows: 1


In [48]:
print(df["project_number"].unique())

<StringArray>
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
Length: 12, dtype: str


In [49]:
print("Total projects:", len(df))

Total projects: 36


In [50]:
processed_path = "../data/processed/kabwe_2024_bwacha_cdf_projects.csv"

df.to_csv(
    processed_path,
    sep="|",
    index=False
)

print("Saved:", processed_path)

Saved: ../data/processed/kabwe_2024_bwacha_cdf_projects.csv


In [51]:
test_df = pd.read_csv(
    processed_path,
    sep="|"
)

print("Rows:", len(test_df))
print("Columns:", len(test_df.columns))

test_df.head()

Rows: 36
Columns: 14


,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [53]:
pdf_url_central = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Community-Projects-Kabwe-Central-received.pdf"
)

response_central = requests.get(
    pdf_url_central,
    timeout=60,
    verify=False
)

print("Status code:", response_central.status_code)
print("Content-Type:", response_central.headers.get("Content-Type"))
print("File size:", len(response_central.content), "bytes")
print("First 20 bytes:", response_central.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 96930 bytes
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


In [54]:
pdf_path_central = (
    "../data/raw/"
    "2024_kabwe_central_cdf_projects.pdf"
)

with open(pdf_path_central, "wb") as file:
    file.write(response_central.content)

print("Saved:", pdf_path_central)

Saved: ../data/raw/2024_kabwe_central_cdf_projects.pdf


In [55]:
with pdfplumber.open(pdf_path_central) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        print(
            f"Page {page_number}: "
            f"{len(tables)} table(s) found"
        )

Number of pages: 2
Page 1: 1 table(s) found
Page 2: 3 table(s) found


In [56]:
with pdfplumber.open(pdf_path_central) as pdf:
    table = pdf.pages[0].extract_tables()[0]

    for row in table[:10]:
        print(row)

['No.', 'Project Name', 'Project Description', 'Type of\nProject', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None, None]
['1', 'Construction of Secondary\nSchool 1X4 Classroom\nBlock', 'Construction of Secondary\nSchool 1X4 Classroom\nBlock', 'Construction', 'Luangwa', 'Kabwe Trust\nSecondary School', '', '', '', '', '']
['2', 'Construction of 1X3\nClassroom Block', 'Construction of 1X3\nClassroom Block', 'Construction', 'Luangwa', 'Kabwe Central\nHospital Special\nSchool Community', '', '', '', '', '']
['3', 'Construction of 1X2\nClassroom Block', 'Construction of 1X2\nClassroom Block', 'Construction', 'Luangwa', 'Kabwe Trust Primary\nSchool', '', '', '', '', '']
['4', 'Construction of 1X3\nClassroom Block', 'Construction of 1X3\nClassroom Block', 'Construction', 'Mpima', 'Mpima Dairy Scheme', '', '', '', '', '']
['5', 'Construction of

In [57]:
central_rows = []

with pdfplumber.open(pdf_path_central) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    # Keep only rows whose first column is a project number
                    if (
                        row[0] is not None
                        and str(row[0]).strip().isdigit()
                    ):
                        central_rows.append(row)

print("Kabwe Central project rows:", len(central_rows))

Kabwe Central project rows: 43


In [58]:
central_columns = [
    "project_number",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status"
]

central_df = pd.DataFrame(
    central_rows,
    columns=central_columns
)

central_df.head()

,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction,Luangwa,Kabwe Trust\nSecondary School,,,,,
1,2,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Luangwa,Kabwe Central\nHospital Special\nSchool Community,,,,,
2,3,Construction of 1X2\nClassroom Block,Construction of 1X2\nClassroom Block,Construction,Luangwa,Kabwe Trust Primary\nSchool,,,,,
3,4,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,
4,5,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima C,,,,,


In [59]:
print("Rows:", len(central_df))
print("Columns:", len(central_df.columns))
print(central_df.columns.tolist())

Rows: 43
Columns: 11
['project_number', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']


In [60]:
central_text_columns = [
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "status"
]

for column in central_text_columns:
    central_df[column] = (
        central_df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [61]:
central_df.insert(1, "year", 2024)
central_df.insert(2, "constituency", "Kabwe Central")

In [62]:
central_df["source_url"] = pdf_url_central
central_df["source_document"] = (
    "2024 Kabwe Central CDF Community Projects Submission"
)

In [63]:
central_df.head()

,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Kabwe Central,Construction of Secondary School 1X4 Classroom Block,Construction of Secondary School 1X4 Classroom Block,Construction,Luangwa,Kabwe Trust Secondary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission
1,2,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Luangwa,Kabwe Central Hospital Special School Community,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission
2,3,2024,Kabwe Central,Construction of 1X2 Classroom Block,Construction of 1X2 Classroom Block,Construction,Luangwa,Kabwe Trust Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission
3,4,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission
4,5,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima C,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission


In [64]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    central_df[column] = central_df[column].apply(
        clean_amount
    )

In [65]:
central_df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


In [66]:
central_df[financial_columns].isna().sum()

application_amount    43
engineers_estimate    43
approved_amount       43
contract_amount       43
dtype: int64

In [67]:
print("Bwacha columns:")
print(df.columns.tolist())

print("\nKabwe Central columns:")
print(central_df.columns.tolist())

Bwacha columns:
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']

Kabwe Central columns:
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']


In [68]:
df["project_type"] = pd.NA

In [69]:
central_columns_final = [
    "project_number",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url",
    "source_document"
]

df = df[central_columns_final]
central_df = central_df[central_columns_final]

In [70]:
kabwe_cdf_projects = pd.concat(
    [df, central_df],
    ignore_index=True
)

print(
    "Total Kabwe CDF projects:",
    len(kabwe_cdf_projects)
)

Total Kabwe CDF projects: 79


In [71]:
kabwe_cdf_projects["constituency"].value_counts()

constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

In [72]:
output_path = (
    "../data/processed/"
    "kabwe_cdf_projects_2024.csv"
)

kabwe_cdf_projects.to_csv(
    output_path,
    sep="|",
    index=False
)

print("Saved:", output_path)

Saved: ../data/processed/kabwe_cdf_projects_2024.csv


In [73]:
kabwe_cdf_projects.info()

<class 'pandas.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   project_number       79 non-null     str   
 1   year                 79 non-null     int64 
 2   constituency         79 non-null     str   
 3   project_name         79 non-null     str   
 4   project_description  79 non-null     str   
 5   project_type         43 non-null     object
 6   ward                 79 non-null     str   
 7   project_site         79 non-null     str   
 8   application_amount   0 non-null      object
 9   engineers_estimate   0 non-null      object
 10  approved_amount      0 non-null      object
 11  contract_amount      0 non-null      object
 12  status               79 non-null     str   
 13  source_url           79 non-null     str   
 14  source_document      79 non-null     str   
dtypes: int64(1), object(5), str(9)
memory usage: 9.4+ KB


In [74]:
print("Rows:", len(kabwe_cdf_projects))
print("Columns:", len(kabwe_cdf_projects.columns))

Rows: 79
Columns: 15


In [75]:
missing = kabwe_cdf_projects.isna().sum()

print(missing)

project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            0
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                  0
source_url              0
source_document         0
dtype: int64


In [76]:
missing_percentage = (
    kabwe_cdf_projects.isna().mean() * 100
).round(2)

print(missing_percentage)

project_number           0.00
year                     0.00
constituency             0.00
project_name             0.00
project_description      0.00
project_type            45.57
ward                     0.00
project_site             0.00
application_amount     100.00
engineers_estimate     100.00
approved_amount        100.00
contract_amount        100.00
status                   0.00
source_url               0.00
source_document          0.00
dtype: float64


In [77]:
empty_values = (
    kabwe_cdf_projects
    .astype(str)
    .apply(lambda column: column.str.strip().eq("").sum())
)

print(empty_values)

project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type            0
ward                    0
project_site            1
application_amount      0
engineers_estimate      0
approved_amount         0
contract_amount         0
status                 79
source_url              0
source_document         0
dtype: int64


In [78]:
duplicate_count = kabwe_cdf_projects.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 0


In [79]:
project_duplicates = kabwe_cdf_projects.duplicated(
    subset=[
        "year",
        "constituency",
        "project_name",
        "ward",
        "project_site"
    ],
    keep=False
)

print(
    "Potential duplicate project records:",
    project_duplicates.sum()
)

Potential duplicate project records: 2


In [80]:
kabwe_cdf_projects[
    project_duplicates
].sort_values(
    ["constituency", "project_name"]
)

,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [81]:
print(
    kabwe_cdf_projects["year"].value_counts()
)

year
2024    79
Name: count, dtype: int64


In [82]:
print(
    kabwe_cdf_projects["constituency"].value_counts()
)

constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64


In [83]:
print(
    kabwe_cdf_projects[
        financial_columns
    ].dtypes
)

application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object


In [84]:
kabwe_cdf_projects[
    financial_columns
].describe()

,application_amount,engineers_estimate,approved_amount,contract_amount
count,0,0,0,0
unique,0,0,0,0
top,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN


In [85]:
for column in financial_columns:
    negative_values = (
        kabwe_cdf_projects[column] < 0
    ).sum()

    print(
        column,
        "negative values:",
        negative_values
    )

application_amount negative values: 0
engineers_estimate negative values: 0
approved_amount negative values: 0
contract_amount negative values: 0


In [86]:
print(
    kabwe_cdf_projects[
        "project_name"
    ].head(20).to_string(index=False)
)

                            Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward
                                       Construction of 1X4 Classroom Block at Kagomba Primary School
                                       Construction of 1X4 Classroom Block at Kagomba Primary School
                                          Construction of 1X3 Classroom Block at Mine Primary School
Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks ...
                                                     Construction of a Primary School in Kawama ward
                                    Construction of 10 Teachers Houses at Chitakata Community School
                   Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School
                                                               Construction of Youth Resource Centre
                                     Construction of a School Hall at Rapheal Kombe Seconda

In [87]:
quality_report = pd.DataFrame({
    "column": kabwe_cdf_projects.columns,
    "data_type": [
        str(dtype)
        for dtype in kabwe_cdf_projects.dtypes
    ],
    "missing_values": [
        kabwe_cdf_projects[column].isna().sum()
        for column in kabwe_cdf_projects.columns
    ],
    "unique_values": [
        kabwe_cdf_projects[column].nunique()
        for column in kabwe_cdf_projects.columns
    ]
})

quality_report

,column,data_type,missing_values,unique_values
0,project_number,str,0,26
1,year,int64,0,1
2,constituency,str,0,2
3,project_name,str,0,75
4,project_description,str,0,75
5,project_type,object,36,8
6,ward,str,0,25
7,project_site,str,0,51
8,application_amount,object,79,0
9,engineers_estimate,object,79,0


In [4]:
import requests

In [7]:
import requests
import urllib3

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

pdf_url_2025_central = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/08/"
    "Proposed-2025-CDF-projects.pdf"
)

response_2025_central = requests.get(
    pdf_url_2025_central,
    timeout=60,
    verify=False
)

print("Status code:", response_2025_central.status_code)
print("Content-Type:", response_2025_central.headers.get("Content-Type"))
print("File size:", len(response_2025_central.content), "bytes")
print("First 20 bytes:", response_2025_central.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 2017833 bytes
First 20 bytes: b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n11 0 '


In [8]:
pdf_path_2025_central = (
    "../data/raw/"
    "2025_proposed_kabwe_central_cdf_projects.pdf"
)

with open(pdf_path_2025_central, "wb") as file:
    file.write(response_2025_central.content)

print("Saved:", pdf_path_2025_central)

Saved: ../data/raw/2025_proposed_kabwe_central_cdf_projects.pdf


In [9]:
pdf_path_2025_central = (
    "../data/raw/"
    "2025_proposed_kabwe_central_cdf_projects.pdf"
)

with open(pdf_path_2025_central, "wb") as file:
    file.write(response_2025_central.content)

print("Saved:", pdf_path_2025_central)

Saved: ../data/raw/2025_proposed_kabwe_central_cdf_projects.pdf


In [12]:
import pdfplumber

In [14]:
with pdfplumber.open(pdf_path_2025_central) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()

        print(
            f"Page {page_number}:",
            "TEXT FOUND" if text else "NO TEXT"
        )

Page 1: NO TEXT
Page 2: NO TEXT
Page 3: NO TEXT
Page 4: NO TEXT
Page 5: NO TEXT


In [15]:
import fitz

document = fitz.open(pdf_path_2025_central)

for page_number, page in enumerate(document, start=1):
    text = page.get_text("text")

    print(
        f"Page {page_number}:",
        len(text),
        "characters"
    )

Page 1: 0 characters
Page 2: 0 characters
Page 3: 0 characters
Page 4: 0 characters
Page 5: 0 characters


In [17]:
!apt-get update -qq
!apt-get install -y tesseract-ocr
!pip install pytesseract pdf2image

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.



   -------------------- ------------------- 1/2 [pdf2image]
   ---------------------------------------- 2/2 [pdf2image]



In [18]:
import pytesseract

print(pytesseract.get_tesseract_version())

TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

In [19]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print(pytesseract.get_tesseract_version())

5.5.3.20260724


In [20]:
from pdf2image import convert_from_path

print("pdf2image is ready")

pdf2image is ready


In [23]:
from pdf2image import convert_from_path

poppler_path = r"C:\poppler-26.07.0\Library\bin"

pages = convert_from_path(
    pdf_path_2025_central,
    dpi=300,
    poppler_path=poppler_path
)

print("Pages converted:", len(pages))

Pages converted: 5


In [24]:
page_text = pytesseract.image_to_string(
    pages[0],
    config="--psm 6"
)

print(page_text[:5000])

_
CDF 2025 PROPOSED COMMUNITY PROJECT KABWE CENTRAL CONSTITUENCY ©
©
NO | NAME OF COMMUNITY PROJECT APPLIED FORSHORTLUSTED; WARD SECTOR COMMENT O
1 Proposed Construction of a standard Maternity Annex at Mpima 5
Mpima health center
.
— =
Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved =
Lukanga Secondary School i
o
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
Kasanda Malombe Secondary School Oo
O
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
Water reticulated Systems with lockable kiosks in Waya Sanitation
communities
Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved
Water reticulated Systems at Kamushanga Market Shelter Sanitation
Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved
Water reticulated Systems at C-gate Priamary School
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Appro

In [25]:
ocr_path = (
    "../data/raw/"
    "2025_kabwe_central_cdf_ocr.txt"
)

with open(ocr_path, "w", encoding="utf-8") as file:
    file.write(page_text)

print("OCR text saved:", ocr_path)

OCR text saved: ../data/raw/2025_kabwe_central_cdf_ocr.txt


In [26]:
print(page_text[:5000])

_
CDF 2025 PROPOSED COMMUNITY PROJECT KABWE CENTRAL CONSTITUENCY ©
©
NO | NAME OF COMMUNITY PROJECT APPLIED FORSHORTLUSTED; WARD SECTOR COMMENT O
1 Proposed Construction of a standard Maternity Annex at Mpima 5
Mpima health center
.
— =
Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved =
Lukanga Secondary School i
o
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
Kasanda Malombe Secondary School Oo
O
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
Water reticulated Systems with lockable kiosks in Waya Sanitation
communities
Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved
Water reticulated Systems at Kamushanga Market Shelter Sanitation
Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved
Water reticulated Systems at C-gate Priamary School
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Appro

In [27]:
all_ocr_text = []

for page_number, page in enumerate(pages, start=1):
    text = pytesseract.image_to_string(
        page,
        config="--psm 6"
    )

    all_ocr_text.append(text)

    print(f"Page {page_number} OCR complete")

Page 1 OCR complete
Page 2 OCR complete
Page 3 OCR complete
Page 4 OCR complete
Page 5 OCR complete


In [28]:
full_ocr_text = "\n".join(all_ocr_text)

print(full_ocr_text[:10000])

_
CDF 2025 PROPOSED COMMUNITY PROJECT KABWE CENTRAL CONSTITUENCY ©
©
NO | NAME OF COMMUNITY PROJECT APPLIED FORSHORTLUSTED; WARD SECTOR COMMENT O
1 Proposed Construction of a standard Maternity Annex at Mpima 5
Mpima health center
.
— =
Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved =
Lukanga Secondary School i
o
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
Kasanda Malombe Secondary School Oo
O
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
Water reticulated Systems with lockable kiosks in Waya Sanitation
communities
Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved
Water reticulated Systems at Kamushanga Market Shelter Sanitation
Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved
Water reticulated Systems at C-gate Priamary School
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Appro

In [29]:
ocr_path = (
    "../data/raw/"
    "2025_kabwe_central_cdf_ocr.txt"
)

with open(ocr_path, "w", encoding="utf-8") as file:
    file.write(full_ocr_text)

print("Complete OCR saved:", ocr_path)

Complete OCR saved: ../data/raw/2025_kabwe_central_cdf_ocr.txt


In [31]:
import re

In [36]:
project_lines = []

for line in lines:
    line = line.strip()

    if re.match(r"^\d+\s", line):
        project_lines.append(line)

for line in project_lines:
    print(line)

1 Proposed Construction of a standard Maternity Annex at Mpima 5
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
12 | Proposed extension and rehabilitation of Waya market Waya Commerce and | Approved 140]
13 | Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved
14 | Completion of Kabwe General Hospital’s Relative waiting Luangwa Health Approved
15 | Construction of an Ablution Block at Mpima Prison Primary | MPIMA Approved
17 | Additional Roads Funding for roads ALL Transport Approved =
19 | Additional funding for Kasanda Market Justin Commerce and | Approved =
20 | Proposed Construction of a 1x4 Classroom block ( CRB) at

In [37]:
print("Number of numbered rows:", len(project_lines))

Number of numbered rows: 25


In [38]:
for i, line in enumerate(project_lines, start=1):
    print(f"{i}: {line}")

1: 1 Proposed Construction of a standard Maternity Annex at Mpima 5
2: 7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
3: 4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
4: 7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
5: 10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
6: 11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
7: 12 | Proposed extension and rehabilitation of Waya market Waya Commerce and | Approved 140]
8: 13 | Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved
9: 14 | Completion of Kabwe General Hospital’s Relative waiting Luangwa Health Approved
10: 15 | Construction of an Ablution Block at Mpima Prison Primary | MPIMA Approved
11: 17 | Additional Roads Funding for roads ALL Transport Approved =
12: 19 | Additional funding for Kasanda Market Justin Commerce and | Approved =
13: 20 | Proposed Cons

In [39]:
ocr_rows_path = "../data/raw/2025_kabwe_central_cdf_project_rows_ocr.txt"

with open(ocr_rows_path, "w", encoding="utf-8") as file:
    for i, line in enumerate(project_lines, start=1):
        file.write(f"{i}: {line}\n")

print("Saved:", ocr_rows_path)

Saved: ../data/raw/2025_kabwe_central_cdf_project_rows_ocr.txt


In [40]:
from pytesseract import Output

ocr_data = pytesseract.image_to_data(
    pages[0],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

ocr_data = ocr_data.dropna(subset=["text"])

ocr_data["text"] = (
    ocr_data["text"]
    .astype(str)
    .str.strip()
)

ocr_data = ocr_data[ocr_data["text"] != ""]

ocr_data = ocr_data.sort_values(
    ["top", "left"]
)

print(
    ocr_data[
        ["top", "left", "width", "text"]
    ].to_string(index=False)
)

 top  left  width            text
  72  3390     55               _
 106  3390     54               ©
 162  1191    295       COMMUNITY
 163   709     85             CDF
 163   808    106            2025
 163   932    242        PROPOSED
 164  1503    194         PROJECT
 165  1713    163           KABWE
 165  1893    206         CENTRAL
 166  2113    344    CONSTITUENCY
 279  3389     56               ©
 330  3390     55               O
 348  2001    172          SECTOR
 349   197     29               |
 353  1667    143            WARD
 353  2589    247         COMMENT
 355   979    185         APPLIED
 355  1192    391 FORSHORTLUSTED;
 356   232    136            NAME
 356   384     60              OF
 356   774    192         PROJECT
 357   110     67              NO
 357   459    299       COMMUNITY
 538  3369     76               5
 581   841    191        standard
 582   745     43              of
 582  1050    216       Maternity
 582  1616    151           Mpima
 583  1279    

In [41]:
project_numbers = []

for line in project_lines:
    match = re.match(r"^(\d+)", line)

    if match:
        project_numbers.append(int(match.group(1)))

print(project_numbers)

[1, 7, 4, 7, 10, 11, 12, 13, 14, 15, 17, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 32, 33]


In [42]:
for i, line in enumerate(project_lines, start=1):
    print(f"\nROW {i}")
    print(line)


ROW 1
1 Proposed Construction of a standard Maternity Annex at Mpima 5

ROW 2
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =

ROW 3
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved

ROW 4
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved

ROW 5
10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c

ROW 6
11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD

ROW 7
12 | Proposed extension and rehabilitation of Waya market Waya Commerce and | Approved 140]

ROW 8
13 | Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved

ROW 9
14 | Completion of Kabwe General Hospital’s Relative waiting Luangwa Health Approved

ROW 10
15 | Construction of an Ablution Block at Mpima Prison Primary | MPIMA Approved

ROW 11
17 | Additional Roads Funding for roads ALL Transport Approved =

ROW 12
19 | Additional funding for Kasanda Market Justin 

In [45]:
# Recreate OCR data for page 1
page1_ocr = pytesseract.image_to_data(
    pages[0],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page1_ocr = page1_ocr.dropna(subset=["text"])

page1_ocr["text"] = (
    page1_ocr["text"]
    .astype(str)
    .str.strip()
)

page1_ocr = page1_ocr[
    page1_ocr["text"] != ""
]

# Find numbers in the NO column
page1_numbers = page1_ocr[
    (page1_ocr["left"] < 210) &
    (page1_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page1_numbers[
        ["top", "left", "text"]
    ].to_string(index=False)
)

 top  left text
 587   111    1
1026    76    7
1296   100    4
2177    93    7


In [46]:
page1_ocr = pytesseract.image_to_data(
    pages[0],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page1_ocr = page1_ocr.dropna(subset=["text"])

page1_ocr["text"] = (
    page1_ocr["text"]
    .astype(str)
    .str.strip()
)

page1_ocr = page1_ocr[
    page1_ocr["text"] != ""
]

page1_numbers = page1_ocr[
    (page1_ocr["left"] < 210) &
    (page1_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page1_numbers[
        ["top", "left", "text"]
    ].to_string(index=False)
)

 top  left text
 587   111    1
1026    76    7
1296   100    4
2177    93    7


In [47]:
row1 = page1_ocr[
    (page1_ocr["top"] >= 500) &
    (page1_ocr["top"] <= 750)
].copy()

print(
    row1[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

 top  left         text
 538  3369            5
 581   841     standard
 582   745           of
 582  1050    Maternity
 582  1616        Mpima
 583  1279        Annex
 584   231     Proposed
 585   450 Construction
 586  1430           at
 587   111            1
 592   802            a
 655   396       health
 656   230        Mpima
 660   546       center
 684  3365            .
 745  3370            =


In [48]:
row2 = page1_ocr[
    (page1_ocr["top"] >= 780) &
    (page1_ocr["top"] <= 1000)
].copy()

print(
    row2[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

 top  left         text
 781  1805            —
 799  3390            =
 815  2310     Approved
 817   738           of
 817   927    Classroom
 818  1172        block
 819   839          1x4
 819  1302            (
 819  1329         CRB)
 819  1615         High
 819  1727        ridge
 820   229     Proposed
 821   446 Construction
 826  1444           at
 828   797            a
 889   655       School
 891   416    Secondary
 893   228      Lukanga
 903  3370            i
 960  3390            o


In [49]:
page1_ocr.sort_values(
    ["top", "left"]
)[
    ["top", "left", "text"]
].to_string(index=False)

' top  left            text\n  72  3390               _\n 106  3390               ©\n 162  1191       COMMUNITY\n 163   709             CDF\n 163   808            2025\n 163   932        PROPOSED\n 164  1503         PROJECT\n 165  1713           KABWE\n 165  1893         CENTRAL\n 166  2113    CONSTITUENCY\n 279  3389               ©\n 330  3390               O\n 348  2001          SECTOR\n 349   197               |\n 353  1667            WARD\n 353  2589         COMMENT\n 355   979         APPLIED\n 355  1192 FORSHORTLUSTED;\n 356   232            NAME\n 356   384              OF\n 356   774         PROJECT\n 357   110              NO\n 357   459       COMMUNITY\n 538  3369               5\n 581   841        standard\n 582   745              of\n 582  1050       Maternity\n 582  1616           Mpima\n 583  1279           Annex\n 584   231        Proposed\n 585   450    Construction\n 586  1430              at\n 587   111               1\n 592   802               a\n 655   396         

In [50]:
working_df = pd.DataFrame({
    "ocr_row": range(1, len(project_lines) + 1),
    "ocr_text": project_lines
})

working_df

NameError: name 'pd' is not defined

In [51]:
import pandas as pd
import re

In [52]:
working_df = pd.DataFrame({
    "ocr_row": range(1, len(project_lines) + 1),
    "ocr_text": project_lines
})

working_df

,ocr_row,ocr_text
0,1,1 Proposed Construction of a standard Maternit...
1,2,7 Proposed Construction of a 1x3 Classroom blo...
2,3,4 | Proposed Construction and installation of ...
3,4,7 | Proposed Construction and installation of ...
4,5,10 | Proposed Construction of an Ablution bloc...
5,6,11 | Procurement of a Hydraulic Tipper Truck A...
6,7,12 | Proposed extension and rehabilitation of ...
7,8,13 | Proposed Completion of Nakoli Market shel...
8,9,14 | Completion of Kabwe General Hospital’s Re...
9,10,15 | Construction of an Ablution Block at Mpim...


In [53]:
working_df[
    ["ocr_row", "no_ocr", "ocr_text"]
].to_string(index=False)

KeyError: "['no_ocr'] not in index"

In [54]:
working_df["no_ocr"] = working_df["ocr_text"].str.extract(
    r"^(\d+)"
)[0]

print(working_df.columns.tolist())

['ocr_row', 'ocr_text', 'no_ocr']


In [55]:
print(
    working_df[
        ["ocr_row", "no_ocr", "ocr_text"]
    ].to_string(index=False)
)

 ocr_row no_ocr                                                                                                              ocr_text
       1      1                                                      1 Proposed Construction of a standard Maternity Annex at Mpima 5
       2      7                                                    7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
       3      4                              4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
       4      7                           7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
       5     10                                           10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
       6     11                                                11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
       7     12                           12 | Proposed extens

In [56]:
working_df["project_text"] = (
    working_df["ocr_text"]
    .str.replace(r"^\d+\s*", "", regex=True)
    .str.strip()
)

working_df[
    ["ocr_row", "no_ocr", "project_text"]
]

,ocr_row,no_ocr,project_text
0,1,1,Proposed Construction of a standard Maternity ...
1,2,7,Proposed Construction of a 1x3 Classroom block...
2,3,4,| Proposed Construction and installation of 02...
3,4,7,| Proposed Construction and installation of 02...
4,5,10,| Proposed Construction of an Ablution block a...
5,6,11,| Procurement of a Hydraulic Tipper Truck All ...
6,7,12,| Proposed extension and rehabilitation of Way...
7,8,13,| Proposed Completion of Nakoli Market shelter...
8,9,14,| Completion of Kabwe General Hospital’s Relat...
9,10,15,| Construction of an Ablution Block at Mpima P...


In [57]:
working_df["project_text"] = (
    working_df["project_text"]
    .str.replace(r"\s*[=—]+$", "", regex=True)
    .str.strip()
)

print(
    working_df[
        ["ocr_row", "no_ocr", "project_text"]
    ].to_string(index=False)
)

 ocr_row no_ocr                                                                                                       project_text
       1      1                                                     Proposed Construction of a standard Maternity Annex at Mpima 5
       2      7                                                     Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa
       3      4                             | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
       4      7                          | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
       5     10                                           | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
       6     11                                                | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
       7     12                           | Proposed extension and rehabilitation o

In [58]:
working_df["project_text"] = (
    working_df["project_text"]
    .str.replace(r"\s*[=—]+\s*$", "", regex=True)
    .str.strip()
)

working_df[
    ["ocr_row", "no_ocr", "project_text"]
]

,ocr_row,no_ocr,project_text
0,1,1,Proposed Construction of a standard Maternity ...
1,2,7,Proposed Construction of a 1x3 Classroom block...
2,3,4,| Proposed Construction and installation of 02...
3,4,7,| Proposed Construction and installation of 02...
4,5,10,| Proposed Construction of an Ablution block a...
5,6,11,| Procurement of a Hydraulic Tipper Truck All ...
6,7,12,| Proposed extension and rehabilitation of Way...
7,8,13,| Proposed Completion of Nakoli Market shelter...
8,9,14,| Completion of Kabwe General Hospital’s Relat...
9,10,15,| Construction of an Ablution Block at Mpima P...


In [59]:
# Show OCR text that appears in the Ward, Sector and Comment areas
page1_columns = page1_ocr[
    page1_ocr["left"] >= 1550
].copy()

print(
    page1_columns[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

 top  left         text
  72  3390            _
 106  3390            ©
 165  1713        KABWE
 165  1893      CENTRAL
 166  2113 CONSTITUENCY
 279  3389            ©
 330  3390            O
 348  2001       SECTOR
 353  1667         WARD
 353  2589      COMMENT
 538  3369            5
 582  1616        Mpima
 684  3365            .
 745  3370            =
 781  1805            —
 799  3390            =
 815  2310     Approved
 819  1615         High
 819  1727        ridge
 903  3370            i
 960  3390            o
1017  3389            =
1052  1614       Chirwa
1129  3389           Oo
1183  3390            O
1287  1571            |
1290  2067          and
1292  2312     Approved
1293  1613         Waya
1293  1915        Water
1359  1915   Sanitation
1673  1564            |
1677  1615      Kalonga
1681  2070          and
1681  2316     Approved
1682  1917        Water
1755  1917   Sanitation
1920  1565            |
1922  1917    Education
1925  1615     Kaputula
1926  2318     A

In [60]:
# Define the approximate column boundaries
WARD_MIN = 1550
WARD_MAX = 1850

SECTOR_MIN = 1850
SECTOR_MAX = 2250

COMMENT_MIN = 2250
COMMENT_MAX = 3000

# Extract text from each column
ward_data = page1_ocr[
    (page1_ocr["left"] >= WARD_MIN) &
    (page1_ocr["left"] < WARD_MAX)
].copy()

sector_data = page1_ocr[
    (page1_ocr["left"] >= SECTOR_MIN) &
    (page1_ocr["left"] < SECTOR_MAX)
].copy()

comment_data = page1_ocr[
    (page1_ocr["left"] >= COMMENT_MIN) &
    (page1_ocr["left"] < COMMENT_MAX)
].copy()

print("WARD:")
print(
    ward_data[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

print("\nSECTOR:")
print(
    sector_data[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

print("\nCOMMENT:")
print(
    comment_data[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

WARD:
 top  left     text
 165  1713    KABWE
 353  1667     WARD
 582  1616    Mpima
 781  1805        —
 819  1615     High
 819  1727    ridge
1052  1614   Chirwa
1287  1571        |
1293  1613     Waya
1673  1564        |
1677  1615  Kalonga
1920  1565        |
1925  1615 Kaputula
2169  1564        |
2175  1615  Nijanji

SECTOR:
 top  left         text
 165  1893      CENTRAL
 166  2113 CONSTITUENCY
 348  2001       SECTOR
1290  2067          and
1293  1915        Water
1359  1915   Sanitation
1681  2070          and
1682  1917        Water
1755  1917   Sanitation
1922  1917    Education
2166  1919    Education

COMMENT:
 top  left     text
 353  2589  COMMENT
 815  2310 Approved
1292  2312 Approved
1681  2316 Approved
1926  2318 Approved
2169  2316 Approved


In [61]:
def get_row_text(data, start_top, end_top, min_left=0, max_left=3000):
    row = data[
        (data["top"] >= start_top) &
        (data["top"] < end_top) &
        (data["left"] >= min_left) &
        (data["left"] < max_left)
    ].copy()

    row = row.sort_values(["top", "left"])

    return " ".join(row["text"].tolist())


# Test with the first project
row1_text = get_row_text(
    page1_ocr,
    500,
    780,
    50,
    3000
)

print(row1_text)

standard of Maternity Mpima Annex Proposed Construction at 1 a health Mpima center


In [62]:
# Display the first project's OCR words with their coordinates
row1 = page1_ocr[
    (page1_ocr["top"] >= 530) &
    (page1_ocr["top"] <= 700)
].copy()

row1 = row1.sort_values(["top", "left"])

print(
    row1[
        ["top", "left", "text"]
    ].to_string(index=False)
)

 top  left         text
 538  3369            5
 581   841     standard
 582   745           of
 582  1050    Maternity
 582  1616        Mpima
 583  1279        Annex
 584   231     Proposed
 585   450 Construction
 586  1430           at
 587   111            1
 592   802            a
 655   396       health
 656   230        Mpima
 660   546       center
 684  3365            .


In [63]:
# Page 1
page1 = pages[0]

# Row 1 vertical range
Y1 = 530
Y2 = 710

# Approximate table column boundaries
columns = {
    "NO": (0, 210),
    "PROJECT": (210, 1550),
    "WARD": (1550, 1850),
    "SECTOR": (1850, 2250),
    "COMMENT": (2250, 3000)
}

for column_name, (x1, x2) in columns.items():
    crop = page1.crop((x1, Y1, x2, Y2))

    text = pytesseract.image_to_string(
        crop,
        config="--psm 6"
    )

    print(f"\n--- {column_name} ---")
    print(text.strip())


--- NO ---
P|

--- PROJECT ---
Proposed Construction of a standard Maternity Annex at
Mpima health center

--- WARD ---
a
| Mpima

--- SECTOR ---
| Health

--- COMMENT ---
| Approved


In [64]:
# Row 2 vertical range
Y1 = 780
Y2 = 970

for column_name, (x1, x2) in columns.items():
    crop = page1.crop((x1, Y1, x2, Y2))

    text = pytesseract.image_to_string(
        crop,
        config="--psm 6"
    )

    print(f"\n--- {column_name} ---")
    print(text.strip())


--- NO ---
P |

--- PROJECT ---
Proposed Construction of a 1x4 Classroom block ( CRB) at
Lukanga Secondary School

--- WARD ---
High ridge

--- SECTOR ---
Education

--- COMMENT ---
Approved


In [65]:
# Row 3 vertical range
Y1 = 1170
Y2 = 1460

for column_name, (x1, x2) in columns.items():
    crop = page1.crop((x1, Y1, x2, Y2))

    text = pytesseract.image_to_string(
        crop,
        config="--psm 6"
    )

    print(f"\n--- {column_name} ---")
    print(text.strip())


--- NO ---
7

--- PROJECT ---
Proposed Construction and installation of 02 Solar powered
Water reticulated Systems with lockable kiosks in Waya

--- WARD ---
Waya

--- SECTOR ---
Water and
Sanitation

--- COMMENT ---
|
| Approved


In [66]:
# Row ranges identified on page 1
row_ranges = [
    (1, 530, 710),
    (2, 780, 970),
    (3, 1170, 1460),
    (4, 1600, 1810),
]

all_rows = []

for row_number, y1, y2 in row_ranges:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page1.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows.append(row_data)

page1_df = pd.DataFrame(all_rows)

page1_df

,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,a Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
3,4,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved


In [67]:
print(page1_df.columns.tolist())
print()
print(page1_df.to_string(index=False))

['no', 'project', 'ward', 'sector', 'comment']

 no                                                                                                           project       ward               sector  comment
  1                                        Proposed Construction of a standard Maternity Annex at Mpima health center    a Mpima               Health Approved
  2                                 Proposed Construction of a 1x4 Classroom block ( CRB) at Lukanga Secondary School High ridge            Education Approved
  3 Proposed Construction and installation of 02 Solar powered Water reticulated Systems with lockable kiosks in Waya       Waya Water and Sanitation Approved
  4 Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamushanga Market Shelter    Kalonga Water and Sanitation Approved


In [68]:
# Clean OCR noise from the ward column
page1_df["ward"] = (
    page1_df["ward"]
    .str.replace(r"^\s*a\s+", "", regex=True)
    .str.strip()
)

# Clean extra spaces inside project names
page1_df["project"] = (
    page1_df["project"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Clean the other text columns
for column in ["ward", "sector", "comment"]:
    page1_df[column] = (
        page1_df[column]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print(page1_df.to_string(index=False))

 no                                                                                                           project       ward               sector  comment
  1                                        Proposed Construction of a standard Maternity Annex at Mpima health center      Mpima               Health Approved
  2                                 Proposed Construction of a 1x4 Classroom block ( CRB) at Lukanga Secondary School High ridge            Education Approved
  3 Proposed Construction and installation of 02 Solar powered Water reticulated Systems with lockable kiosks in Waya       Waya Water and Sanitation Approved
  4 Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamushanga Market Shelter    Kalonga Water and Sanitation Approved


In [69]:
# Get OCR data for Page 2
page2_ocr = pytesseract.image_to_data(
    pages[1],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page2_ocr = page2_ocr.dropna(subset=["text"])
page2_ocr["text"] = page2_ocr["text"].astype(str).str.strip()
page2_ocr = page2_ocr[page2_ocr["text"] != ""]

# Display numeric tokens that may indicate project rows
page2_numbers = page2_ocr[
    (page2_ocr["left"] < 210) &
    (page2_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page2_numbers[["top", "left", "text"]]
    .to_string(index=False)
)

 top  left text
 659   116   10
 907   117   11
1145   116   12
1389   117   13
1636   116   14
1883   115   15


In [70]:
# Page 2
page2 = pages[1]

# Row ranges for Page 2
row_ranges_p2 = [
    (10, 550, 800),
    (11, 800, 1050),
    (12, 1050, 1290),
    (13, 1290, 1535),
    (14, 1535, 1780),
    (15, 1780, 2020),
]

all_rows_p2 = []

for row_number, y1, y2 in row_ranges_p2:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page2.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p2.append(row_data)

page2_df = pd.DataFrame(all_rows_p2)

print(page2_df.to_string(index=False))

 no                                                              project    ward            sector     comment
 10   Proposed Construction of an Ablution block at BOCCs Primary School Katondo         Education    Approved
 11                              Procurement of a Hydraulic Tipper Truck                 Transport    Approved
 12 sinning Proposed extension and rehabilitation of Waya market shelter    Waya Commerce ai Trade nd Approved
 13                         Proposed Completion of Nakoli Market shelter  Nakoli Commerce al Trade id Approved
 14      Completion of Kabwe General Hospital’s Relative waiting Shelter Luangwa            Health    Approved
 15      Construction of an Ablution Block at Mpima Prison Primar SchooL y MPIMA         Education    Approved


In [71]:
# Inspect Page 2 row 10 more carefully
row10 = page2.crop((0, 550, 3000, 800))

# OCR the complete row using a different page segmentation mode
row10_text = pytesseract.image_to_string(
    row10,
    config="--psm 4"
)

print(row10_text)

Katondo Education Approved

10 | Proposed Construction of an Ablution block at BOCCs
Primary School



In [72]:
# Inspect Row 11 more carefully
row11 = page2.crop((0, 800, 3000, 1050))

row11_text = pytesseract.image_to_string(
    row11,
    config="--psm 4"
)

print(row11_text)

Procurement of a Hydraulic Tipper Truck Approved

i |



In [73]:
# Inspect Row 11 Ward and Sector columns

for column_name in ["WARD", "SECTOR"]:
    x1, x2 = columns[column_name]

    crop = page2.crop((x1, 800, x2, 1050))

    text = pytesseract.image_to_string(
        crop,
        config="--psm 6"
    )

    print(f"\n--- {column_name} ---")
    print(text.strip())


--- WARD ---


--- SECTOR ---
Transport


In [74]:
# Inspect Row 12
row12 = page2.crop((0, 1050, 3000, 1290))

row12_text = pytesseract.image_to_string(
    row12,
    config="--psm 4"
)

print(row12_text)

12 | Proposed extension and rehabilitation of Waya market Commerce and | Approved
shelter



In [75]:
# Inspect Row 13
row13 = page2.crop((0, 1290, 3000, 1535))

row13_text = pytesseract.image_to_string(
    row13,
    config="--psm 4"
)

print(row13_text)

Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved



In [76]:
# Inspect Row 14
row14 = page2.crop((0, 1535, 3000, 1780))

row14_text = pytesseract.image_to_string(
    row14,
    config="--psm 4"
)

print(row14_text)

Health

Completion of Kabwe General Hospital’s Relative waiting Luangwa Approved

Shelter




In [77]:
# Inspect Row 15
row15 = page2.crop((0, 1780, 3000, 2020))

row15_text = pytesseract.image_to_string(
    row15,
    config="--psm 4"
)

print(row15_text)

Construction of an Ablution Block at Mpima Prison Primary | MPIMA Education Approved

SchooL




In [78]:
# Get OCR data for Page 3
page3_ocr = pytesseract.image_to_data(
    pages[2],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page3_ocr = page3_ocr.dropna(subset=["text"])
page3_ocr["text"] = page3_ocr["text"].astype(str).str.strip()
page3_ocr = page3_ocr[page3_ocr["text"] != ""]

# Find possible row numbers
page3_numbers = page3_ocr[
    (page3_ocr["left"] < 210) &
    (page3_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page3_numbers[["top", "left", "text"]]
    .to_string(index=False)
)

 top  left text
 256    90   17
 726    93   19
 978    92   20
1377    93   21
1777    91   22
2027    89   23


In [79]:
# Page 3
page3 = pages[2]

# Row ranges based on the detected OCR positions
row_ranges_p3 = [
    (17, 150, 600),
    (19, 600, 850),
    (20, 850, 1200),
    (21, 1200, 1550),
    (22, 1550, 1900),
    (23, 1900, 2250),
]

all_rows_p3 = []

for row_number, y1, y2 in row_ranges_p3:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page3.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p3.append(row_data)

page3_df = pd.DataFrame(all_rows_p3)

print(page3_df.to_string(index=False))

 no                                                                                   project          ward                     sector                                                                                                                                                     comment
 17         Additional Roads Funding for roads Additional funding for Njanii Market structure     ALL NJANI Transport Commerce = Trade                                                                                                                                       Approved ind Approved
 19                                                     Additional funding for Kasanda Market  Justin Kabwe           Commerce = Trade                                                                                                                                                ind Approved
 20               Proposed Construction of a 1x4 Classroom block ( CRB) at Mpima Dairy School         Mpima                  Ed

In [80]:
# Inspect Row 17 more carefully
row17 = page3.crop((0, 150, 3300, 600))

row17_text = pytesseract.image_to_string(
    row17,
    config="--psm 4"
)

print(row17_text)

SF Sew eee Fe 8 OS OO HO eee ee vu = wears Se Weel soca & ae

17 | Additional Roads Funding for roads Approved

18 | Additional funding for Njanii Market structure Commerce and | Approved




In [81]:
# Inspect Row 19 more carefully
row19 = page3.crop((0, 600, 3300, 850))

row19_text = pytesseract.image_to_string(
    row19,
    config="--psm 4"
)

print(row19_text)

Additional funding for Kasanda Market Commerce and | Approved



In [82]:
# Inspect Row 20
row20 = page3.crop((0, 850, 3300, 1200))

row20_text = pytesseract.image_to_string(
    row20,
    config="--psm 4"
)

print(row20_text)

20 | Proposed Construction of a 1x4 Classroom block ( CRB) at
Mpima Dairy School

Education

Not approved because the ward
already got an allocation for one
project, nence under the principal of



In [83]:
# Extract all remaining rows from Page 3

row_ranges_p3_all = [
    (17, 150, 600),
    (18, 600, 850),
    (19, 850, 1200),
    (20, 1200, 1550),
    (21, 1550, 1900),
    (22, 1900, 2150),
    (23, 2150, 2450),
]

all_rows_p3 = []

for row_number, y1, y2 in row_ranges_p3_all:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page3.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p3.append(row_data)

page3_df = pd.DataFrame(all_rows_p3)

print(page3_df.to_string(index=False))

 no                                                                                   project          ward                     sector                                                                                                                                                     comment
 17         Additional Roads Funding for roads Additional funding for Njanii Market structure     ALL NJANI Transport Commerce = Trade                                                                                                                                       Approved ind Approved
 18                                                     Additional funding for Kasanda Market  Justin Kabwe           Commerce = Trade                                                                                                                                                ind Approved
 19               Proposed Construction of a 1x4 Classroom block ( CRB) at Mpima Dairy School         Mpima                  Ed

In [84]:
# Look at all OCR tokens near the NO column on Page 3
# This helps us identify the actual vertical positions of every row.

left_side = page3_ocr[
    page3_ocr["left"] < 300
].copy()

left_side = left_side.sort_values(["top", "left"])

print(
    left_side[["top", "left", "text"]]
    .to_string(index=False)
)

 top  left       text
 249   168          |
 254   212 Additional
 256    90         17
 477   215 Additional
 717   168          |
 725   216 Additional
 726    93         19
 934   196          a
 969   174          |
 976   219   Proposed
 978    92         20
1053   218      Mpima
1367   175          |
1375   219   Proposed
1377    93         21
1449   216  Secondary
1765   172          |
1774   217   Proposed
1777    91         22
1850   218      Kabwe
2014   175          |
2023   218   Proposed
2027    89         23
2098   276     Family
2105   215         at


In [85]:
# Correct row ranges for Page 3
row_ranges_p3 = [
    (17, 200, 470),
    (18, 470, 720),
    (19, 720, 970),
    (20, 970, 1365),
    (21, 1365, 1760),
    (22, 1760, 2015),
    (23, 2015, 2350),
]

all_rows_p3 = []

for row_number, y1, y2 in row_ranges_p3:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page3.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p3.append(row_data)

page3_df = pd.DataFrame(all_rows_p3)

print(page3_df.to_string(index=False))

 no                                                                                   project               ward           sector                                                                                                                                      comment
 17                                                        Additional Roads Funding for roads                ALL        Transport                                                                                                                                     Approved
 18                                            Additional funding for Njanii Market structure              NJANI Commerce = Trade                                                                                                                                 ind Approved
 19                                                     Additional funding for Kasanda Market ) Justin Kabwe ——— Commerce = Trade                                                          

In [86]:
# Search Page 3 OCR for possible project row numbers 24–33

remaining_numbers = page3_ocr[
    page3_ocr["text"].str.match(
        r"^(24|25|26|27|28|29|30|31|32|33)$",
        na=False
    )
].copy()

print(
    remaining_numbers[
        ["top", "left", "text"]
    ].sort_values("top").to_string(index=False)
)

Empty DataFrame
Columns: [top, left, text]
Index: []


In [87]:
# Show OCR text on the left side of Page 3 from below Row 23
# This helps us locate the remaining project rows.

remaining_left = page3_ocr[
    (page3_ocr["top"] > 2200) &
    (page3_ocr["left"] < 300)
].copy()

remaining_left = remaining_left.sort_values(["top", "left"])

print(
    remaining_left[
        ["top", "left", "text"]
    ].to_string(index=False)
)

Empty DataFrame
Columns: [top, left, text]
Index: []


In [88]:
print("Page 3 image size:", page3.size)
print("Page 3 height:", page3.height)

print("\nLast OCR positions:")
print(
    page3_ocr[["top", "left", "text"]]
    .sort_values("top")
    .tail(30)
    .to_string(index=False)
)

Page 3 image size: (3509, 2480)
Page 3 height: 2480

Last OCR positions:
 top  left         text
1851   516      Primary
1852   383        Trust
2012  2872 Insufficient
2014  1615            |
2014   175            |
2014  2711          due
2014  2483     approved
2018  1664      Lukanga
2019  1452        block
2019  2808           to
2019  2386          Not
2021   879       double
2021   730           of
2022   787          the
2022  1205    classroom
2023   450   Completion
2023   218     Proposed
2027    89           23
2027  1050       storey
2085  2792          all
2086  2712          for
2088  2383        funds
2088  2858     projects
2092  2586        cater
2093  2522           to
2095   885       School
2098   276       Family
2098   605    Community
2100   440       Future
2105   215           at


In [89]:
print("Number of pages:", len(pages))

for i, page in enumerate(pages):
    print(f"Page {i + 1}: {page.size}")

Number of pages: 5
Page 1: (3509, 2480)
Page 2: (3509, 2480)
Page 3: (3509, 2480)
Page 4: (3509, 2480)
Page 5: (3509, 2480)


In [90]:
# Get OCR data for Page 4
page4_ocr = pytesseract.image_to_data(
    pages[3],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page4_ocr = page4_ocr.dropna(subset=["text"])
page4_ocr["text"] = page4_ocr["text"].astype(str).str.strip()
page4_ocr = page4_ocr[page4_ocr["text"] != ""]

# Look for possible project numbers on the left side
page4_numbers = page4_ocr[
    (page4_ocr["left"] < 300) &
    (page4_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page4_numbers[
        ["top", "left", "text"]
    ].sort_values("top").to_string(index=False)
)

 top  left text
 234   166   24
 615   167   25
 854   168   26
1165   173   27
1692   177   28
2156   180   29


In [91]:
# Page 4
page4 = pages[3]

# Row ranges for Page 4
row_ranges_p4 = [
    (24, 150, 500),
    (25, 500, 800),
    (26, 800, 1100),
    (27, 1100, 1600),
    (28, 1600, 2050),
    (29, 2050, 2400),
]

all_rows_p4 = []

for row_number, y1, y2 in row_ranges_p4:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page4.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p4.append(row_data)

page4_df = pd.DataFrame(all_rows_p4)

print(page4_df.to_string(index=False))

 no                                                                  project       ward               sector                                                                                                                                                                        comment
 24     Proposed Construction of an Ablution block at Gombe Secondary School       Waya            Education                                                                    Not approved as Waya ward project under educational se which was approved, since resources had to be evenly
 25    . Proposed Construction of an Ablution block at Kakama Primary School    Kalonga            Education                                                     distributed in all the 14 ward A project was already approved this ward, and for equity sake, c wards had to be considered
 26 ) Proposed Construction of an Ablution block at Kamushanj Market shelter za Kalonga Water and Sanitation                                        

In [92]:
# Verify Rows 24–26 using a wider full-row OCR crop

verification_ranges_p4 = [
    (24, 180, 500),
    (25, 500, 800),
    (26, 800, 1100),
]

for row_number, y1, y2 in verification_ranges_p4:
    print("=" * 100)
    print(f"ROW {row_number}")
    
    crop = page4.crop((0, y1, 3300, y2))
    
    text = pytesseract.image_to_string(
        crop,
        config="--psm 4"
    )
    
    text = re.sub(r"\s+", " ", text)
    print(text.strip())

ROW 24
24 | Proposed Construction of an Ablution block at Gombe Waya Education Not approved as Waya ward had a | Secondary School project under educational sector | which was approved, since resources had to be evenly |
ROW 25
distributed in all the 14 wards. 25 | Proposed Construction of an Ablution block at Kakama Kalonga Education A project was already approved under Primary School this ward, and for equity sake, other wards had to be considered
ROW 26
Water and Sanitation Kalonga has had a one or two projects approved ,hence the need to consider other wards amongst the 14. 26 | Proposed Construction of an Ablution block at Kamushanga | Kalonga Market shelter


In [93]:
# Verify Rows 27–29 using full-row OCR

verification_ranges_p4_2 = [
    (27, 1100, 1600),
    (28, 1600, 2050),
    (29, 2050, 2400),
]

for row_number, y1, y2 in verification_ranges_p4_2:
    print("=" * 100)
    print(f"ROW {row_number}")
    
    crop = page4.crop((0, y1, 3300, y2))
    
    text = pytesseract.image_to_string(
        crop,
        config="--psm 4"
    )
    
    text = re.sub(r"\s+", " ", text)
    print(text.strip())

ROW 27
27 | Supply and installation of a Micro Burn Unit at Nakoli Clinic | Nakoli Health Not approved due to Insufficient | funds and furthermore, CDF prioritizes primary health care needs like OPD,Maternity annexes and no records showed a high incidence of burn cases in the
ROW 28
catcnment area 28 | Completion of Chindwin barrack school wall fence KAPUTULA | Education Not approved due to Insufficient funds and furthermore, the need is not priority compared to other needs, as the school is next to a defense unit that currently provides
ROW 29
urity. 29 | Procurement of equipment for Katondo Maternity Annex Katondo Health Not approved as the ward was already considered for a project under the educational sector


In [94]:
# Get OCR data for Page 5
page5_ocr = pytesseract.image_to_data(
    pages[4],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page5_ocr = page5_ocr.dropna(subset=["text"])
page5_ocr["text"] = page5_ocr["text"].astype(str).str.strip()
page5_ocr = page5_ocr[page5_ocr["text"] != ""]

# Look for possible project numbers on the left side
page5_numbers = page5_ocr[
    (page5_ocr["left"] < 300) &
    (page5_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page5_numbers[
        ["top", "left", "text"]
    ].sort_values("top").to_string(index=False)
)

 top  left text
 199   167   30
 444   164   31
 689   165   32
1154   164   33


In [95]:
# Page 5
page5 = pages[4]

# Row ranges for Page 5
row_ranges_p5 = [
    (30, 150, 440),
    (31, 440, 690),
    (32, 690, 1150),
    (33, 1150, 2400),
]

all_rows_p5 = []

for row_number, y1, y2 in row_ranges_p5:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page5.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p5.append(row_data)

page5_df = pd.DataFrame(all_rows_p5)

print(page5_df.to_string(index=False))

 no                                                                                   project             ward       sector                                                                                                                                                 comment
 30                 ) Procurement of equipment for Magandanyama Maternit Annex and laboratory    y oavd Ramust    Health 10                                                                                              Not approved due to Insuffi funds to cater for all project
 31                                        Procurement of equipment for Mpima Maternity Annex            Mpima       Health                                                                            Not approved as another pr under the health sector was approved in this ward
 32                                      Procurement of equipment for Kamakuti Maternity Anne           x Waya       Health Not approved due to Insuffi funds to cater for a

In [96]:
# Verify Rows 30–33 using full-row OCR

verification_ranges_p5 = [
    (30, 150, 440),
    (31, 440, 690),
    (32, 690, 1150),
    (33, 1150, 2400),
]

for row_number, y1, y2 in verification_ranges_p5:
    print("=" * 100)
    print(f"ROW {row_number}")
    
    crop = page5.crop((0, y1, 3300, y2))
    
    text = pytesseract.image_to_string(
        crop,
        config="--psm 4"
    )
    
    text = re.sub(r"\s+", " ", text)
    print(text.strip())

ROW 30
30 | Procurement of equipment for Magandanyama Maternity David Health Not approved due to Insufficient Annex and laboratory Ramusho funds to cater for all projects
ROW 31
Procurement of equipment for Mpima Maternity Annex Mpima Health Not approved as another project under the health sector was already approved in this ward
ROW 32
Procurement of equipment for Kamakuti Maternity Annex Health Not approved due to Insufficient funds to cater for all projects -There was no needs assessment report from the department of Health and usage data to justify the equipment requested
ROW 33
David - Ramusho Education 3 | Proposed Construction of a 1x4 Classroom block ( CRB) at David Ramusho Secondary School Not approved due to Insufficient funds and also this ward has a project approved already ,jhence the need to consider other wards


In [97]:
# Combine all 33 extracted rows from Pages 1–5

proposed_2025_df = pd.concat(
    [
        page1_df,
        page2_df,
        page3_df,
        page4_df,
        page5_df
    ],
    ignore_index=True
)

# Sort by project number
proposed_2025_df = proposed_2025_df.sort_values(
    by="no"
).reset_index(drop=True)

print("Total rows:", len(proposed_2025_df))
print("Total columns:", len(proposed_2025_df.columns))

display(proposed_2025_df)

Total rows: 27
Total columns: 5


,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
3,4,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved
4,10,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved
5,11,Procurement of a Hydraulic Tipper Truck,,Transport,Approved
6,12,sinning Proposed extension and rehabilitation ...,Waya,Commerce ai Trade,nd Approved
7,13,Proposed Completion of Nakoli Market shelter,Nakoli,Commerce al Trade,id Approved
8,14,Completion of Kabwe General Hospital’s Relativ...,Luangwa,Health,Approved
9,15,Construction of an Ablution Block at Mpima Pri...,y MPIMA,Education,Approved


In [98]:
# Find all possible project numbers on Page 1
page1_numbers = page1_ocr[
    (page1_ocr["left"] < 300) &
    (page1_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page1_numbers[
        ["top", "left", "text"]
    ].sort_values("top").to_string(index=False)
)

 top  left text
 587   111    1
1026    76    7
1296   100    4
2177    93    7


In [99]:
# Inspect the middle section of Page 1 where Rows 5–9 should appear

crop = page1.crop((0, 450, 3300, 2200))

text = pytesseract.image_to_string(
    crop,
    config="--psm 4"
)

text = re.sub(r"\s+", " ", text)

print(text.strip())

Swe owes F UV Sra VS SS eee = 1 Proposed Construction of a standard Maternity Annex at Mpima Health Approved Mpima health center Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved Lukanga Secondary School Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa Education Approved Kasanda Malombe Secondary School Proposed Construction and installation of 02 Solar powered | Waya Water and Approved Water reticulated Systems with lockable kiosks in Waya Sanitation communities Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved Water reticulated Systems at Kamushanga Market Shelter Sanitation Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved Water reticulated Systems at C-gate Priamary School Dreanncad fanctriirtian and inctallatinan atfNS Calar nawuraran Cdiiratinn Annrovoed


In [100]:
# Inspect the lower section of Page 1 for Rows 6–9

crop = page1.crop((0, 1700, 3300, 2480))

text = pytesseract.image_to_string(
    crop,
    config="--psm 4"
)

text = re.sub(r"\s+", " ", text)

print(text.strip())

FRU VPUSEY CUNSUIUCLIVUE GU HIstahakiult Ui UL OUI VU GIVilg VWVGLCt au Water reticulated Systems at Kamushanga Market Shelter Sanitation Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved Water reticulated Systems at C-gate Priamary School Proposed Construction and installation of 02 Solar powered Education Approved Water reticulated Systems at Njanji Market facilities


In [3]:
verified_1_33 = [
    [1, "Proposed Construction of a standard Maternity Annex at Mpima health center",
     "Mpima", "Health", "Approved"],

    [2, "Proposed Construction of a 1x4 Classroom block (CRB) at Lukanga Secondary School",
     "High ridge", "Education", "Approved"],

    [3, "Proposed Construction of a 1x3 Classroom block (CRB) at Kasanda Malombe Secondary School",
     "Chirwa", "Education", "Approved"],

    [4, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems with lockable kiosks in Waya communities",
     "Waya", "Water and Sanitation", "Approved"],

    [5, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamushanga Market Shelter",
     "Kalonga", "Water and Sanitation", "Approved"],

    [6, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at C-gate Primary School",
     "Kaputula", "Education", "Approved"],

    [7, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Njanji Market facilities",
     "Njanji", "Education", "Approved"],

    [8, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamakuti Maternity annex",
     "Waya", "Health", "Approved"],

    [9, "Proposed Construction of an Ablution block at C-gate Primary School",
     "Kaputula", "Education", "Approved"],

    [10, "Proposed Construction of an Ablution block at BOCCs Primary School",
     "Katondo", "Education", "Approved"],

    [11, "Procurement of a Hydraulic Tipper Truck",
     "All", "Transport", "Approved"],

    [12, "Proposed extension and rehabilitation of Waya market shelter",
     "Waya", "Commerce and Trade", "Approved"],

    [13, "Proposed Completion of Nakoli Market shelter",
     "Nakoli", "Commerce and Trade", "Approved"],

    [14, "Completion of Kabwe General Hospital's Relative waiting Shelter",
     "Luangwa", "Health", "Approved"],

    [15, "Construction of an Ablution Block at Mpima Prison Primary School",
     "MPIMA", "Education", "Approved"],

    [16, "Procurement of 500 ordinary desks and 40 special desks",
     "ALL", "Education", "Approved"],

    [17, "Additional Roads Funding for roads",
     "ALL", "Transport", "Approved"],

    [18, "Additional funding for Njanii Market structure",
     "NJANI", "Commerce and Trade", "Approved"],

    [19, "Additional funding for Kasanda Market",
     "Justin Kabwe", "Commerce and Trade", "Approved"],

    [20, "Proposed Construction of a 1x4 Classroom block (CRB) at Mpima Dairy School",
     "Mpima", "Education",
     "Not approved because the ward already got an allocation for one project, hence under the principal of equity, resources had to be evenly distributed to cater for all projects"],

    [21, "Proposed Completion of a 1x2 Science Laboratory at Mine Secondary School",
     "Justine Kabwe", "Education",
     "Not approved due to Insufficient funds to cater for all projects and priority was given to projects with wider community benefit or urgent needs"],

    [22, "Proposed Construction of a 1x3 Classroom block (CRB) at Kabwe Trust Primary School",
     "Luangwa", "Education",
     "Not approved due to Insufficient funds to cater for all projects"],

    [23, "Proposed Completion of the double storey classroom block at Family Future Community School",
     "Lukanga", "Education",
     "Not approved due to Insufficient funds to cater for all projects"],

    [24, "Proposed Construction of an Ablution block at Gombe Secondary School",
     "Waya", "Education",
     "Not approved as Waya ward had a project under educational sector which was approved, since resources had to be evenly distributed in all the 14 wards."],

    [25, "Proposed Construction of an Ablution block at Kakama Primary School",
     "Kalonga", "Education",
     "A project was already approved under this ward, and for equity sake, other wards had to be considered"],

    [26, "Proposed Construction of an Ablution block at Kamushanga Market shelter",
     "Kalonga", "Water and Sanitation",
     "Kalonga has had a one or two projects approved, hence the need to consider other wards amongst the 14."],

    [27, "Supply and installation of a Micro Burn Unit at Nakoli Clinic",
     "Nakoli", "Health",
     "Not approved due to Insufficient funds and furthermore, CDF prioritizes primary health care needs like OPD, Maternity annexes and no records showed a high incidence of burn cases in the catchment area"],

    [28, "Completion of Chindwin barrack school wall fence",
     "KAPUTULA", "Education",
     "Not approved due to Insufficient funds and furthermore, the need is not priority compared to other needs, as the school is next to a defense unit that currently provides security."],

    [29, "Procurement of equipment for Katondo Maternity Annex",
     "Katondo", "Health",
     "Not approved as the ward was already considered for a project under the educational sector"],

    [30, "Procurement of equipment for Magandanyama Maternity Annex and laboratory",
     "David Ramusho", "Health",
     "Not approved due to Insufficient funds to cater for all projects"],

    [31, "Procurement of equipment for Mpima Maternity Annex",
     "Mpima", "Health",
     "Not approved as another project under the health sector was already approved in this ward"],

    [32, "Procurement of equipment for Kamakuti Maternity Annex",
     "Waya", "Health",
     "Not approved due to Insufficient funds to cater for all projects - There was no needs assessment report from the department of Health and usage data to justify the equipment requested"],

    [33, "Proposed Construction of a 1x4 Classroom block (CRB) at David Ramusho Secondary School",
     "David Ramusho", "Education",
     "Not approved due to Insufficient funds and also this ward has a project approved already, hence the need to consider other wards"],
]

verified_df = pd.DataFrame(
    verified_1_33,
    columns=["no", "project", "ward", "sector", "comment"]
)

display(verified_df)

,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction of a 1x3 Classroom block...,Chirwa,Education,Approved
3,4,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
4,5,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved
5,6,Proposed Construction and installation of 02 S...,Kaputula,Education,Approved
6,7,Proposed Construction and installation of 02 S...,Njanji,Education,Approved
7,8,Proposed Construction and installation of 02 S...,Waya,Health,Approved
8,9,Proposed Construction of an Ablution block at ...,Kaputula,Education,Approved
9,10,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved


In [4]:
# Standardize the verified 2025 CDF dataset

clean_2025_df = verified_df.copy()

# Clean whitespace in all text columns
text_columns = ["project", "ward", "sector", "comment"]

for column in text_columns:
    clean_2025_df[column] = (
        clean_2025_df[column]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Standardize ward names
ward_mapping = {
    "ALL": "ALL",
    "All": "ALL",
    "MPIMA": "Mpima",
    "NJANI": "Njanji",
    "KAPUTULA": "Kaputula",
}

clean_2025_df["ward"] = (
    clean_2025_df["ward"]
    .replace(ward_mapping)
)

# Standardize common sector OCR/transcription variations
sector_mapping = {
    "Commerce ai Trade": "Commerce and Trade",
    "Commerce al Trade": "Commerce and Trade",
    "Commerce = Trade": "Commerce and Trade",
}

clean_2025_df["sector"] = (
    clean_2025_df["sector"]
    .replace(sector_mapping)
)

# Display the cleaned dataset
display(clean_2025_df)

,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction of a 1x3 Classroom block...,Chirwa,Education,Approved
3,4,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
4,5,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved
5,6,Proposed Construction and installation of 02 S...,Kaputula,Education,Approved
6,7,Proposed Construction and installation of 02 S...,Njanji,Education,Approved
7,8,Proposed Construction and installation of 02 S...,Waya,Health,Approved
8,9,Proposed Construction of an Ablution block at ...,Kaputula,Education,Approved
9,10,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved


In [5]:
# Validate the cleaned 2025 CDF dataset

print("Rows:", len(clean_2025_df))
print("Columns:", len(clean_2025_df.columns))

print("\nColumn names:")
print(clean_2025_df.columns.tolist())

print("\nMissing values:")
print(clean_2025_df.isna().sum())

print("\nEmpty text values:")
for column in ["project", "ward", "sector", "comment"]:
    empty_count = (
        clean_2025_df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    print(f"{column}: {empty_count}")

print("\nDuplicate rows:", clean_2025_df.duplicated().sum())

print("\nDuplicate project numbers:")
print(
    clean_2025_df[
        clean_2025_df["no"].duplicated(keep=False)
    ][["no", "project"]]
)

print("\nProject numbers present:")
print(sorted(clean_2025_df["no"].tolist()))

print("\nSector values:")
print(clean_2025_df["sector"].value_counts())

print("\nWard values:")
print(clean_2025_df["ward"].value_counts())

Rows: 33
Columns: 5

Column names:
['no', 'project', 'ward', 'sector', 'comment']

Missing values:
no         0
project    0
ward       0
sector     0
comment    0
dtype: int64

Empty text values:
project: 0
ward: 0
sector: 0
comment: 0

Duplicate rows: 0

Duplicate project numbers:
Empty DataFrame
Columns: [no, project]
Index: []

Project numbers present:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]

Sector values:
sector
Education               16
Health                   8
Commerce and Trade       4
Water and Sanitation     3
Transport                2
Name: count, dtype: int64

Ward values:
ward
Waya             5
Mpima            4
Kalonga          3
Kaputula         3
ALL              3
Njanji           2
Katondo          2
Nakoli           2
Luangwa          2
David Ramusho    2
High ridge       1
Chirwa           1
Justin Kabwe     1
Justine Kabwe    1
Lukanga          1
Name: count, dtype: int64


In [6]:
# Add metadata to the verified 2025 CDF dataset

final_2025_df = clean_2025_df.copy()

final_2025_df.insert(1, "year", 2025)
final_2025_df.insert(2, "constituency", "Kabwe Central")

final_2025_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/08/"
    "Proposed-2025-CDF-projects.pdf"
)

final_2025_df["source_document"] = (
    "Proposed 2025 CDF Projects"
)

# Display the final structure
display(final_2025_df)

print("Rows:", len(final_2025_df))
print("Columns:", len(final_2025_df.columns))

,no,year,constituency,project,ward,sector,comment,source_url,source_document
0,1,2025,Kabwe Central,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
1,2,2025,Kabwe Central,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
2,3,2025,Kabwe Central,Proposed Construction of a 1x3 Classroom block...,Chirwa,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
3,4,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
4,5,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
5,6,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Kaputula,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
6,7,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Njanji,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
7,8,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Waya,Health,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
8,9,2025,Kabwe Central,Proposed Construction of an Ablution block at ...,Kaputula,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
9,10,2025,Kabwe Central,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects


Rows: 33
Columns: 9


In [7]:
# Final quality checks for the 2025 CDF dataset

print("Rows:", len(final_2025_df))
print("Columns:", len(final_2025_df.columns))

print("\nColumns:")
print(final_2025_df.columns.tolist())

print("\nMissing values:")
print(final_2025_df.isna().sum())

print("\nDuplicate rows:", final_2025_df.duplicated().sum())

print("\nProject numbers complete:")
expected_numbers = list(range(1, 34))
actual_numbers = final_2025_df["no"].tolist()

print(actual_numbers == expected_numbers)

print("\nYears:")
print(final_2025_df["year"].unique())

print("\nConstituency:")
print(final_2025_df["constituency"].unique())

print("\nWard names:")
print(sorted(final_2025_df["ward"].unique()))

print("\nPotentially inconsistent ward names:")
print(
    final_2025_df[
        final_2025_df["ward"].str.contains(
            "Justin|Justine",
            case=False,
            na=False
        )
    ][["no", "project", "ward"]]
)

Rows: 33
Columns: 9

Columns:
['no', 'year', 'constituency', 'project', 'ward', 'sector', 'comment', 'source_url', 'source_document']

Missing values:
no                 0
year               0
constituency       0
project            0
ward               0
sector             0
comment            0
source_url         0
source_document    0
dtype: int64

Duplicate rows: 0

Project numbers complete:
True

Years:
[2025]

Constituency:
<StringArray>
['Kabwe Central']
Length: 1, dtype: str

Ward names:
['ALL', 'Chirwa', 'David Ramusho', 'High ridge', 'Justin Kabwe', 'Justine Kabwe', 'Kalonga', 'Kaputula', 'Katondo', 'Luangwa', 'Lukanga', 'Mpima', 'Nakoli', 'Njanji', 'Waya']

Potentially inconsistent ward names:
    no                                            project           ward
18  19              Additional funding for Kasanda Market   Justin Kabwe
20  21  Proposed Completion of a 1x2 Science Laborator...  Justine Kabwe


In [8]:
# Export the final 2025 CDF dataset as a pipe-delimited CSV

output_path = "../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2025.csv"

final_2025_df.to_csv(
    output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("CSV exported successfully.")
print("File:", output_path)
print("Rows:", len(final_2025_df))
print("Columns:", len(final_2025_df.columns))

CSV exported successfully.
File: ../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2025.csv
Rows: 33
Columns: 9


In [9]:
# Verify the exported CSV

check_df = pd.read_csv(
    output_path,
    sep="|"
)

print("Rows:", len(check_df))
print("Columns:", len(check_df.columns))
print("\nColumns:")
print(check_df.columns.tolist())

print("\nFirst 5 rows:")
display(check_df.head())

Rows: 33
Columns: 9

Columns:
['no', 'year', 'constituency', 'project', 'ward', 'sector', 'comment', 'source_url', 'source_document']

First 5 rows:


,no,year,constituency,project,ward,sector,comment,source_url,source_document
0,1,2025,Kabwe Central,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
1,2,2025,Kabwe Central,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
2,3,2025,Kabwe Central,Proposed Construction of a 1x3 Classroom block...,Chirwa,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
3,4,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
4,5,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects


In [10]:
# Inspect the combined 2024 CDF projects dataset

print("Rows:", len(kabwe_cdf_projects))
print("Columns:", len(kabwe_cdf_projects.columns))

print("\nColumns:")
print(kabwe_cdf_projects.columns.tolist())

print("\nFirst 5 rows:")
display(kabwe_cdf_projects.head())

print("\nConstituencies:")
print(kabwe_cdf_projects["constituency"].value_counts())

print("\nYears:")
print(kabwe_cdf_projects["year"].value_counts())

NameError: name 'kabwe_cdf_projects' is not defined

In [11]:
import os

bwacha_path = "../data/raw/2024_bwacha_cdf_projects.pdf"
central_path = "../data/raw/2024_kabwe_central_cdf_projects.pdf"

print("Bwacha PDF exists:", os.path.exists(bwacha_path))
print("Kabwe Central PDF exists:", os.path.exists(central_path))

if os.path.exists(bwacha_path):
    print("Bwacha size:", os.path.getsize(bwacha_path), "bytes")

if os.path.exists(central_path):
    print("Kabwe Central size:", os.path.getsize(central_path), "bytes")

Bwacha PDF exists: True
Kabwe Central PDF exists: True
Bwacha size: 93259 bytes
Kabwe Central size: 96930 bytes


In [12]:
import pdfplumber
import pandas as pd

# Extract tables from the 2024 Bwacha CDF PDF

all_rows_bwacha = []

with pdfplumber.open(bwacha_path) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        print(f"Page {page_number}: {len(tables)} table(s) found")

        for table in tables:
            for row in table:
                if row:
                    all_rows_bwacha.append(row)

# Keep rows whose first cell is a project number
project_rows_bwacha = []

for row in all_rows_bwacha:
    if (
        row[0] is not None
        and str(row[0]).strip().isdigit()
    ):
        project_rows_bwacha.append(row)

print("\nProject rows extracted:", len(project_rows_bwacha))

# Create DataFrame
bwacha_df = pd.DataFrame(
    project_rows_bwacha,
    columns=[
        "project_number",
        "project_name",
        "project_description",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status"
    ]
)

display(bwacha_df.head())

Number of pages: 5
Page 1: 1 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 1 table(s) found

Project rows extracted: 36


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,


In [13]:
print("Rows:", len(bwacha_df))
print("Columns:", len(bwacha_df.columns))

print("\nColumn names:")
print(bwacha_df.columns.tolist())

print("\nFirst 10 rows:")
display(bwacha_df.head(10))

print("\nLast 5 rows:")
display(bwacha_df.tail())

Rows: 36
Columns: 10

Column names:
['project_number', 'project_name', 'project_description', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']

First 10 rows:


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,
5,6,Construction of a Primary\nSchool in Kawama ward,Construction of a Primary\nSchool in Kawama ward,Kawama,Kawama ward,,,,,
6,7,Construction of 10 Teachers\nHouses at Chitaka...,Construction of 10 Teachers\nHouses at Chitaka...,Muwowo\nEast,Chitakata\nCommnuity\nSchool,,,,,
7,8,Construction of 1x4 Classroom\nBlock at Mukobe...,Construction of 1x4 Classroom\nBlock at Mukobe...,Muwowo\nEast,Mukobeko\nCorrectional\nDay\nSecondary\nSchool,,,,,
8,9,Construction of Youth\nResource Centre,Construction of Youth\nResource Centre,Bwacha,Bwacha ward,,,,,
9,10,Construction of a School Hall at\nRapheal Komb...,Construction of a School Hall\nat Rapheal Komb...,Chimaniman\ni,Rapheal\nKombe\nSecondary\nSchool,,,,,



Last 5 rows:


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
31,5,Procurement of 8 Skip Bins in\nNgungu ward,Procurement of 8 Skip Bins in\nNgungu ward,Ngungu,Ngungu ward,,,,,
32,1,Rehabilitation of Muleya\nStadium,Rehabilitation of Muleya\nStadium,Bwacha,Bwacha ward,,,,,
33,1,Procurement of 2 Megaphones\nand 500 Garden Ch...,Procurement of 2\nMegaphones and 500 Garden\nC...,Chimaniman\ni,Chimanimani\nward,,,,,
34,2,Procurement of 4 Tents for\nFuneral Occassions...,Procurement of 4 Tents for\nFuneral Occassions...,Kawama,Kawama ward,,,,,
35,3,Construction of Council Office\n(For Councilor...,Construction of Council Office\n(For Councilor...,Kawama,Kawama ward,,,,,


In [14]:
pd.set_option("display.max_colwidth", 80)

display(
    bwacha_df[
        [
            "project_number",
            "project_name",
            "ward",
            "application_amount",
            "engineers_estimate",
            "approved_amount",
            "contract_amount",
            "status"
        ]
    ].head(10)
)

,project_number,project_name,ward,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward,Kangomba,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool,Kangomba,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool,Kangomba,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine Primary School,Kangomba,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of Toilets for Pre-\nSchool Pupils...",Kangomba,,,,,
5,6,Construction of a Primary\nSchool in Kawama ward,Kawama,,,,,
6,7,Construction of 10 Teachers\nHouses at Chitakata\nCommunity School,Muwowo\nEast,,,,,
7,8,Construction of 1x4 Classroom\nBlock at Mukobeko\nCorrectional Day Secondary...,Muwowo\nEast,,,,,
8,9,Construction of Youth\nResource Centre,Bwacha,,,,,
9,10,Construction of a School Hall at\nRapheal Kombe Secondary\nSchool,Chimaniman\ni,,,,,


In [15]:
# Make a copy so the original extracted dataframe remains unchanged
clean_bwacha_df = bwacha_df.copy()

# Text columns extracted from the PDF
text_columns = [
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "status"
]

# Remove line breaks, tabs, repeated spaces, and surrounding whitespace
for column in text_columns:
    clean_bwacha_df[column] = (
        clean_bwacha_df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("Rows:", len(clean_bwacha_df))
print("Columns:", len(clean_bwacha_df.columns))

display(clean_bwacha_df.head(10))

Rows: 36
Columns: 10


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - K...,Kangomba,Kangomba ward,,,,,
1,2,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,
2,3,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,
3,4,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,,,,,
4,5,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...",Kangomba,Mary Chidgey Community School,,,,,
5,6,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,,,,,
6,7,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,,,,,
7,8,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,Muwowo East,Mukobeko Correctional Day Secondary School,,,,,
8,9,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,,,,,
9,10,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,,,,,


In [16]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

def clean_amount(value):
    value = str(value).strip()

    # Treat empty/missing values as missing data
    if value == "" or value.lower() in ["nan", "none", "n/a", "na", "-"]:
        return pd.NA

    # Remove currency symbols, commas, spaces and other non-numeric characters
    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "":
        return pd.NA

    return float(value)


for column in financial_columns:
    clean_bwacha_df[column] = clean_bwacha_df[column].apply(clean_amount)


print("Financial columns cleaned.")

display(
    clean_bwacha_df[
        [
            "project_number",
            "project_name",
            "application_amount",
            "engineers_estimate",
            "approved_amount",
            "contract_amount"
        ]
    ].head(10)
)

Financial columns cleaned.


,project_number,project_name,application_amount,engineers_estimate,approved_amount,contract_amount
0,1,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,<NA>,<NA>,<NA>,<NA>
1,2,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,<NA>,<NA>,<NA>
2,3,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,<NA>,<NA>,<NA>
3,4,Construction of 1X3 Classroom Block at Mine Primary School,<NA>,<NA>,<NA>,<NA>
4,5,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...",<NA>,<NA>,<NA>,<NA>
5,6,Construction of a Primary School in Kawama ward,<NA>,<NA>,<NA>,<NA>
6,7,Construction of 10 Teachers Houses at Chitakata Community School,<NA>,<NA>,<NA>,<NA>
7,8,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,<NA>,<NA>,<NA>,<NA>
8,9,Construction of Youth Resource Centre,<NA>,<NA>,<NA>,<NA>
9,10,Construction of a School Hall at Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>


In [17]:
print("Missing financial values:")
print(clean_bwacha_df[financial_columns].isna().sum())

print("\nFinancial data types:")
print(clean_bwacha_df[financial_columns].dtypes)

Missing financial values:
application_amount    36
engineers_estimate    36
approved_amount       36
contract_amount       36
dtype: int64

Financial data types:
application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object


In [18]:
print("Project numbers:")
print(clean_bwacha_df["project_number"].tolist())

print("\nNumber of projects:", len(clean_bwacha_df))

expected_numbers = list(range(1, len(clean_bwacha_df) + 1))

print(
    "Project numbers sequential:",
    clean_bwacha_df["project_number"].astype(int).tolist() == expected_numbers
)

Project numbers:
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '1', '2', '1', '1', '1', '2', '3', '4', '5', '6', '7', '8', '9', '1', '2', '1', '2', '3', '4', '5', '1', '1', '2', '3']

Number of projects: 36
Project numbers sequential: False


In [19]:
print("Duplicate complete rows:",
      clean_bwacha_df.duplicated().sum())

print("Duplicate project numbers:",
      clean_bwacha_df["project_number"].duplicated().sum())

Duplicate complete rows: 0
Duplicate project numbers: 24


In [20]:
for column in text_columns:
    empty_count = (
        clean_bwacha_df[column]
        .isna()
        .sum()
        +
        (clean_bwacha_df[column].astype("string").str.strip() == "").sum()
    )

    print(f"{column}: {empty_count} empty/missing")

project_name: 0 empty/missing
project_description: 0 empty/missing
ward: 0 empty/missing
project_site: 1 empty/missing
status: 36 empty/missing


In [21]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

display(clean_bwacha_df)

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,
1,2,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,
2,3,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,
3,4,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,
4,5,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,
5,6,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,
6,7,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,
7,8,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,
8,9,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,
9,10,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,


In [22]:
final_bwacha_df = clean_bwacha_df.copy()

final_bwacha_df.insert(1, "year", 2024)
final_bwacha_df.insert(2, "constituency", "Bwacha")

final_bwacha_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

final_bwacha_df["source_document"] = (
    "2024 Bwacha CDF Community Projects Submission"
)

display(final_bwacha_df.head())

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [23]:
bwacha_columns = [
    "project_number",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url",
    "source_document"
]

In [24]:
final_bwacha_df["project_type"] = pd.NA

final_bwacha_df = final_bwacha_df[
    [
        "project_number",
        "year",
        "constituency",
        "project_name",
        "project_description",
        "project_type",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status",
        "source_url",
        "source_document"
    ]
]

display(final_bwacha_df.head())

,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [25]:
import pdfplumber
import pandas as pd

kabwe_central_path = "../data/raw/2024_kabwe_central_cdf_projects.pdf"

all_rows_central = []

with pdfplumber.open(kabwe_central_path) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        print(
            f"Page {page_number}: "
            f"{len(tables)} table(s) found"
        )

        for table in tables:
            for row in table:
                if row:
                    all_rows_central.append(row)

project_rows_central = []

for row in all_rows_central:
    if (
        row[0] is not None
        and str(row[0]).strip().isdigit()
    ):
        project_rows_central.append(row)

print(
    "\nProject rows extracted:",
    len(project_rows_central)
)

Number of pages: 2
Page 1: 1 table(s) found
Page 2: 3 table(s) found

Project rows extracted: 43


In [26]:
central_df = pd.DataFrame(
    project_rows_central,
    columns=[
        "project_number",
        "project_name",
        "project_description",
        "project_type",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status"
    ]
)

print("Rows:", len(central_df))
print("Columns:", len(central_df.columns))

print("\nColumns:")
print(central_df.columns.tolist())

display(central_df.head(10))

Rows: 43
Columns: 11

Columns:
['project_number', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']


,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction,Luangwa,Kabwe Trust\nSecondary School,,,,,
1,2,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Luangwa,Kabwe Central\nHospital Special\nSchool Community,,,,,
2,3,Construction of 1X2\nClassroom Block,Construction of 1X2\nClassroom Block,Construction,Luangwa,Kabwe Trust Primary\nSchool,,,,,
3,4,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,
4,5,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima C,,,,,
5,6,Construction of a School,Construction of a School,Construction,Mpima,Kamuchanga,,,,,
6,7,Construction of 1X5\nClassroom Block and Wall\nfence,Construction of 1X5\nClassroom Block and Wall\nfence,Construction,Mpima,Nabusanga Zone\n(Primary School),,,,,
7,8,Construction of 1X3\nClassroom Block at Neem\nTress Secondary School,Construction of 1X3\nClassroom Block at Neem\nTress Secondary School,Construction,Lukanga,Neem Tree\nSecondary School,,,,,
8,9,Construction of 1X4\nClassroom Block at David\nRamushu Combined\nSchool,Construction of 1X4\nClassroom Block at David\nRamushu Combined School,Construction,D/Ramushu,David Ramushu\nCombined School,,,,,
9,10,Procurement of Laboratory\nRequirements at St.\nDominic Savio Secondary\nSchool,Procurement of Laboratory\nRequirements at St. Dominic\nSavio Secondary School,Procurement,J/Kabwe,St. Dominic Savio\nSecondary School,,,,,


In [27]:
clean_central_df = central_df.copy()

central_text_columns = [
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "status"
]

for column in central_text_columns:
    clean_central_df[column] = (
        clean_central_df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [28]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    clean_central_df[column] = (
        clean_central_df[column].apply(clean_amount)
    )

In [29]:
print("Rows:", len(clean_central_df))
print("Columns:", len(clean_central_df.columns))

print("\nMissing values:")
print(clean_central_df.isna().sum())

print("\nDuplicate complete rows:",
      clean_central_df.duplicated().sum())

print("\nDuplicate project numbers:",
      clean_central_df["project_number"].duplicated().sum())

print("\nProject numbers:")
print(clean_central_df["project_number"].tolist())

print("\nProject types:")
print(clean_central_df["project_type"].value_counts(dropna=False))

print("\nWards:")
print(clean_central_df["ward"].value_counts(dropna=False))

print("\nStatuses:")
print(clean_central_df["status"].value_counts(dropna=False))

Rows: 43
Columns: 11

Missing values:
project_number          0
project_name            0
project_description     0
project_type            0
ward                    0
project_site            0
application_amount     43
engineers_estimate     43
approved_amount        43
contract_amount        43
status                  0
dtype: int64

Duplicate complete rows: 0

Duplicate project numbers: 17

Project numbers:
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '1', '2', '3', '4', '5', '6', '7', '8', '1', '1', '2', '1', '2', '3', '4', '5', '1']

Project types:
project_type
Construction      34
Procurement        2
Rehabilitation     2
Water Articula     1
Construction/      1
Provision          1
Construction,      1
Enhancement        1
Name: count, dtype: Int64

Wards:
ward
Nakoli                     9
Kaputula                   6
Mpima                      5
Lukanga                    4
C

In [30]:
display(clean_central_df)

,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary School 1X4 Classroom Block,Construction of Secondary School 1X4 Classroom Block,Construction,Luangwa,Kabwe Trust Secondary School,<NA>,<NA>,<NA>,<NA>,
1,2,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Luangwa,Kabwe Central Hospital Special School Community,<NA>,<NA>,<NA>,<NA>,
2,3,Construction of 1X2 Classroom Block,Construction of 1X2 Classroom Block,Construction,Luangwa,Kabwe Trust Primary School,<NA>,<NA>,<NA>,<NA>,
3,4,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima Dairy Scheme,<NA>,<NA>,<NA>,<NA>,
4,5,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima C,<NA>,<NA>,<NA>,<NA>,
5,6,Construction of a School,Construction of a School,Construction,Mpima,Kamuchanga,<NA>,<NA>,<NA>,<NA>,
6,7,Construction of 1X5 Classroom Block and Wall fence,Construction of 1X5 Classroom Block and Wall fence,Construction,Mpima,Nabusanga Zone (Primary School),<NA>,<NA>,<NA>,<NA>,
7,8,Construction of 1X3 Classroom Block at Neem Tress Secondary School,Construction of 1X3 Classroom Block at Neem Tress Secondary School,Construction,Lukanga,Neem Tree Secondary School,<NA>,<NA>,<NA>,<NA>,
8,9,Construction of 1X4 Classroom Block at David Ramushu Combined School,Construction of 1X4 Classroom Block at David Ramushu Combined School,Construction,D/Ramushu,David Ramushu Combined School,<NA>,<NA>,<NA>,<NA>,
9,10,Procurement of Laboratory Requirements at St. Dominic Savio Secondary School,Procurement of Laboratory Requirements at St. Dominic Savio Secondary School,Procurement,J/Kabwe,St. Dominic Savio Secondary School,<NA>,<NA>,<NA>,<NA>,


In [31]:
final_central_df = clean_central_df.copy()

final_central_df.insert(1, "year", 2024)
final_central_df.insert(2, "constituency", "Kabwe Central")

final_central_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Community-Projects-Kabwe-Central-received.pdf"
)

final_central_df["source_document"] = (
    "2024 Kabwe Central CDF Community Projects Submission"
)

In [32]:
common_columns = [
    "project_number",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url",
    "source_document"
]

final_bwacha_df = final_bwacha_df[common_columns]
final_central_df = final_central_df[common_columns]

print("Bwacha columns match:",
      final_bwacha_df.columns.tolist() == common_columns)

print("Kabwe Central columns match:",
      final_central_df.columns.tolist() == common_columns)

Bwacha columns match: True
Kabwe Central columns match: True


In [33]:
cdf_2024_df = pd.concat(
    [
        final_bwacha_df,
        final_central_df
    ],
    ignore_index=True
)

print("Total 2024 projects:", len(cdf_2024_df))
print("Total columns:", len(cdf_2024_df.columns))

display(cdf_2024_df.head(10))

Total 2024 projects: 79
Total columns: 15


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,<NA>,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,<NA>,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,<NA>,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,<NA>,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,<NA>,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [34]:
print("===== DATASET VALIDATION =====")

print("\nRows:")
print(len(cdf_2024_df))

print("\nColumns:")
print(len(cdf_2024_df.columns))

print("\nColumn names:")
print(cdf_2024_df.columns.tolist())

print("\nMissing values:")
print(cdf_2024_df.isna().sum())

print("\nDuplicate complete rows:")
print(cdf_2024_df.duplicated().sum())

print("\nDuplicate project numbers within constituency:")

duplicates = cdf_2024_df.duplicated(
    subset=["year", "constituency", "project_number"]
).sum()

print(duplicates)

print("\nProjects by constituency:")
print(cdf_2024_df["constituency"].value_counts())

print("\nProjects by status:")
print(cdf_2024_df["status"].value_counts(dropna=False))

print("\nProjects by ward:")
print(cdf_2024_df["ward"].value_counts(dropna=False))

===== DATASET VALIDATION =====

Rows:
79

Columns:
15

Column names:
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']

Missing values:
project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            0
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                  0
source_url              0
source_document         0
dtype: int64

Duplicate complete rows:
0

Duplicate project numbers within constituency:
41

Projects by constituency:
constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

Projects by status:
status
    79
Name: count, dtype: Int64

Projects by ward:

In [35]:
print("Financial data types:")

print(
    cdf_2024_df[
        [
            "application_amount",
            "engineers_estimate",
            "approved_amount",
            "contract_amount"
        ]
    ].dtypes
)

Financial data types:
application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object


In [3]:
display(
    cdf_2024_df[
        [
            "project_number",
            "constituency",
            "project_name",
            "application_amount",
            "engineers_estimate",
            "approved_amount",
            "contract_amount"
        ]
    ]
)

NameError: name 'cdf_2024_df' is not defined

In [37]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

display(cdf_2024_df)

,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - K...,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,<NA>,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,<NA>,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,<NA>,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,<NA>,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,<NA>,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission


In [4]:
# ============================================================
# 2024 BWACHA CDF PROJECTS
# ============================================================

bwacha_path = "../data/raw/2024_bwacha_cdf_projects.pdf"

all_rows_bwacha = []

with pdfplumber.open(bwacha_path) as pdf:
    print("Bwacha PDF pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        print(
            f"Page {page_number}: "
            f"{len(tables)} table(s) found"
        )

        for table in tables:
            for row in table:
                if row:
                    all_rows_bwacha.append(row)


project_rows_bwacha = []

for row in all_rows_bwacha:
    if (
        row[0] is not None
        and str(row[0]).strip().isdigit()
    ):
        project_rows_bwacha.append(row)


print(
    "\nBwacha project rows extracted:",
    len(project_rows_bwacha)
)


bwacha_df = pd.DataFrame(
    project_rows_bwacha,
    columns=[
        "project_number",
        "project_name",
        "project_description",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status"
    ]
)

print("Bwacha rows:", len(bwacha_df))
print("Bwacha columns:", len(bwacha_df.columns))

display(bwacha_df.head())

Bwacha PDF pages: 5
Page 1: 1 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 1 table(s) found

Bwacha project rows extracted: 36
Bwacha rows: 36
Bwacha columns: 10


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,


In [5]:
clean_bwacha_df = bwacha_df.copy()

text_columns = [
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "status"
]

for column in text_columns:
    clean_bwacha_df[column] = (
        clean_bwacha_df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]


def clean_amount(value):
    value = str(value).strip()

    if value == "" or value.lower() in [
        "nan",
        "none",
        "n/a",
        "na",
        "-"
    ]:
        return pd.NA

    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "":
        return pd.NA

    return float(value)


for column in financial_columns:
    clean_bwacha_df[column] = (
        clean_bwacha_df[column].apply(clean_amount)
    )


print("Bwacha cleaning completed.")
display(clean_bwacha_df.head())

Bwacha cleaning completed.


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,
1,2,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,
2,3,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,
3,4,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,
4,5,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,


In [6]:
final_bwacha_df = clean_bwacha_df.copy()

final_bwacha_df.insert(1, "year", 2024)
final_bwacha_df.insert(2, "constituency", "Bwacha")

# Bwacha source does not provide project_type
final_bwacha_df["project_type"] = pd.NA

final_bwacha_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

final_bwacha_df["source_document"] = (
    "2024 Bwacha CDF Community Projects Submission"
)


common_columns = [
    "project_number",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url",
    "source_document"
]

final_bwacha_df = final_bwacha_df[common_columns]

print("Final Bwacha dataset:")
print("Rows:", len(final_bwacha_df))
print("Columns:", len(final_bwacha_df.columns))

display(final_bwacha_df.head())

Final Bwacha dataset:
Rows: 36
Columns: 15


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission


In [7]:
# ============================================================
# 2024 KABWE CENTRAL CDF PROJECTS
# ============================================================

kabwe_central_path = (
    "../data/raw/2024_kabwe_central_cdf_projects.pdf"
)

all_rows_central = []

with pdfplumber.open(kabwe_central_path) as pdf:
    print("Kabwe Central PDF pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        print(
            f"Page {page_number}: "
            f"{len(tables)} table(s) found"
        )

        for table in tables:
            for row in table:
                if row:
                    all_rows_central.append(row)


project_rows_central = []

for row in all_rows_central:
    if (
        row[0] is not None
        and str(row[0]).strip().isdigit()
    ):
        project_rows_central.append(row)


print(
    "\nKabwe Central project rows extracted:",
    len(project_rows_central)
)


central_df = pd.DataFrame(
    project_rows_central,
    columns=[
        "project_number",
        "project_name",
        "project_description",
        "project_type",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status"
    ]
)

print("Kabwe Central rows:", len(central_df))
print("Kabwe Central columns:", len(central_df.columns))

display(central_df.head())

Kabwe Central PDF pages: 2
Page 1: 1 table(s) found
Page 2: 3 table(s) found

Kabwe Central project rows extracted: 43
Kabwe Central rows: 43
Kabwe Central columns: 11


,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary\nSchool 1X4 Classroo...,Construction of Secondary\nSchool 1X4 Classroo...,Construction,Luangwa,Kabwe Trust\nSecondary School,,,,,
1,2,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Luangwa,Kabwe Central\nHospital Special\nSchool Community,,,,,
2,3,Construction of 1X2\nClassroom Block,Construction of 1X2\nClassroom Block,Construction,Luangwa,Kabwe Trust Primary\nSchool,,,,,
3,4,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,
4,5,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima C,,,,,


In [8]:
clean_central_df = central_df.copy()

central_text_columns = [
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "status"
]

for column in central_text_columns:
    clean_central_df[column] = (
        clean_central_df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


for column in financial_columns:
    clean_central_df[column] = (
        clean_central_df[column].apply(clean_amount)
    )


print("Kabwe Central cleaning completed.")

display(clean_central_df.head())

Kabwe Central cleaning completed.


,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary School 1X4 Classroom...,Construction of Secondary School 1X4 Classroom...,Construction,Luangwa,Kabwe Trust Secondary School,<NA>,<NA>,<NA>,<NA>,
1,2,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Luangwa,Kabwe Central Hospital Special School Community,<NA>,<NA>,<NA>,<NA>,
2,3,Construction of 1X2 Classroom Block,Construction of 1X2 Classroom Block,Construction,Luangwa,Kabwe Trust Primary School,<NA>,<NA>,<NA>,<NA>,
3,4,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima Dairy Scheme,<NA>,<NA>,<NA>,<NA>,
4,5,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima C,<NA>,<NA>,<NA>,<NA>,


In [9]:
final_central_df = clean_central_df.copy()

final_central_df.insert(1, "year", 2024)
final_central_df.insert(2, "constituency", "Kabwe Central")

final_central_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Community-Projects-Kabwe-Central-received.pdf"
)

final_central_df["source_document"] = (
    "2024 Kabwe Central CDF Community Projects Submission"
)

final_central_df = final_central_df[common_columns]

print("Final Kabwe Central dataset:")
print("Rows:", len(final_central_df))
print("Columns:", len(final_central_df.columns))

display(final_central_df.head())

Final Kabwe Central dataset:
Rows: 43
Columns: 15


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Kabwe Central,Construction of Secondary School 1X4 Classroom...,Construction of Secondary School 1X4 Classroom...,Construction,Luangwa,Kabwe Trust Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...
1,2,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Luangwa,Kabwe Central Hospital Special School Community,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...
2,3,2024,Kabwe Central,Construction of 1X2 Classroom Block,Construction of 1X2 Classroom Block,Construction,Luangwa,Kabwe Trust Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...
3,4,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima Dairy Scheme,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...
4,5,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima C,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...


In [10]:
# ============================================================
# COMBINE 2024 CDF DATASETS
# ============================================================

cdf_2024_df = pd.concat(
    [
        final_bwacha_df,
        final_central_df
    ],
    ignore_index=True
)

print("===== 2024 KABWE CDF DATASET =====")
print("Total rows:", len(cdf_2024_df))
print("Total columns:", len(cdf_2024_df.columns))

print("\nProjects by constituency:")
print(cdf_2024_df["constituency"].value_counts())

display(cdf_2024_df.head(10))

===== 2024 KABWE CDF DATASET =====
Total rows: 79
Total columns: 15

Projects by constituency:
constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,<NA>,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakat...,Construction of 10 Teachers Houses at Chitakat...,<NA>,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobek...,Construction of 1x4 Classroom Block at Mukobek...,<NA>,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,<NA>,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe...,Construction of a School Hall at Rapheal Kombe...,<NA>,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission


In [11]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("========== FINAL VALIDATION ==========")

print("\n1. Dataset dimensions")
print("Rows:", len(cdf_2024_df))
print("Columns:", len(cdf_2024_df.columns))


print("\n2. Column names")
print(cdf_2024_df.columns.tolist())


print("\n3. Missing values")
print(cdf_2024_df.isna().sum())


print("\n4. Duplicate complete rows")
print(cdf_2024_df.duplicated().sum())


print("\n5. Duplicate project IDs within constituency")
duplicate_projects = cdf_2024_df.duplicated(
    subset=[
        "year",
        "constituency",
        "project_number"
    ]
).sum()

print(duplicate_projects)


print("\n6. Projects by constituency")
print(
    cdf_2024_df["constituency"]
    .value_counts()
)


print("\n7. Projects by ward")
print(
    cdf_2024_df["ward"]
    .value_counts(dropna=False)
)


print("\n8. Projects by status")
print(
    cdf_2024_df["status"]
    .value_counts(dropna=False)
)


print("\n9. Financial data types")
print(
    cdf_2024_df[
        financial_columns
    ].dtypes
)

========== FINAL VALIDATION ==========

1. Dataset dimensions
Rows: 79
Columns: 15

2. Column names
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']

3. Missing values
project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            0
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                  0
source_url              0
source_document         0
dtype: int64

4. Duplicate complete rows
0

5. Duplicate project IDs within constituency
41

6. Projects by constituency
constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

7. Projects by ward
ward
Nakoli            

In [12]:
output_path_2024 = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv"
)

cdf_2024_df.to_csv(
    output_path_2024,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("2024 dataset saved successfully:")
print(output_path_2024)

2024 dataset saved successfully:
../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2024.csv


In [13]:
check_2024_df = pd.read_csv(
    output_path_2024,
    sep="|"
)

print("========== EXPORT CHECK ==========")

print("Rows:", len(check_2024_df))
print("Columns:", len(check_2024_df.columns))

print(
    "Rows match:",
    len(check_2024_df) == len(cdf_2024_df)
)

print(
    "Columns match:",
    check_2024_df.columns.tolist()
    == cdf_2024_df.columns.tolist()
)

display(check_2024_df.head())

========== EXPORT CHECK ==========
Rows: 79
Columns: 15
Rows match: True
Columns match: True


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,NaN,Kangomba,Kangomba ward,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,NaN,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,NaN,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,NaN,Kangomba,Mine Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",NaN,Kangomba,Mary Chidgey Community School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission


In [14]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

display(cdf_2024_df)

,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - K...,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,<NA>,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,<NA>,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,<NA>,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,<NA>,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,<NA>,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 Bwacha CDF Community Projects Submission


In [15]:
budget_pdf_path = "../data/raw/2025_kabwe_obb_budget.pdf"

import os

print("File exists:", os.path.exists(budget_pdf_path))

if os.path.exists(budget_pdf_path):
    print(
        "File size:",
        round(os.path.getsize(budget_pdf_path) / 1024, 2),
        "KB"
    )

File exists: False


In [17]:
import requests
import os
import urllib3

# Suppress the warning caused by the council website's
# invalid/untrusted SSL certificate.
urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

budget_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

budget_pdf_path = (
    "../data/raw/2025_kabwe_obb_budget.pdf"
)

response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

if response.status_code == 200:
    with open(budget_pdf_path, "wb") as file:
        file.write(response.content)

    print("\nDownloaded successfully.")
    print(
        "File size:",
        round(
            os.path.getsize(budget_pdf_path) / 1024,
            2
        ),
        "KB"
    )
else:
    print("\nDownload failed.")
    print("Response:", response.text[:500])

Status code: 200
Content type: application/pdf

Downloaded successfully.
File size: 773.87 KB


In [18]:
print(
    "File exists:",
    os.path.exists(budget_pdf_path)
)

if os.path.exists(budget_pdf_path):
    print(
        "File size:",
        round(
            os.path.getsize(budget_pdf_path) / 1024,
            2
        ),
        "KB"
    )

File exists: True
File size: 773.87 KB


In [19]:
with open(budget_pdf_path, "rb") as file:
    first_bytes = file.read(20)

print("First bytes:", first_bytes)

if first_bytes.startswith(b"%PDF"):
    print("Valid PDF file.")
else:
    print("WARNING: Downloaded file does not appear to be a PDF.")

First bytes: b'%PDF-1.7\r\n%\xb5\xb5\xb5\xb5\r\n1 0'
Valid PDF file.


In [20]:
import fitz

budget_doc = fitz.open(budget_pdf_path)

print("Number of pages:", len(budget_doc))

# Extract text from all pages
budget_pages = []

for page_number, page in enumerate(budget_doc, start=1):
    text = page.get_text()

    budget_pages.append({
        "page": page_number,
        "text": text
    })

print("Pages processed:", len(budget_pages))

Number of pages: 54
Pages processed: 54


In [21]:
budget_keywords = [
    "CDF",
    "Community Projects",
    "Women and Youth",
    "Empowerment",
    "Bursaries",
    "CDF Administration",
    "Skills Development"
]

for keyword in budget_keywords:

    matches = []

    for item in budget_pages:
        if keyword.lower() in item["text"].lower():
            matches.append(item["page"])

    print(f"{keyword}: {matches[:30]}")

CDF: [6, 8, 9, 11, 12, 13, 48]
Community Projects: [8, 9, 11, 12, 13, 48]
Women and Youth: [8, 9, 11, 12, 13, 48]
Empowerment: [6, 8, 9, 11, 12, 13, 48]
Bursaries: [6, 8, 9, 11, 12, 13]
CDF Administration: [8, 9, 12]
Skills Development: [1, 6, 7, 8, 9, 10, 11, 12, 13, 27, 28, 48, 51]


In [22]:
cdf_pages = []

for item in budget_pages:
    if "cdf" in item["text"].lower():
        cdf_pages.append(item)

print("Pages containing CDF:", len(cdf_pages))

for item in cdf_pages[:10]:

    print(
        f"\n{'=' * 70}\n"
        f"PAGE {item['page']}\n"
        f"{'=' * 70}"
    )

    print(item["text"][:5000])

Pages containing CDF: 7

PAGE 6
OUTPUT BASED  ANNUAL BUDGET
Page 6
920
5
HEA
D
KABWE MUNICIPAL COUNCIL
The summary estimates by economic classification shows that K49.0 million representing 27.04 percent 
of the total budget has been allocated to Personal Emoluments to facilitate payments of salaries and 
wages. In addition, K52.2 million representing 28.83 percent of the total budget has been allocated 
towards Assets acquisition such as procurement of utility vehicles and office equipment, renovation of 
council chamber and implementation of various CDF infrastructure development. 
Further, K33.5 million representing 18.52 percent of the total budget has been channelled to the Use of 
Goods and Services to facilitate the operation of the programmes.  Furthermore, Grants and other 
payments (Transfers) has been allocated K34.9 million representing 19.24 percent of the budget to 
cover Youth and Women Empowerments Grants, Boarding Secondary School and Skills Development 
Bursaries as w

In [23]:
import pdfplumber

budget_tables = []

with pdfplumber.open(budget_pdf_path) as pdf:

    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(
        pdf.pages,
        start=1
    ):

        tables = page.extract_tables()

        if tables:

            print(
                f"Page {page_number}: "
                f"{len(tables)} table(s)"
            )

            for table in tables:

                budget_tables.append({
                    "page": page_number,
                    "table": table
                })

print(
    "\nTotal tables extracted:",
    len(budget_tables)
)

Number of pages: 54
Page 1: 7 table(s)
Page 2: 1 table(s)
Page 3: 1 table(s)
Page 4: 1 table(s)
Page 5: 3 table(s)
Page 7: 1 table(s)
Page 8: 1 table(s)
Page 9: 1 table(s)
Page 11: 1 table(s)
Page 12: 2 table(s)
Page 13: 1 table(s)
Page 14: 2 table(s)
Page 15: 2 table(s)
Page 16: 2 table(s)
Page 17: 2 table(s)
Page 18: 3 table(s)
Page 19: 1 table(s)
Page 20: 1 table(s)
Page 21: 2 table(s)
Page 22: 1 table(s)
Page 23: 1 table(s)
Page 24: 1 table(s)
Page 25: 2 table(s)
Page 26: 1 table(s)
Page 27: 2 table(s)
Page 28: 1 table(s)
Page 29: 2 table(s)
Page 30: 1 table(s)
Page 31: 1 table(s)
Page 32: 1 table(s)
Page 33: 1 table(s)
Page 34: 2 table(s)
Page 35: 1 table(s)
Page 36: 2 table(s)
Page 37: 2 table(s)
Page 38: 2 table(s)
Page 39: 1 table(s)
Page 40: 3 table(s)
Page 41: 3 table(s)
Page 43: 1 table(s)
Page 44: 2 table(s)
Page 45: 2 table(s)
Page 46: 2 table(s)
Page 47: 3 table(s)
Page 48: 1 table(s)
Page 54: 1 table(s)

Total tables extracted: 78


In [24]:
cdf_tables = []

for item in budget_tables:

    table = item["table"]

    table_text = " ".join(
        str(cell)
        for row in table
        for cell in row
        if cell is not None
    )

    if "cdf" in table_text.lower():

        cdf_tables.append(item)


print(
    "Tables containing CDF:",
    len(cdf_tables)
)

for item in cdf_tables[:20]:

    print(
        "Page:",
        item["page"]
    )

Tables containing CDF: 1
Page: 13


In [25]:
for item in cdf_tables:

    print(
        f"\n{'=' * 80}"
    )

    print(
        f"PAGE {item['page']}"
    )

    print(
        f"{'=' * 80}"
    )

    for row in item["table"]:
        print(row)


PAGE 13
['Key Output and Output Indicator', '2023', None, '2024', None, '2025']
[None, 'Target', 'Actual', 'Target', 'Actual*', 'Target']
['Community Projects Completed\n01 Number of desks to be procured\n02 Number of Maternity wings constructed\n03 Number of Ambulances to be procured', '(0)\n2\n-', '8,992\n2\n-', '1,200\n5\n2', '(0)\n2\n-', '2,500\n1\n-']
['District Roads Graded\n01 Kilometer of roads graded', '(0)', '(0)', '(0)', '(0)', '100']
['Community, women and youth empowered\n01 Number of youth groups empowered\n02 Number of women groups empowered', '10\n50', '6\n61', '15\n60', '10\n50', '20\n65']
['Empowerment loans disbursed\n01 Number of business entities accessing loans', '45', '51', '50', '45', '70']
['CDF activities administered\n01 Number of CDFC meetings held\n02 Number of project monitoring visits carried out\n03 Number of CDF projects branded', '4\n4\n(0)', '12\n4\n(0)', '6\n15\n(0)', '4\n4\n(0)', '6\n8\n39']
['Skills development beneficiaries sponsored\n01 Number o

In [26]:
cdf_indicators = [
    ["Community Projects Completed", "Number of desks to be procured", 2, 8992, 1200, 0, 2500],
    ["Community Projects Completed", "Number of Maternity wings constructed", pd.NA, 2, 5, 2, 1],
    ["Community Projects Completed", "Number of Ambulances to be procured", pd.NA, pd.NA, 2, pd.NA, pd.NA],

    ["District Roads Graded", "Kilometer of roads graded", pd.NA, pd.NA, pd.NA, pd.NA, 100],

    ["Community, women and youth empowered", "Number of youth groups empowered", 10, 6, 15, 10, 20],
    ["Community, women and youth empowered", "Number of women groups empowered", 50, 61, 60, 50, 65],

    ["Empowerment loans disbursed", "Number of business entities accessing loans", 45, 51, 50, 45, 70],

    ["CDF activities administered", "Number of CDFC meetings held", 4, 12, 6, 4, 6],
    ["CDF activities administered", "Number of project monitoring visits carried out", 4, 4, 15, 4, 8],
    ["CDF activities administered", "Number of CDF projects branded", pd.NA, pd.NA, pd.NA, pd.NA, 39],

    ["Skills development beneficiaries sponsored", "Number of beneficiaries trained under skill development", 500, 1039, 600, 500, 800],

    ["Secondary School boarding beneficiaries sponsored", "Number of beneficiaries trained under secondary boarding", 40, 33, 60, 40, 50],
]

cdf_indicators_df = pd.DataFrame(
    cdf_indicators,
    columns=[
        "indicator_category",
        "indicator",
        "2023_target",
        "2023_actual",
        "2024_target",
        "2024_actual",
        "2025_target"
    ]
)

cdf_indicators_df.insert(0, "year", 2025)

cdf_indicators_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

cdf_indicators_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget"
)

display(cdf_indicators_df)

,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2,8992,1200,0,2500,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,<NA>,2,5,2,1,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,<NA>,<NA>,<NA>,<NA>,100,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10,6,15,10,20,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50,61,60,50,65,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45,51,50,45,70,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4,12,6,4,6,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4,4,15,4,8,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,<NA>,<NA>,<NA>,<NA>,39,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [27]:
print("========== CDF INDICATOR VALIDATION ==========")

print("Rows:", len(cdf_indicators_df))
print("Columns:", len(cdf_indicators_df.columns))

print("\nColumns:")
print(cdf_indicators_df.columns.tolist())

print("\nMissing values:")
print(cdf_indicators_df.isna().sum())

print("\nDuplicate rows:")
print(cdf_indicators_df.duplicated().sum())

print("\nData types:")
print(cdf_indicators_df.dtypes)

========== CDF INDICATOR VALIDATION ==========
Rows: 12
Columns: 10

Columns:
['year', 'indicator_category', 'indicator', '2023_target', '2023_actual', '2024_target', '2024_actual', '2025_target', 'source_url', 'source_document']

Missing values:
year                  0
indicator_category    0
indicator             0
2023_target           4
2023_actual           3
2024_target           2
2024_actual           3
2025_target           1
source_url            0
source_document       0
dtype: int64

Duplicate rows:
0

Data types:
year                   int64
indicator_category       str
indicator                str
2023_target           object
2023_actual           object
2024_target           object
2024_actual           object
2025_target           object
source_url               str
source_document          str
dtype: object


In [28]:
indicator_output_path = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
)

cdf_indicators_df.to_csv(
    indicator_output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Saved:", indicator_output_path)

Saved: ../data/processed/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv


In [29]:
check_indicators_df = pd.read_csv(
    indicator_output_path,
    sep="|"
)

print("Rows:", len(check_indicators_df))
print("Columns:", len(check_indicators_df.columns))
print(check_indicators_df.columns.tolist())

display(check_indicators_df)

Rows: 12
Columns: 10
['year', 'indicator_category', 'indicator', '2023_target', '2023_actual', '2024_target', '2024_actual', '2025_target', 'source_url', 'source_document']


,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2.0,8992.0,1200.0,0.0,2500.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,NaN,2.0,5.0,2.0,1.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,NaN,NaN,2.0,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,NaN,NaN,NaN,NaN,100.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10.0,6.0,15.0,10.0,20.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50.0,61.0,60.0,50.0,65.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45.0,51.0,50.0,45.0,70.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4.0,12.0,6.0,4.0,6.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4.0,4.0,15.0,4.0,8.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,NaN,NaN,NaN,NaN,39.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [30]:
financial_keywords = [
    "Community Projects",
    "Women and Youth Empowerment",
    "CDF Administration",
    "Secondary School",
    "Skills Development",
    "72,116,301"
]

for keyword in financial_keywords:
    print(f"\n{'=' * 80}")
    print(f"SEARCHING FOR: {keyword}")
    print(f"{'=' * 80}")

    found = False

    for item in budget_pages:
        if keyword.lower() in item["text"].lower():
            print(f"\n--- PAGE {item['page']} ---")
            print(item["text"][:6000])
            found = True

    if not found:
        print("Not found in extracted text.")


SEARCHING FOR: Community Projects

--- PAGE 8 ---
OUTPUT BASED  ANNUAL BUDGET
Page 8
920
5
HEA
D
KABWE MUNICIPAL COUNCIL
Table  3: Budget Allocation by Programme and Sub-Programme
Approved
Approved 
2025 BUDGET
PROGRAMME/SUB-PROGRAMME
Expendit
ure*
2024 BUDGET
2023 BUDGET
Expendit
ure
 Estimate
Constituency Development
1
(0)
61,271,284
72,116,301
(0)
(0)
Community Projects 
779
(0)
34,924,632
43,925,383
(0)
(0)
Women and Youth Empowerment 
780
(0)
11,641,544
12,456,452
(0)
(0)
CDF Administration 
781
(0)
3,063,564
3,278,014
(0)
(0)
Secondary School and Skills Development Bursaries 
782
(0)
11,641,544
12,456,452
(0)
(0)
Local Governance
2
(0)
6,125,473
5,827,497
(0)
(0)
Legislative Functions 
003
(0)
3,767,535
3,349,182
(0)
(0)
Citizen Engagement 
043
(0)
2,357,938
2,478,315
(0)
(0)
Integrated Development Planning
3
(0)
6,342,899
5,387,125
(0)
(0)
Environmental Planning 
006
(0)
893,433
3,677,653
(0)
(0)
Spatial Planning 
021
(0)
5,449,465
1,709,472
(0)
(0)
Economic and Business Develo

In [31]:
for item in budget_tables:
    table_text = " ".join(
        str(cell)
        for row in item["table"]
        for cell in row
        if cell is not None
    )

    if any(
        keyword.lower() in table_text.lower()
        for keyword in [
            "community projects",
            "women and youth",
            "cdf administration",
            "bursaries"
        ]
    ):
        print(f"\n{'=' * 100}")
        print(f"PAGE {item['page']}")
        print(f"{'=' * 100}")

        for row in item["table"]:
            print(row)


PAGE 13
['Key Output and Output Indicator', '2023', None, '2024', None, '2025']
[None, 'Target', 'Actual', 'Target', 'Actual*', 'Target']
['Community Projects Completed\n01 Number of desks to be procured\n02 Number of Maternity wings constructed\n03 Number of Ambulances to be procured', '(0)\n2\n-', '8,992\n2\n-', '1,200\n5\n2', '(0)\n2\n-', '2,500\n1\n-']
['District Roads Graded\n01 Kilometer of roads graded', '(0)', '(0)', '(0)', '(0)', '100']
['Community, women and youth empowered\n01 Number of youth groups empowered\n02 Number of women groups empowered', '10\n50', '6\n61', '15\n60', '10\n50', '20\n65']
['Empowerment loans disbursed\n01 Number of business entities accessing loans', '45', '51', '50', '45', '70']
['CDF activities administered\n01 Number of CDFC meetings held\n02 Number of project monitoring visits carried out\n03 Number of CDF projects branded', '4\n4\n(0)', '12\n4\n(0)', '6\n15\n(0)', '4\n4\n(0)', '6\n8\n39']
['Skills development beneficiaries sponsored\n01 Number o

In [32]:
cdf_indicators = [
    ["Community Projects Completed", "Number of desks to be procured", 2, 8992, 1200, 0, 2500],
    ["Community Projects Completed", "Number of Maternity wings constructed", pd.NA, 2, 5, 2, 1],
    ["Community Projects Completed", "Number of Ambulances to be procured", pd.NA, pd.NA, 2, pd.NA, pd.NA],

    ["District Roads Graded", "Kilometer of roads graded", 0, 0, 0, 0, 100],

    ["Community, women and youth empowered", "Number of youth groups empowered", 10, 6, 15, 10, 20],
    ["Community, women and youth empowered", "Number of women groups empowered", 50, 61, 60, 50, 65],

    ["Empowerment loans disbursed", "Number of business entities accessing loans", 45, 51, 50, 45, 70],

    ["CDF activities administered", "Number of CDFC meetings held", 4, 12, 6, 4, 6],
    ["CDF activities administered", "Number of project monitoring visits carried out", 4, 4, 15, 4, 8],
    ["CDF activities administered", "Number of CDF projects branded", 0, 0, 0, 0, 39],

    ["Skills development beneficiaries sponsored", "Number of beneficiaries trained under skill development", 500, 1039, 600, 500, 800],

    ["Secondary School boarding beneficiaries sponsored", "Number of beneficiaries trained under secondary boarding", 40, 33, 60, 40, 50],
]

cdf_indicators_df = pd.DataFrame(
    cdf_indicators,
    columns=[
        "indicator_category",
        "indicator",
        "2023_target",
        "2023_actual",
        "2024_target",
        "2024_actual",
        "2025_target"
    ]
)

cdf_indicators_df.insert(0, "year", 2025)

cdf_indicators_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

cdf_indicators_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget"
)

display(cdf_indicators_df)

,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2,8992,1200,0,2500,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,<NA>,2,5,2,1,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0,0,0,0,100,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10,6,15,10,20,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50,61,60,50,65,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45,51,50,45,70,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4,12,6,4,6,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4,4,15,4,8,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0,0,0,0,39,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [33]:
print("========== INDICATOR DATASET VALIDATION ==========")

print("Rows:", len(cdf_indicators_df))
print("Columns:", len(cdf_indicators_df.columns))

print("\nColumn names:")
print(cdf_indicators_df.columns.tolist())

print("\nMissing values:")
print(cdf_indicators_df.isna().sum())

print("\nDuplicate rows:")
print(cdf_indicators_df.duplicated().sum())

print("\nIndicator categories:")
print(cdf_indicators_df["indicator_category"].value_counts())

print("\nDataset:")
display(cdf_indicators_df)

========== INDICATOR DATASET VALIDATION ==========
Rows: 12
Columns: 10

Column names:
['year', 'indicator_category', 'indicator', '2023_target', '2023_actual', '2024_target', '2024_actual', '2025_target', 'source_url', 'source_document']

Missing values:
year                  0
indicator_category    0
indicator             0
2023_target           2
2023_actual           1
2024_target           0
2024_actual           1
2025_target           1
source_url            0
source_document       0
dtype: int64

Duplicate rows:
0

Indicator categories:
indicator_category
Community Projects Completed                         3
CDF activities administered                          3
Community, women and youth empowered                 2
District Roads Graded                                1
Empowerment loans disbursed                          1
Skills development beneficiaries sponsored           1
Secondary School boarding beneficiaries sponsored    1
Name: count, dtype: int64

Dataset:


,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2,8992,1200,0,2500,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,<NA>,2,5,2,1,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0,0,0,0,100,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10,6,15,10,20,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50,61,60,50,65,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45,51,50,45,70,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4,12,6,4,6,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4,4,15,4,8,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0,0,0,0,39,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [34]:
indicator_output_path = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
)

cdf_indicators_df.to_csv(
    indicator_output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Saved:", indicator_output_path)

Saved: ../data/processed/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv


In [35]:
check_indicators_df = pd.read_csv(
    indicator_output_path,
    sep="|"
)

print("Rows:", len(check_indicators_df))
print("Columns:", len(check_indicators_df.columns))
print(check_indicators_df.columns.tolist())

display(check_indicators_df)

Rows: 12
Columns: 10
['year', 'indicator_category', 'indicator', '2023_target', '2023_actual', '2024_target', '2024_actual', '2025_target', 'source_url', 'source_document']


,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2.0,8992.0,1200,0.0,2500.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,NaN,2.0,5,2.0,1.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,NaN,NaN,2,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0.0,0.0,0,0.0,100.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10.0,6.0,15,10.0,20.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50.0,61.0,60,50.0,65.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45.0,51.0,50,45.0,70.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4.0,12.0,6,4.0,6.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4.0,4.0,15,4.0,8.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0.0,0.0,0,0.0,39.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [36]:
source_record = pd.DataFrame([
    {
        "id": "S004",
        "source_name": "2025 Kabwe Municipal Council OBB Budget",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/05/"
            "2025-KABWE-M-COUNCIL-OBB.pdf"
        ),
        "description": (
            "2025 Output Based Budget containing CDF budget allocations "
            "and key output/output indicator targets and actuals."
        )
    }
])

display(source_record)

,id,source_name,source_type,url,description
0,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and key output/ou...


In [37]:
source_inventory_df = pd.concat(
    [source_inventory_df, source_record],
    ignore_index=True
)

display(source_inventory_df)

NameError: name 'source_inventory_df' is not defined

In [38]:
source_inventory_path = "../sources/source_inventory.csv"

source_inventory_df = pd.read_csv(
    source_inventory_path
)

print("Existing sources:", len(source_inventory_df))
display(source_inventory_df)

Existing sources: 2


,source_id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm/,Official Kabwe Municipal Council website
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency


In [39]:
source_record = pd.DataFrame([
    {
        "id": "S004",
        "source_name": "2025 Kabwe Municipal Council OBB Budget",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/05/"
            "2025-KABWE-M-COUNCIL-OBB.pdf"
        ),
        "description": (
            "2025 Output Based Budget containing CDF budget allocations "
            "and key output/output indicator targets and actuals."
        )
    }
])

source_inventory_df = pd.concat(
    [source_inventory_df, source_record],
    ignore_index=True
)

display(source_inventory_df)

,source_id,source_name,source_type,url,description,id
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm/,Official Kabwe Municipal Council website,NaN
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency,NaN
2,NaN,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and key output/ou...,S004


In [40]:
source_inventory_df.to_csv(
    source_inventory_path,
    index=False,
    encoding="utf-8"
)

print("Source inventory updated successfully.")

Source inventory updated successfully.


In [42]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

BASE_URL = "https://www.kabwecouncil.gov.zm"

response = requests.get(
    BASE_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(BASE_URL, link["href"])

    links.append({
        "text": text,
        "url": url
    })

links_df = pd.DataFrame(links)

print("Links found:", len(links_df))
display(links_df.head())

Status code: 200
Links found: 122


,text,url
0,,https://www.kabwecouncil.gov.zm#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169


In [43]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

BASE_URL = "https://www.kabwecouncil.gov.zm"

response = requests.get(
    BASE_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(BASE_URL, link["href"])

    links.append({
        "text": text,
        "url": url
    })

links_df = pd.DataFrame(links)

print("Links found:", len(links_df))
display(links_df.head())

Status code: 200
Links found: 122


,text,url
0,,https://www.kabwecouncil.gov.zm#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169


In [52]:
financial_statement_links = links_df[
    links_df["text"].str.contains(
        "financial statement|audited",
        case=False,
        na=False
    )
]

print("Financial statement links found:", len(financial_statement_links))
display(financial_statement_links)

Financial statement links found: 0


,text,url


In [53]:
financial_statement_urls = links_df[
    links_df["url"].str.contains(
        "financial|audit",
        case=False,
        na=False
    )
]

print("Financial-related URLs found:", len(financial_statement_urls))
display(financial_statement_urls)

Financial-related URLs found: 0


,text,url


In [54]:
idp_links = links_df[
    links_df["text"].str.contains(
        "IDP|Integrated Development",
        case=False,
        na=False
    )
]

print("IDP links found:", len(idp_links))
display(idp_links)

IDP links found: 0


,text,url


In [55]:
idp_urls = links_df[
    links_df["url"].str.contains(
        "idp|integrated",
        case=False,
        na=False
    )
]

print("IDP-related URLs found:", len(idp_urls))
display(idp_urls)

IDP-related URLs found: 0


,text,url


In [56]:
idp_links = links_df[
    links_df["text"].str.contains(
        "IDP|Integrated Development",
        case=False,
        na=False
    )
]

print("IDP links found:", len(idp_links))
display(idp_links)

IDP links found: 0


,text,url


In [57]:
idp_urls = links_df[
    links_df["url"].str.contains(
        "idp|integrated",
        case=False,
        na=False
    )
]

print("IDP-related URLs found:", len(idp_urls))
display(idp_urls)

IDP-related URLs found: 0


,text,url


In [58]:
publications_url = "https://www.kabwecouncil.gov.zm/?page_id=195"

response = requests.get(
    publications_url,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

publications_soup = BeautifulSoup(
    response.text,
    "html.parser"
)

publication_links = []

for link in publications_soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(
        publications_url,
        link["href"]
    )

    publication_links.append({
        "text": text,
        "url": url
    })

publications_df = pd.DataFrame(publication_links)

print("Publication links found:", len(publications_df))
display(publications_df)

Status code: 200
Publication links found: 161


,text,url
0,,https://www.kabwecouncil.gov.zm/?page_id=195#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm/?page_id=195#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169
5,Who we are,https://www.kabwecouncil.gov.zm/?page_id=118
6,Departments,https://www.kabwecouncil.gov.zm/?page_id=770
7,office of the the town clerk,https://www.kabwecouncil.gov.zm/?page_id=2634
8,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
9,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640


In [59]:
idp_publications = publications_df[
    publications_df["text"].str.contains(
        "IDP|Integrated Development|Development Plan",
        case=False,
        na=False
    )
]

print("IDP publications found:", len(idp_publications))
display(idp_publications)

IDP publications found: 2


,text,url
88,Kabwe District Integrated Development Plan (IDP) 2023 – 2028,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...
89,Kabwe District Integrated Development Plan (IDP) 2023 – 2033 Citizen Version,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-District-Ci...


In [60]:
idp_publication_urls = publications_df[
    publications_df["url"].str.contains(
        "idp|development",
        case=False,
        na=False
    )
]

print("IDP-related URLs found:", len(idp_publication_urls))
display(idp_publication_urls)

IDP-related URLs found: 6


,text,url
88,Kabwe District Integrated Development Plan (IDP) 2023 – 2028,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...
89,Kabwe District Integrated Development Plan (IDP) 2023 – 2033 Citizen Version,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-District-Ci...
90,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...
130,GUIDELINES ON THE ESTABLISHMENT OF WARD DEVELOPMENT COMMITTEES,https://www.kabwecouncil.gov.zm/wp-content/uploads/2023/12/GUIDELINES-ON-THE...
139,The Constituency Development Fund Act No.11 of 2018,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/11/The-Constituency-...
140,The local Government street vending nuisances Amendment No. 2 Regulation 2018,https://www.kabwecouncil.gov.zm/wp-contenthttps://www.kabwecouncil.gov.zm/wp...


In [62]:
idp_full = publications_df[
    publications_df["text"].str.contains(
        r"Kabwe District Integrated Development Plan \(IDP\) 2023",
        case=False,
        na=False
    )
]

display(idp_full[["text", "url"]])

,text,url
88,Kabwe District Integrated Development Plan (IDP) 2023 – 2028,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...
89,Kabwe District Integrated Development Plan (IDP) 2023 – 2033 Citizen Version,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-District-Ci...


In [63]:
idp_url = idp_full.iloc[0]["url"]

print("IDP URL:")
print(idp_url)

IDP URL:
https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-IDP_Final-Version-1.pdf


In [64]:
idp_pdf_path = (
    "../data/raw/"
    "2023_2028_kabwe_district_idp.pdf"
)

idp_response = requests.get(
    idp_url,
    timeout=120,
    verify=False
)

print("Status code:", idp_response.status_code)
print("Content-Type:", idp_response.headers.get("Content-Type"))
print("File size:", len(idp_response.content), "bytes")

if idp_response.status_code == 200:
    with open(idp_pdf_path, "wb") as file:
        file.write(idp_response.content)

    print("IDP downloaded successfully.")
else:
    print("Download failed.")

Status code: 200
Content-Type: application/pdf
File size: 17615491 bytes
IDP downloaded successfully.


In [65]:
import os

print("File exists:", os.path.exists(idp_pdf_path))

if os.path.exists(idp_pdf_path):
    print(
        "File size:",
        round(
            os.path.getsize(idp_pdf_path) / 1024,
            2
        ),
        "KB"
    )

    with open(idp_pdf_path, "rb") as file:
        first_bytes = file.read(20)

    print("First bytes:", first_bytes)

    if first_bytes.startswith(b"%PDF"):
        print("Valid PDF file.")
    else:
        print("WARNING: File does not appear to be a PDF.")

File exists: True
File size: 17202.63 KB
First bytes: b'%PDF-1.7\r\n%\xb5\xb5\xb5\xb5\r\n1 0'
Valid PDF file.


In [66]:
idp_doc = fitz.open(idp_pdf_path)

print("Number of pages:", len(idp_doc))

for page_number, page in enumerate(idp_doc, start=1):
    text = page.get_text()

    print(
        f"Page {page_number}: "
        f"{len(text)} characters"
    )

Number of pages: 354
Page 1: 261 characters
Page 2: 164 characters
Page 3: 452 characters
Page 4: 14 characters
Page 5: 1760 characters
Page 6: 1808 characters
Page 7: 1748 characters
Page 8: 2227 characters
Page 9: 2712 characters
Page 10: 2131 characters
Page 11: 3138 characters
Page 12: 475 characters
Page 13: 571 characters
Page 14: 4076 characters
Page 15: 878 characters
Page 16: 1005 characters
Page 17: 1262 characters
Page 18: 467 characters
Page 19: 2202 characters
Page 20: 1527 characters
Page 21: 2019 characters
Page 22: 604 characters
Page 23: 2005 characters
Page 24: 1057 characters
Page 25: 898 characters
Page 26: 712 characters
Page 27: 1646 characters
Page 28: 1220 characters
Page 29: 1218 characters
Page 30: 1159 characters
Page 31: 618 characters
Page 32: 1140 characters
Page 33: 137 characters
Page 34: 1244 characters
Page 35: 1631 characters
Page 36: 1003 characters
Page 37: 1478 characters
Page 38: 1000 characters
Page 39: 2202 characters
Page 40: 1128 characters
Pa

In [67]:
# Search the IDP for sections that are likely to contain
# useful structured data for our dataset.

idp_keywords = [
    "population",
    "ward",
    "project",
    "development",
    "sector",
    "health",
    "education",
    "road",
    "water",
    "sanitation",
    "budget",
    "revenue",
    "implementation",
    "indicator"
]

for keyword in idp_keywords:
    matching_pages = []

    for page_number, page in enumerate(idp_doc, start=1):
        text = page.get_text()

        if keyword.lower() in text.lower():
            matching_pages.append(page_number)

    print(
        f"{keyword}: "
        f"{len(matching_pages)} pages"
    )
    print("Pages:", matching_pages[:30])

population: 78 pages
Pages: [5, 9, 11, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 42, 43, 46, 54, 55, 56, 58, 59, 60, 62, 63, 64, 68, 70]
ward: 125 pages
Pages: [5, 6, 8, 9, 16, 21, 22, 23, 26, 28, 29, 32, 33, 38, 56, 58, 60, 69, 74, 78, 81, 82, 85, 87, 88, 90, 97, 104, 113, 124]
project: 39 pages
Pages: [3, 6, 7, 9, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 54, 55, 57, 83, 88, 90, 91, 92, 94, 109, 111, 128, 135]
development: 194 pages
Pages: [1, 2, 3, 5, 6, 7, 8, 9, 11, 12, 13, 14, 16, 17, 19, 20, 25, 26, 32, 35, 37, 39, 40, 43, 45, 48, 55, 57, 58, 62]
sector: 66 pages
Pages: [9, 11, 14, 15, 19, 27, 30, 35, 38, 47, 48, 51, 56, 60, 61, 63, 64, 65, 66, 68, 70, 77, 78, 80, 83, 84, 88, 89, 90, 93]
health: 93 pages
Pages: [10, 11, 16, 17, 25, 30, 32, 38, 39, 48, 57, 58, 60, 68, 88, 89, 90, 91, 92, 94, 96, 97, 112, 114, 115, 116, 117, 123, 124, 125]
education: 61 pages
Pages: [9, 10, 11, 14, 16, 17, 25, 30, 32, 36, 37, 39, 61, 88, 89, 91, 92, 97, 98, 99, 102, 103, 104, 10

In [68]:
# Inspect the IDP pages most likely to contain
# development projects, sector information and priorities.

for page_number in range(88, 95):
    page = idp_doc[page_number - 1]
    text = page.get_text()

    print("\n" + "=" * 100)
    print(f"PAGE {page_number}")
    print("=" * 100)
    print(text[:7000])


PAGE 88
 
 
70 
BWACHA 
MUWOWO EAST 
3,235 
670 
52.4 
BWACHA 
MUWOWO WEST 
1,953 
392 
86.5 
BWACHA 
NGUNGU 
6,068 
1,251 
21.1 
BWACHA 
ZAMBEZI 
3,785 
757 
96.2 
Source: ZAMSTAT, 20222 
 
An analysis by ward shows that over 90% of the households in Chililalila, Chinyanja, 
Kan’gomba, Makululu and Zambezi did not have access to electricity whereas only 
Bwacha, Chimanimani and Ngungu showed below 30% of households without access 
to electricity. The high percentage number of household without access to electricity 
denotes heavy reliance on unsustainable sources of energy such as charcoal and 
firewood which contributes to environmental degradation. 
 
3.7.6 
Issues Arising from The Public Participation Process 
Submissions from the Public participation process showed that rural and peri urban 
wards such as Chinyanja, Muwowo East and West, Mpima, Zambezi and Luansanse 
were not connected to the power grid and that the community relied on firewood and 
charcoal which was detrimental

In [69]:
# Search the IDP for pages containing likely project/programme tables.

keywords = [
    "Project List",
    "Projects",
    "Proposed Projects",
    "Capital Projects",
    "Development Projects",
    "Project Name",
    "Programme",
    "Projects and Programmes",
    "Implementation Plan"
]

for keyword in keywords:
    matches = []

    for page_number, page in enumerate(idp_doc, start=1):
        text = page.get_text()

        if keyword.lower() in text.lower():
            matches.append(page_number)

    print(f"{keyword}: {matches}")

Project List: []
Projects: [57, 111, 212, 351]
Proposed Projects: []
Capital Projects: []
Development Projects: []
Project Name: []
Programme: [15, 16, 38, 57, 62, 78, 81, 82, 90, 110, 114, 130, 131, 132, 146, 222, 232, 294, 296, 297, 351, 352]
Projects and Programmes: []
Implementation Plan: [11, 13, 47, 233, 249, 280]


In [70]:
cdf_indicators = [
    [
        "Community Projects Completed",
        "Number of desks to be procured",
        0, 8992, 1200, 0, 2500
    ],
    [
        "Community Projects Completed",
        "Number of Maternity wings constructed",
        2, 2, 5, 2, 1
    ],
    [
        "Community Projects Completed",
        "Number of Ambulances to be procured",
        pd.NA, pd.NA, 2, pd.NA, pd.NA
    ],
    [
        "District Roads Graded",
        "Kilometer of roads graded",
        0, 0, 0, 0, 100
    ],
    [
        "Community, women and youth empowered",
        "Number of youth groups empowered",
        10, 6, 15, 10, 20
    ],
    [
        "Community, women and youth empowered",
        "Number of women groups empowered",
        50, 61, 60, 50, 65
    ],
    [
        "Empowerment loans disbursed",
        "Number of business entities accessing loans",
        45, 51, 50, 45, 70
    ],
    [
        "CDF activities administered",
        "Number of CDFC meetings held",
        4, 12, 6, 4, 6
    ],
    [
        "CDF activities administered",
        "Number of project monitoring visits carried out",
        4, 4, 15, 4, 8
    ],
    [
        "CDF activities administered",
        "Number of CDF projects branded",
        0, 0, 0, 0, 39
    ],
    [
        "Skills development beneficiaries sponsored",
        "Number of beneficiaries trained under skill development",
        500, 1039, 600, 500, 800
    ],
    [
        "Secondary School boarding beneficiaries sponsored",
        "Number of beneficiaries trained under secondary boarding",
        40, 33, 60, 40, 50
    ]
]

cdf_indicators_df = pd.DataFrame(
    cdf_indicators,
    columns=[
        "output",
        "output_indicator",
        "target_2023",
        "actual_2023",
        "target_2024",
        "actual_2024",
        "target_2025"
    ]
)

cdf_indicators_df.insert(0, "year", 2025)

cdf_indicators_df["source_url"] = budget_url

cdf_indicators_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget"
)

cdf_indicators_df

,year,output,output_indicator,target_2023,actual_2023,target_2024,actual_2024,target_2025,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,0,8992,1200,0,2500,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,2,2,5,2,1,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0,0,0,0,100,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10,6,15,10,20,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50,61,60,50,65,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45,51,50,45,70,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4,12,6,4,6,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4,4,15,4,8,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0,0,0,0,39,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [71]:
print("========== OUTPUT INDICATOR VALIDATION ==========")

print("Rows:", len(cdf_indicators_df))
print("Columns:", len(cdf_indicators_df.columns))

print("\nColumn names:")
print(cdf_indicators_df.columns.tolist())

print("\nMissing values:")
print(cdf_indicators_df.isna().sum())

print("\nDuplicate rows:")
print(cdf_indicators_df.duplicated().sum())

print("\nOutputs:")
print(cdf_indicators_df["output"].value_counts())

print("\nYear:")
print(cdf_indicators_df["year"].unique())

========== OUTPUT INDICATOR VALIDATION ==========
Rows: 12
Columns: 10

Column names:
['year', 'output', 'output_indicator', 'target_2023', 'actual_2023', 'target_2024', 'actual_2024', 'target_2025', 'source_url', 'source_document']

Missing values:
year                0
output              0
output_indicator    0
target_2023         1
actual_2023         1
target_2024         0
actual_2024         1
target_2025         1
source_url          0
source_document     0
dtype: int64

Duplicate rows:
0

Outputs:
output
Community Projects Completed                         3
CDF activities administered                          3
Community, women and youth empowered                 2
District Roads Graded                                1
Empowerment loans disbursed                          1
Skills development beneficiaries sponsored           1
Secondary School boarding beneficiaries sponsored    1
Name: count, dtype: int64

Year:
[2025]


In [72]:
indicator_output_path = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
)

cdf_indicators_df.to_csv(
    indicator_output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Saved:", indicator_output_path)

Saved: ../data/processed/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv


In [73]:
check_indicators_df = pd.read_csv(
    indicator_output_path,
    sep="|"
)

print("Rows:", len(check_indicators_df))
print("Columns:", len(check_indicators_df.columns))
print(check_indicators_df.columns.tolist())

display(check_indicators_df)

Rows: 12
Columns: 10
['year', 'output', 'output_indicator', 'target_2023', 'actual_2023', 'target_2024', 'actual_2024', 'target_2025', 'source_url', 'source_document']


,year,output,output_indicator,target_2023,actual_2023,target_2024,actual_2024,target_2025,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,0.0,8992.0,1200,0.0,2500.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,2.0,2.0,5,2.0,1.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,NaN,NaN,2,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0.0,0.0,0,0.0,100.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10.0,6.0,15,10.0,20.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50.0,61.0,60,50.0,65.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45.0,51.0,50,45.0,70.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4.0,12.0,6,4.0,6.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4.0,4.0,15,4.0,8.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0.0,0.0,0,0.0,39.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [74]:
# ============================================================
# FINAL TASK 1 DATASET QUALITY CHECK
# ============================================================

import os
import pandas as pd

processed_dir = "../data/processed"

datasets = {
    "2024 CDF Projects": (
        "db-unza26-csc4792-kabwe_cdf_projects_2024.csv"
    ),
    "2025 Proposed CDF Projects": (
        "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
    ),
    "2024-2025 CDF Budget": (
        "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv"
    ),
    "2025 CDF Output Indicators": (
        "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
    )
}

print("=" * 70)
print("FINAL TASK 1 QUALITY CONTROL")
print("=" * 70)

quality_results = []

for dataset_name, filename in datasets.items():

    path = os.path.join(processed_dir, filename)

    print("\n" + "-" * 70)
    print(dataset_name)
    print("-" * 70)

    # Check file exists
    file_exists = os.path.exists(path)
    print("File exists:", file_exists)

    if not file_exists:
        print("WARNING: File not found.")
        continue

    # Read using required pipe delimiter
    df = pd.read_csv(path, sep="|")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("Columns:")
    print(df.columns.tolist())

    print("Duplicate rows:", df.duplicated().sum())

    print("Missing values:", df.isna().sum().sum())

    print("Index column present:",
          "Unnamed: 0" in df.columns)

    quality_results.append({
        "dataset": dataset_name,
        "filename": filename,
        "file_exists": file_exists,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "has_index_column": "Unnamed: 0" in df.columns
    })


quality_df = pd.DataFrame(quality_results)

print("\n" + "=" * 70)
print("QUALITY CONTROL SUMMARY")
print("=" * 70)

display(quality_df)

FINAL TASK 1 QUALITY CONTROL

----------------------------------------------------------------------
2024 CDF Projects
----------------------------------------------------------------------
File exists: True
Rows: 79
Columns: 15
Columns:
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']
Duplicate rows: 0
Missing values: 432
Index column present: False

----------------------------------------------------------------------
2025 Proposed CDF Projects
----------------------------------------------------------------------
File exists: True
Rows: 33
Columns: 9
Columns:
['no', 'year', 'constituency', 'project', 'ward', 'sector', 'comment', 'source_url', 'source_document']
Duplicate rows: 0
Missing values: 0
Index column present: False

---------------------------------------------------------------

,dataset,filename,file_exists,rows,columns,duplicate_rows,missing_values,has_index_column
0,2024 CDF Projects,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,True,79,15,0,432,False
1,2025 Proposed CDF Projects,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,True,33,9,0,0,False
2,2025 CDF Output Indicators,db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv,True,12,10,0,4,False


In [75]:
print("========== 2025 OUTPUT INDICATOR MISSING VALUES ==========")

indicator_missing = cdf_indicators_df[
    cdf_indicators_df.isna().any(axis=1)
]

display(indicator_missing)

========== 2025 OUTPUT INDICATOR MISSING VALUES ==========


,year,output,output_indicator,target_2023,actual_2023,target_2024,actual_2024,target_2025,source_url,source_document
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [76]:
print("Missing values by column:")

print(
    cdf_indicators_df.isna().sum()
)

Missing values by column:
year                0
output              0
output_indicator    0
target_2023         1
actual_2023         1
target_2024         0
actual_2024         1
target_2025         1
source_url          0
source_document     0
dtype: int64


In [77]:
budget_output_path = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv"
)

print("Budget file exists:", os.path.exists(budget_output_path))

Budget file exists: False


In [78]:
# ============================================================
# EXPORT 2024-2025 CDF BUDGET DATASET
# ============================================================

cdf_budget_data = [
    ["Constituency Development", 61271284, 72116301],
    ["Community Projects", 34924632, 43925383],
    ["Women and Youth Empowerment", 11641544, 12456452],
    ["CDF Administration", 3063564, 3278014],
    ["Secondary School and Skills Development Bursaries", 11641544, 12456452],
]

cdf_budget_df = pd.DataFrame(
    cdf_budget_data,
    columns=[
        "budget_category",
        "budget_2024",
        "budget_2025"
    ]
)

cdf_budget_df.insert(
    0,
    "council",
    "Kabwe Municipal Council"
)

cdf_budget_df.insert(
    1,
    "fund",
    "Constituency Development Fund (CDF)"
)

cdf_budget_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

cdf_budget_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget"
)

budget_output_path = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv"
)

cdf_budget_df.to_csv(
    budget_output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Saved:", budget_output_path)

Saved: ../data/processed/db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv


In [79]:
print("========== BUDGET VALIDATION ==========")

print("Rows:", len(cdf_budget_df))
print("Columns:", len(cdf_budget_df.columns))

print("\n2024 CDF total:",
      cdf_budget_df["budget_2024"].sum())

print("2025 CDF total:",
      cdf_budget_df["budget_2025"].sum())

print("\nDuplicate rows:",
      cdf_budget_df.duplicated().sum())

print("\nMissing values:")
print(cdf_budget_df.isna().sum())

========== BUDGET VALIDATION ==========
Rows: 5
Columns: 7

2024 CDF total: 122542568
2025 CDF total: 144232602

Duplicate rows: 0

Missing values:
council            0
fund               0
budget_category    0
budget_2024        0
budget_2025        0
source_url         0
source_document    0
dtype: int64


In [80]:
print(
    "Budget CSV exists:",
    os.path.exists(budget_output_path)
)

print(
    "File size:",
    os.path.getsize(budget_output_path),
    "bytes"
)

Budget CSV exists: True
File size: 1255 bytes


In [81]:
# ============================================================
# FINAL TASK 1 QUALITY CONTROL - ALL DATASETS
# ============================================================

import os
import pandas as pd

processed_dir = "../data/processed"

datasets = {
    "2024 CDF Projects": (
        "db-unza26-csc4792-kabwe_cdf_projects_2024.csv"
    ),
    "2025 Proposed CDF Projects": (
        "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
    ),
    "2024-2025 CDF Budget": (
        "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv"
    ),
    "2025 CDF Output Indicators": (
        "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
    )
}

quality_results = []

print("=" * 80)
print("FINAL TASK 1 QUALITY CONTROL")
print("=" * 80)

for dataset_name, filename in datasets.items():

    path = os.path.join(processed_dir, filename)

    exists = os.path.exists(path)

    print("\n" + "-" * 80)
    print(dataset_name)
    print("-" * 80)

    print("File exists:", exists)

    if not exists:
        print("WARNING: File not found.")
        continue

    df = pd.read_csv(path, sep="|")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())
    print("Accidental index column:", "Unnamed: 0" in df.columns)

    quality_results.append({
        "dataset": dataset_name,
        "filename": filename,
        "file_exists": exists,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "has_index_column": "Unnamed: 0" in df.columns
    })

quality_df = pd.DataFrame(quality_results)

print("\n" + "=" * 80)
print("FINAL QUALITY CONTROL SUMMARY")
print("=" * 80)

display(quality_df)

FINAL TASK 1 QUALITY CONTROL

--------------------------------------------------------------------------------
2024 CDF Projects
--------------------------------------------------------------------------------
File exists: True
Rows: 79
Columns: 15
Duplicate rows: 0
Missing values: 432
Accidental index column: False

--------------------------------------------------------------------------------
2025 Proposed CDF Projects
--------------------------------------------------------------------------------
File exists: True
Rows: 33
Columns: 9
Duplicate rows: 0
Missing values: 0
Accidental index column: False

--------------------------------------------------------------------------------
2024-2025 CDF Budget
--------------------------------------------------------------------------------
File exists: True
Rows: 5
Columns: 7
Duplicate rows: 0
Missing values: 0
Accidental index column: False

--------------------------------------------------------------------------------
2025 CDF Output I

,dataset,filename,file_exists,rows,columns,duplicate_rows,missing_values,has_index_column
0,2024 CDF Projects,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,True,79,15,0,432,False
1,2025 Proposed CDF Projects,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,True,33,9,0,0,False
2,2024-2025 CDF Budget,db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv,True,5,7,0,0,False
3,2025 CDF Output Indicators,db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv,True,12,10,0,4,False


In [82]:
# ============================================================
# FINAL SOURCE INVENTORY
# ============================================================

source_inventory_data = [
    {
        "id": "S001",
        "source_name": "Kabwe Municipal Council Website",
        "source_type": "Website",
        "url": "https://www.kabwecouncil.gov.zm",
        "description": "Official Kabwe Municipal Council website and digital source directory."
    },
    {
        "id": "S002",
        "source_name": "2024 Bwacha CDF Community Projects Submission",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2024/11/"
            "2024-Bwacha-community-projects-Recieved.pdf"
        ),
        "description": "2024 CDF community project submissions for Bwacha Constituency."
    },
    {
        "id": "S003",
        "source_name": "2024 Kabwe Central CDF Community Projects Submission",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2024/11/"
            "2024-Community-Projects-Kabwe-Central-received.pdf"
        ),
        "description": "2024 CDF community project submissions for Kabwe Central Constituency."
    },
    {
        "id": "S004",
        "source_name": "2025 Kabwe Municipal Council OBB Budget",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/05/"
            "2025-KABWE-M-COUNCIL-OBB.pdf"
        ),
        "description": "2025 Output Based Budget containing CDF budget allocations and output indicators."
    },
    {
        "id": "S005",
        "source_name": "Kabwe District Integrated Development Plan 2023-2028",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2024/09/"
            "Kabwe-Approved-IDP_Final-Version-1.pdf"
        ),
        "description": "Approved Kabwe District Integrated Development Plan consulted during source discovery."
    }
]

source_inventory_df = pd.DataFrame(source_inventory_data)

source_inventory_path = "../sources/source_inventory.csv"

source_inventory_df.to_csv(
    source_inventory_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", source_inventory_path)
display(source_inventory_df)

Saved: ../sources/source_inventory.csv


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and digital source directory.
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency.
2,S003,2024 Kabwe Central CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Pr...,2024 CDF community project submissions for Kabwe Central Constituency.
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and output indica...
4,S005,Kabwe District Integrated Development Plan 2023-2028,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...,Approved Kabwe District Integrated Development Plan consulted during source ...


In [83]:
# Add the 2025 Proposed CDF Projects source

source_record = pd.DataFrame([
    {
        "id": "S006",
        "source_name": "Proposed 2025 CDF Projects",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/08/"
            "Proposed-2025-CDF-projects.pdf"
        ),
        "description": (
            "Proposed 2025 CDF community projects for "
            "Kabwe Central Constituency, including ward, sector "
            "and approval comments."
        )
    }
])

source_inventory_df = pd.concat(
    [source_inventory_df, source_record],
    ignore_index=True
)

source_inventory_df.to_csv(
    source_inventory_path,
    index=False,
    encoding="utf-8"
)

print("Updated source inventory:")
display(source_inventory_df)

Updated source inventory:


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and digital source directory.
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency.
2,S003,2024 Kabwe Central CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Pr...,2024 CDF community project submissions for Kabwe Central Constituency.
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and output indica...
4,S005,Kabwe District Integrated Development Plan 2023-2028,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...,Approved Kabwe District Integrated Development Plan consulted during source ...
5,S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/Proposed-2025-CDF...,"Proposed 2025 CDF community projects for Kabwe Central Constituency, includi..."


In [84]:
# ============================================================
# FINAL TASK 1 CONSISTENCY CHECK
# ============================================================

import os
import pandas as pd

processed_dir = "../data/processed"
source_inventory_path = "../sources/source_inventory.csv"

# ------------------------------------------------------------
# 1. Check processed CSV files
# ------------------------------------------------------------

expected_files = [
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    "db-unza26-csc4792-kabwe_cdf_projects_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
]

print("=" * 80)
print("PROCESSED DATASET FILE CHECK")
print("=" * 80)

file_results = []

for filename in expected_files:

    path = os.path.join(processed_dir, filename)

    exists = os.path.exists(path)

    size = os.path.getsize(path) if exists else 0

    file_results.append({
        "filename": filename,
        "exists": exists,
        "size_bytes": size
    })

file_check_df = pd.DataFrame(file_results)

display(file_check_df)


# ------------------------------------------------------------
# 2. Check source inventory
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE INVENTORY CHECK")
print("=" * 80)

source_check_df = pd.read_csv(source_inventory_path)

print("Number of sources:", len(source_check_df))
print("Columns:", source_check_df.columns.tolist())

print("\nMissing values:")
print(source_check_df.isna().sum())

print("\nDuplicate source IDs:")
print(source_check_df["id"].duplicated().sum())

display(source_check_df)


# ------------------------------------------------------------
# 3. Check CSV delimiters by reading them with '|'
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PIPE-DELIMITER CHECK")
print("=" * 80)

delimiter_results = []

for filename in expected_files:

    path = os.path.join(processed_dir, filename)

    if os.path.exists(path):

        df = pd.read_csv(path, sep="|")

        delimiter_results.append({
            "filename": filename,
            "rows": len(df),
            "columns": len(df.columns),
            "pipe_delimiter_working": len(df.columns) > 1
        })

delimiter_check_df = pd.DataFrame(delimiter_results)

display(delimiter_check_df)


# ------------------------------------------------------------
# 4. Overall result
# ------------------------------------------------------------

all_files_exist = file_check_df["exists"].all()

source_inventory_valid = (
    source_check_df["id"].duplicated().sum() == 0
    and source_check_df.isna().sum().sum() == 0
)

delimiter_valid = delimiter_check_df[
    "pipe_delimiter_working"
].all()

print("\n" + "=" * 80)
print("OVERALL RESULT")
print("=" * 80)

print("All required CSV files exist:", all_files_exist)
print("Source inventory is complete:", source_inventory_valid)
print("Pipe delimiter is working:", delimiter_valid)

if all_files_exist and source_inventory_valid and delimiter_valid:
    print("\nTASK 1 DATA CONSISTENCY CHECK: PASSED")
else:
    print("\nTASK 1 DATA CONSISTENCY CHECK: NEEDS ATTENTION")

PROCESSED DATASET FILE CHECK


,filename,exists,size_bytes
0,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,True,26450
1,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,True,9599
2,db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv,True,1255
3,db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv,True,2735



SOURCE INVENTORY CHECK
Number of sources: 6
Columns: ['id', 'source_name', 'source_type', 'url', 'description']

Missing values:
id             0
source_name    0
source_type    0
url            0
description    0
dtype: int64

Duplicate source IDs:
0


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and digital source directory.
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency.
2,S003,2024 Kabwe Central CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Pr...,2024 CDF community project submissions for Kabwe Central Constituency.
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and output indica...
4,S005,Kabwe District Integrated Development Plan 2023-2028,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...,Approved Kabwe District Integrated Development Plan consulted during source ...
5,S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/Proposed-2025-CDF...,"Proposed 2025 CDF community projects for Kabwe Central Constituency, includi..."



PIPE-DELIMITER CHECK


,filename,rows,columns,pipe_delimiter_working
0,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,79,15,True
1,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,33,9,True
2,db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv,5,7,True
3,db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv,12,10,True



OVERALL RESULT
All required CSV files exist: True
Source inventory is complete: True
Pipe delimiter is working: True

TASK 1 DATA CONSISTENCY CHECK: PASSED


In [85]:
# ============================================================
# FIND REVENUE / FINANCIAL TABLES IN THE 2025 OBB
# ============================================================

import fitz

budget_doc = fitz.open("../data/raw/2025_kabwe_obb_budget.pdf")

revenue_keywords = [
    "revenue",
    "local revenue",
    "own source revenue",
    "LGEF",
    "market fees",
    "levy",
    "rates"
]

revenue_pages = []

for page_number, page in enumerate(budget_doc, start=1):

    text = page.get_text()

    matched_keywords = [
        keyword
        for keyword in revenue_keywords
        if keyword.lower() in text.lower()
    ]

    if matched_keywords:
        revenue_pages.append({
            "page": page_number,
            "keywords": matched_keywords
        })

revenue_pages_df = pd.DataFrame(revenue_pages)

display(revenue_pages_df)

,page,keywords
0,2,"[revenue, levy, rates]"
1,3,"[revenue, market fees]"
2,4,"[revenue, levy]"
3,5,"[revenue, rates]"
4,8,[revenue]
5,10,[revenue]
6,34,[revenue]
7,35,"[revenue, own source revenue]"
8,51,"[revenue, own source revenue]"


In [86]:
# ============================================================
# INSPECT REVENUE-RELATED OBB PAGES
# ============================================================

for page_number in revenue_pages_df["page"]:

    page = budget_doc[page_number - 1]
    text = page.get_text()

    print("\n" + "=" * 100)
    print(f"PAGE {page_number}")
    print("=" * 100)
    print(text[:12000])


PAGE 2
OUTPUT BASED  ANNUAL BUDGET
Page 2
920
5
HEA
D
KABWE MUNICIPAL COUNCIL
Local taxes/rates
01
APPROVED 
BUDGET 2025
REVISED 
BUDGET 2026
BUDGET 
ESTIMATE 2027
CODE
REVENUE DESCRIPTION
Residential
4,453,818
4,453,818
4,453,818
001
Commercial
5,536,418
5,536,418
5,536,418
002
Industrial
2,635,280
2,635,280
2,635,280
003
Hospitality
572,837
630,121
693,133
004
13,198,353
13,255,637
13,318,649
SubItem Total
Personal levy
450,000
495,000
220,000
001
450,000
495,000
220,000
SubItem Total


PAGE 3
OUTPUT BASED  ANNUAL BUDGET
Page 3
920
5
HEA
D
KABWE MUNICIPAL COUNCIL
Fees and Charges
02
APPROVED 
BUDGET 2025
REVISED 
BUDGET 2026
BUDGET 
ESTIMATE 2027
CODE
REVENUE DESCRIPTION
Consent fees
25,000
27,500
30,250
001
Survey fees
700,000
770,000
862,400
002
Building inspection-fees
250,000
300,000
300,000
003
Plan scrutiny fee
2,508,000
2,508,000
2,508,000
004
Change of premise use
295,000
324,500
347,864
005
Container/Ntemba fees
50,000
75,000
75,000
006
Rentals/lease of Council’s properties

In [87]:
page_5 = budget_doc[4]

text_5 = page_5.get_text()

print(text_5)


OUTPUT BASED  ANNUAL BUDGET
Page 5
920
5
HEA
D
KABWE MUNICIPAL COUNCIL
4.0   BUDGET SUMMARY
Kabwe Municipal Council's estimates of revenue and expenditure for the year 2025 stands at 
approximately K181.1 million, representing a percentage increase of 28.7 percent from the 2024 Budget 
which stood at K140.7 million. This is due to an increased allocation to the Constituency Development 
Fund for the two (2) constituencies namely Kabwe Central and Bwacha from K61.3 million in 2024 to 
K72.1 million, the inclusion of Cash for work of K19.2 million, Zambia devolution support programme 
grant of K8.3 million and increase of grants for devolved Ministries from 7.2 to K8.8 million. The budget 
has been allocated to (16) Sixteen programmes tabulated in table 2
 Budget Allocation by Economic Classification
Table:1  
Figure 1:  Budget Allocation by Economic Classification
ECONOMIC CLASSIFICATION  
2023 
APPROVED 
BUDGET (K)     
No   
2024 
APPROVED 
BUDGET (K)   
2025 BUDGET 
ESTIMATE (K)  
(0

In [88]:
# 2025-2027 Council Funding / National Support Dataset

funding_data = [
    ["Constituency Development Fund", "CDF", 72116301, 72116301, 72116301],
    ["Roads Grant", "Grant", 3200587, 3200587, 3200587],
    ["Health Grant", "Grant", 3649035, 3649035, 3649035],
    ["Local Government Equalisation Fund", "LGEF", 23857381, 23857381, 23857381],
    ["Grants in lieu of Rates", "Grant", 1200000, 1200000, 1200000],
    ["Other Grants", "Grant", 29494551, 10527645, 10797645],
]

funding_df = pd.DataFrame(
    funding_data,
    columns=[
        "revenue_description",
        "funding_category",
        "approved_budget_2025",
        "revised_budget_2026",
        "budget_estimate_2027",
    ]
)

funding_df

,revenue_description,funding_category,approved_budget_2025,revised_budget_2026,budget_estimate_2027
0,Constituency Development Fund,CDF,72116301,72116301,72116301
1,Roads Grant,Grant,3200587,3200587,3200587
2,Health Grant,Grant,3649035,3649035,3649035
3,Local Government Equalisation Fund,LGEF,23857381,23857381,23857381
4,Grants in lieu of Rates,Grant,1200000,1200000,1200000
5,Other Grants,Grant,29494551,10527645,10797645


In [89]:
funding_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

funding_df["source_document"] = "2025 Kabwe Municipal Council OBB Budget"

funding_df

,revenue_description,funding_category,approved_budget_2025,revised_budget_2026,budget_estimate_2027,source_url,source_document
0,Constituency Development Fund,CDF,72116301,72116301,72116301,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,Roads Grant,Grant,3200587,3200587,3200587,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,Health Grant,Grant,3649035,3649035,3649035,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,Local Government Equalisation Fund,LGEF,23857381,23857381,23857381,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,Grants in lieu of Rates,Grant,1200000,1200000,1200000,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,Other Grants,Grant,29494551,10527645,10797645,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [90]:
funding_df["approved_budget_2025"].sum()

np.int64(133517855)

In [91]:
funding_df["revised_budget_2026"].sum()

np.int64(114550949)

In [92]:
funding_df["budget_estimate_2027"].sum()

np.int64(114820949)

In [93]:
funding_output = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_council_funding_2025_2027.csv"
)

funding_df.to_csv(
    funding_output,
    sep="|",
    index=False
)

print(f"Saved: {funding_output}")

Saved: ../data/processed/db-unza26-csc4792-kabwe_council_funding_2025_2027.csv


In [94]:
check_funding = pd.read_csv(
    funding_output,
    sep="|"
)

print("Rows:", len(check_funding))
print("Columns:", len(check_funding.columns))
print("Duplicates:", check_funding.duplicated().sum())

display(check_funding)

Rows: 6
Columns: 7
Duplicates: 0


,revenue_description,funding_category,approved_budget_2025,revised_budget_2026,budget_estimate_2027,source_url,source_document
0,Constituency Development Fund,CDF,72116301,72116301,72116301,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,Roads Grant,Grant,3200587,3200587,3200587,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,Health Grant,Grant,3649035,3649035,3649035,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,Local Government Equalisation Fund,LGEF,23857381,23857381,23857381,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,Grants in lieu of Rates,Grant,1200000,1200000,1200000,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,Other Grants,Grant,29494551,10527645,10797645,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [95]:
# Local Revenue Streams: 2025-2027
# Source: 2025 Kabwe Municipal Council OBB Budget, Pages 2-4

local_revenue_data = [
    # Local taxes/rates
    ["Local taxes/rates", "001", "Residential", 4453818, 4453818, 4453818],
    ["Local taxes/rates", "002", "Commercial", 5536418, 5536418, 5536418],
    ["Local taxes/rates", "003", "Industrial", 2635280, 2635280, 2635280],
    ["Local taxes/rates", "004", "Hospitality", 572837, 630121, 693133],

    # Personal levy
    ["Personal levy", "001", "Personal levy", 450000, 495000, 220000],

    # Fees and Charges
    ["Fees and Charges", "001", "Consent fees", 25000, 27500, 30250],
    ["Fees and Charges", "002", "Survey fees", 700000, 770000, 862400],
    ["Fees and Charges", "003", "Building inspection-fees", 250000, 300000, 300000],
    ["Fees and Charges", "004", "Plan scrutiny fee", 2508000, 2508000, 2508000],
    ["Fees and Charges", "005", "Change of premise use", 295000, 324500, 347864],
    ["Fees and Charges", "006", "Container/Ntemba fees", 50000, 75000, 75000],
    ["Fees and Charges", "007", "Rentals/lease of Council’s properties", 896400, 986040, 1084644],
    ["Fees and Charges", "008", "Non-Land Application forms fees", 600000, 660000, 726000],
    ["Fees and Charges", "009", "Rentals from houses", 85092, 93601, 102961],
    ["Fees and Charges", "011", "Search fees", 7500, 8250, 9075],
    ["Fees and Charges", "012", "Notice board advert fees", 2500, 2750, 3025],
    ["Fees and Charges", "013", "Market fees", 785664, 864230, 950653],
    ["Fees and Charges", "014", "Parking fees", 50000, 50000, 50000],
    ["Fees and Charges", "016", "Loading fees (buses, trucks, trains, taxies etc.)", 393360, 432696, 475966],
    ["Fees and Charges", "017", "Affidavit fees", 6000, 6600, 7260],
    ["Fees and Charges", "020", "Hire of halls", 114684, 126152, 139824],
    ["Fees and Charges", "021", "Hire of grounds/stadia", 13000, 60000, 75000],
    ["Fees and Charges", "024", "Recommendations fees", 672000, 732600, 807800],
    ["Fees and Charges", "027", "Body remains (inspections) fees", 600, 660, 739],
    ["Fees and Charges", "033", "Refuse disposal", 1257030, 1348793, 1443209],
    ["Fees and Charges", "035", "Commercial & non-commercial Exhibitions", 10800, 600, 800],
    ["Fees and Charges", "038", "Library membership fees", 8000, 8800, 9680],
    ["Fees and Charges", "041", "Dumb site fees", 39600, 43560, 48787],
    ["Fees and Charges", "045", "Notice of marriage fees", 61250, 67375, 74113],
    ["Fees and Charges", "046", "Abattoir/meat inspection fees", 11500, 12650, 14168],
    ["Fees and Charges", "047", "Registration of clubs and societies", 100000, 110000, 121000],
    ["Fees and Charges", "051", "Farm produce Fee", 150000, 200000, 200000],
    ["Fees and Charges", "053", "Certification of documents", 2160, 2160, 3600],
    ["Fees and Charges", "055", "Illegal Parking of vehicles", 990000, 990000, 990000],
    ["Fees and Charges", "063", "Billboards and banners", 800000, 880000, 985600],
    ["Fees and Charges", "064", "Hire of Transport and Equipment", 25400, 189000, 288000],
    ["Fees and Charges", "065", "Council Minutes Extracts", 90000, 90000, 90000],
    ["Fees and Charges", "066", "Penalties", 750000, 580500, 640210],
    ["Fees and Charges", "067", "Ablution Fee", 283680, 312048, 343253],
    ["Fees and Charges", "072", "Booth fees", 28000, 30800, 34496],
    ["Fees and Charges", "074", "Sale of Bid Documents", 50000, 50000, 50000],
    ["Fees and Charges", "078", "Erection of Tombstone", 5000, 5000, 5000],
    ["Fees and Charges", "082", "Telecommunication site rentals", 96045, 96045, 96045],
    ["Fees and Charges", "099", "Other fees and charges", 824750, 907225, 998058],

    # Licenses
    ["Licenses", "002", "Liquor licence", 366000, 402600, 442860],
    ["Licenses", "003", "Firearm and ammunition licence", 26000, 26600, 27260],
    ["Licenses", "004", "Petroleum Storage licence", 400000, 440000, 492800],
    ["Licenses", "005", "Dog licence", 40000, 44000, 49280],

    # Levies
    ["Levies", "001", "Livestock Movement levy", 32400, 32400, 32400],
    ["Levies", "002", "Birds levy", 60000, 66000, 72600],
    ["Levies", "004", "Pole levy", 40000, 44000, 49280],
    ["Levies", "006", "Sand levy", 43000, 47300, 52976],
    ["Levies", "011", "Telecommunication Mast", 220500, 220500, 220500],
    ["Levies", "017", "Trading (Wholesale) Business Levy", 100000, 110000, 121000],
    ["Levies", "018", "Trading (Retail) Consumable groceries business", 880100, 968110, 1064921],
    ["Levies", "021", "Manufacturing", 62500, 68750, 75625],
    ["Levies", "029", "Professional Occupation", 249900, 274890, 302379],
    ["Levies", "030", "Scrap Metal Dealers", 12500, 13750, 15125],
    ["Levies", "031", "Car Wash", 20000, 22000, 24640],

    # Permits
    ["Permits", "001", "Health permits", 1847500, 2032250, 2276120],
    ["Permits", "008", "Burial permits and grave sites", 585000, 643500, 720720],
    ["Permits", "009", "Fire certificate", 3245000, 3245000, 3634400],
    ["Permits", "010", "Extension of Business hours permits", 20000, 22000, 24640],
    ["Permits", "099", "Primary, Secondary and Tertiary permits", 140000, 154000, 172480],

    # Charges
    ["Charges", "001", "Service Charges Residential plots", 5200000, 5720000, 6406400],
    ["Charges", "002", "Service Charges Industrial plots", 600000, 660000, 739200],
    ["Charges", "004", "Premium Plot Commercial", 1500000, 1650000, 1848000],
    ["Charges", "007", "Land Application Charges", 150000, 165000, 181500],
    ["Charges", "009", "Change of ownership", 4250, 4675, 5143],
    ["Charges", "010", "Sub-division of plot", 144000, 158400, 174240],
    ["Charges", "011", "Land regularisation", 75000, 82500, 90750],
    ["Charges", "012", "Change of Land use", 212800, 234000, 260000],
    ["Charges", "099", "Land Charges", 1700000, 1870000, 2000000],
]

local_revenue_df = pd.DataFrame(
    local_revenue_data,
    columns=[
        "revenue_category",
        "revenue_code",
        "revenue_description",
        "approved_budget_2025",
        "revised_budget_2026",
        "budget_estimate_2027",
    ]
)

local_revenue_df.head()

,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,budget_estimate_2027
0,Local taxes/rates,001,Residential,4453818,4453818,4453818
1,Local taxes/rates,002,Commercial,5536418,5536418,5536418
2,Local taxes/rates,003,Industrial,2635280,2635280,2635280
3,Local taxes/rates,004,Hospitality,572837,630121,693133
4,Personal levy,001,Personal levy,450000,495000,220000


In [96]:
print("Revenue streams:", len(local_revenue_df))
print("\nRevenue streams by category:")
print(local_revenue_df["revenue_category"].value_counts())

Revenue streams: 73

Revenue streams by category:
revenue_category
Fees and Charges     39
Levies               11
Charges               9
Permits               5
Local taxes/rates     4
Licenses              4
Personal levy         1
Name: count, dtype: int64


In [98]:
local_revenue_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

local_revenue_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget, Pages 2-4"
)

In [99]:
local_revenue_output = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_local_revenue_2025_2027.csv"
)

local_revenue_df.to_csv(
    local_revenue_output,
    sep="|",
    index=False
)

print(f"Saved: {local_revenue_output}")

Saved: ../data/processed/db-unza26-csc4792-kabwe_local_revenue_2025_2027.csv


In [100]:
check_local_revenue = pd.read_csv(
    local_revenue_output,
    sep="|"
)

print("Rows:", len(check_local_revenue))
print("Columns:", len(check_local_revenue.columns))
print("Duplicates:", check_local_revenue.duplicated().sum())
print("\nMissing values:")
print(check_local_revenue.isna().sum())

Rows: 73
Columns: 8
Duplicates: 0

Missing values:
revenue_category        0
revenue_code            0
revenue_description     0
approved_budget_2025    0
revised_budget_2026     0
budget_estimate_2027    0
source_url              0
source_document         0
dtype: int64


In [101]:
# Council Budget by Economic Classification
# Source: 2025 Kabwe Municipal Council OBB Budget, Page 5

economic_budget_data = [
    ["21", "Personal Emoluments", 46609938, 48701693],
    ["22", "Goods and Services", 26095222, 33797534],
    ["26", "Grants and Other Payments (Transfers)", 16298162, 34845939],
    ["31", "Non-Financial Assets", 41160229, 52204514],
    ["32", "Financial Assets", 6984926, 7473871],
    ["41", "Current Liabilities (Payable within one year)", 3580192, 4056000],
]

economic_budget_df = pd.DataFrame(
    economic_budget_data,
    columns=[
        "economic_code",
        "economic_classification",
        "approved_budget_2024",
        "budget_estimate_2025",
    ]
)

economic_budget_df

,economic_code,economic_classification,approved_budget_2024,budget_estimate_2025
0,21,Personal Emoluments,46609938,48701693
1,22,Goods and Services,26095222,33797534
2,26,Grants and Other Payments (Transfers),16298162,34845939
3,31,Non-Financial Assets,41160229,52204514
4,32,Financial Assets,6984926,7473871
5,41,Current Liabilities (Payable within one year),3580192,4056000


In [102]:
print(
    "2024 total:",
    economic_budget_df["approved_budget_2024"].sum()
)

print(
    "2025 total:",
    economic_budget_df["budget_estimate_2025"].sum()
)

2024 total: 140728669
2025 total: 181079551


In [103]:
economic_budget_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

economic_budget_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget, Page 5"
)

In [104]:
economic_budget_output = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_economic_budget_2024_2025.csv"
)

economic_budget_df.to_csv(
    economic_budget_output,
    sep="|",
    index=False
)

print(f"Saved: {economic_budget_output}")

Saved: ../data/processed/db-unza26-csc4792-kabwe_economic_budget_2024_2025.csv


In [105]:
check_economic_budget = pd.read_csv(
    economic_budget_output,
    sep="|"
)

print("Rows:", len(check_economic_budget))
print("Columns:", len(check_economic_budget.columns))
print("Duplicates:", check_economic_budget.duplicated().sum())

display(check_economic_budget)

Rows: 6
Columns: 6
Duplicates: 0


,economic_code,economic_classification,approved_budget_2024,budget_estimate_2025,source_url,source_document
0,21,Personal Emoluments,46609938,48701693,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
1,22,Goods and Services,26095222,33797534,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
2,26,Grants and Other Payments (Transfers),16298162,34845939,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
3,31,Non-Financial Assets,41160229,52204514,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
4,32,Financial Assets,6984926,7473871,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
5,41,Current Liabilities (Payable within one year),3580192,4056000,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"


In [106]:
for page_number in range(6, 11):
    page = budget_doc[page_number - 1]
    text = page.get_text()

    print("\n" + "=" * 100)
    print(f"PAGE {page_number}")
    print("=" * 100)
    print(text[:15000])


PAGE 6
OUTPUT BASED  ANNUAL BUDGET
Page 6
920
5
HEA
D
KABWE MUNICIPAL COUNCIL
The summary estimates by economic classification shows that K49.0 million representing 27.04 percent 
of the total budget has been allocated to Personal Emoluments to facilitate payments of salaries and 
wages. In addition, K52.2 million representing 28.83 percent of the total budget has been allocated 
towards Assets acquisition such as procurement of utility vehicles and office equipment, renovation of 
council chamber and implementation of various CDF infrastructure development. 
Further, K33.5 million representing 18.52 percent of the total budget has been channelled to the Use of 
Goods and Services to facilitate the operation of the programmes.  Furthermore, Grants and other 
payments (Transfers) has been allocated K34.9 million representing 19.24 percent of the budget to 
cover Youth and Women Empowerments Grants, Boarding Secondary School and Skills Development 
Bursaries as well as cash for work.
Th

In [107]:
# Kabwe Municipal Council Programme and Sub-Programme Budget
# Source: 2025 Kabwe Municipal Council OBB Budget, Pages 8-10

programme_budget_data = [
    # Constituency Development
    ["1", "Constituency Development", "Programme", "", 61271284, 72116301],
    ["779", "Constituency Development", "Sub-Programme", "Community Projects", 34924632, 43925383],
    ["780", "Constituency Development", "Sub-Programme", "Women and Youth Empowerment", 11641544, 12456452],
    ["781", "Constituency Development", "Sub-Programme", "CDF Administration", 3063564, 3278014],
    ["782", "Constituency Development", "Sub-Programme", "Secondary School and Skills Development Bursaries", 11641544, 12456452],

    # Local Governance
    ["2", "Local Governance", "Programme", "", 6125473, 5827497],
    ["003", "Local Governance", "Sub-Programme", "Legislative Functions", 3767535, 3349182],
    ["043", "Local Governance", "Sub-Programme", "Citizen Engagement", 2357938, 2478315],

    # Integrated Development Planning
    ["3", "Integrated Development Planning", "Programme", "", 6342899, 5387125],
    ["006", "Integrated Development Planning", "Sub-Programme", "Environmental Planning", 893433, 3677653],
    ["021", "Integrated Development Planning", "Sub-Programme", "Spatial Planning", 5449465, 1709472],

    # Economic and Business Development
    ["4", "Economic and Business Development", "Programme", "", 151477, 168721],
    ["038", "Economic and Business Development", "Sub-Programme", "Trade Facilitation and Licensing", 151477, 168721],

    # Public Health and Environmental Protection
    ["5", "Public Health and Environmental Protection", "Programme", "", 3498334, 4611170],
    ["015", "Public Health and Environmental Protection", "Sub-Programme", "Cemetery and Funeral Services", 156074, 728585],
    ["019", "Public Health and Environmental Protection", "Sub-Programme", "Health Inspections", 1268384, 2799110],
    ["023", "Public Health and Environmental Protection", "Sub-Programme", "Pest Control", 18740, 335315],
    ["024", "Public Health and Environmental Protection", "Sub-Programme", "Pollution Control", 27266, 26400],
    ["027", "Public Health and Environmental Protection", "Sub-Programme", "Solid Waste Management", 1994296, 694450],
    ["034", "Public Health and Environmental Protection", "Sub-Programme", "Water Supply and Sanitation Services", 33575, 27310],

    # Housing and Community Amenities
    ["6", "Housing and Community Amenities", "Programme", "", 13975738, 27194430],
    ["008", "Housing and Community Amenities", "Sub-Programme", "Roads and Drainages", 5040099, 16076588],
    ["011", "Housing and Community Amenities", "Sub-Programme", "Parks and Gardens", 1707973, 687249],
    ["012", "Housing and Community Amenities", "Sub-Programme", "Markets and Bus Stations", 2983191, 1698806],
    ["026", "Housing and Community Amenities", "Sub-Programme", "Public Housing", 2904099, 4973907],
    ["031", "Housing and Community Amenities", "Sub-Programme", "Street Lighting", 1340376, 3757881],

    # Recreation, Culture and Religion
    ["7", "Recreation Culture and Religion", "Programme", "", 1067283, 844145],
    ["001", "Recreation Culture and Religion", "Sub-Programme", "Cultural Affairs", 7759, 61724],
    ["042", "Recreation Culture and Religion", "Sub-Programme", "Sports Promotion", 1059524, 782421],

    # Education and Skills Development
    ["8", "Education and Skills Development", "Programme", "", 33780, 38731],
    ["001", "Education and Skills Development", "Sub-Programme", "District Archives", 0, 19241],
    ["005", "Education and Skills Development", "Sub-Programme", "Early Childhood Education", 33780, 19490],

    # Public Order and Safety
    ["10", "Public Order and Safety", "Programme", "", 6822011, 10917603],
    ["018", "Public Order and Safety", "Sub-Programme", "Community Policing", 1686555, 3487804],
    ["041", "Public Order and Safety", "Sub-Programme", "Fire Protection Services", 5135456, 7429799],

    # Management and Support Services
    ["11", "Management and Support Services", "Programme", "", 32145481, 22039403],
    ["001", "Management and Support Services", "Sub-Programme", "Human Resource and Administration", 13358205, 6773166],
    ["009", "Management and Support Services", "Sub-Programme", "Executive Management", 1847378, 880392],
    ["016", "Management and Support Services", "Sub-Programme", "Procurement", 1263583, 713018],
    ["028", "Management and Support Services", "Sub-Programme", "Financial Management-Auditing", 803216, 636825],
    ["035", "Management and Support Services", "Sub-Programme", "Financial Management-Accounting", 10685644, 10084742],
    ["036", "Management and Support Services", "Sub-Programme", "Legal Services", 2732692, 2393867],
    ["062", "Management and Support Services", "Sub-Programme", "Public Relations", 1454763, 557393],

    # Resource Mobilisation and Management
    ["12", "Resource Mobilisation and Management", "Programme", "", 2346892, 4209839],
    ["067", "Resource Mobilisation and Management", "Sub-Programme", "Revenue Mobilisation and Enhancement", 2346892, 4209839],

    # District Health Services
    ["13", "District Health Services", "Programme", "", 3100894, 3649036],
    ["001", "District Health Services", "Sub-Programme", "Primary Health Services", 3100894, 3101039],
    ["002", "District Health Services", "Sub-Programme", "District Health Co-ordination", 0, 547997],

    # Transport Services
    ["15", "Transport Services", "Programme", "", 3742847, 3200587],
    ["001", "Transport Services", "Sub-Programme", "Road Transport", 3742847, 3200587],

    # Agricultural Services
    ["16", "Agricultural Services", "Programme", "", 0, 549076],
    ["071", "Agricultural Services", "Sub-Programme", "Agricultural Crop Production, Advisory and Technical Services", 0, 328430],
    ["072", "Agricultural Services", "Sub-Programme", "Agribusiness Development and Marketing", 0, 32400],
    ["073", "Agricultural Services", "Sub-Programme", "Agriculture Co-ordination", 0, 188246],

    # Fisheries and Livestock
    ["17", "Fisheries and Livestock", "Programme", "", 104276, 486160],
    ["074", "Fisheries and Livestock", "Sub-Programme", "Fisheries and Livestock Marketing", 0, 58616],
    ["075", "Fisheries and Livestock", "Sub-Programme", "Animal Health Services", 104276, 121955],
    ["076", "Fisheries and Livestock", "Sub-Programme", "Fisheries Production and Productivity Improvement", 0, 101125],
    ["077", "Fisheries and Livestock", "Sub-Programme", "Livestock Production and Productivity Improvement", 0, 82924],
    ["078", "Fisheries and Livestock", "Sub-Programme", "District Fisheries and Livestock Coordination", 0, 121540],

    # Social Protection and Community Development
    ["18", "Social Protection and Community Development", "Programme", "", 0, 19839728],
    ["079", "Social Protection and Community Development", "Sub-Programme", "District Social Welfare", 0, 19455843],
    ["080", "Social Protection and Community Development", "Sub-Programme", "Community Development", 0, 383885],
]

programme_budget_df = pd.DataFrame(
    programme_budget_data,
    columns=[
        "programme_code",
        "programme",
        "record_type",
        "sub_programme",
        "approved_budget_2024",
        "budget_estimate_2025",
    ]
)

programme_budget_df.head(10)

,programme_code,programme,record_type,sub_programme,approved_budget_2024,budget_estimate_2025
0,1,Constituency Development,Programme,,61271284,72116301
1,779,Constituency Development,Sub-Programme,Community Projects,34924632,43925383
2,780,Constituency Development,Sub-Programme,Women and Youth Empowerment,11641544,12456452
3,781,Constituency Development,Sub-Programme,CDF Administration,3063564,3278014
4,782,Constituency Development,Sub-Programme,Secondary School and Skills Development Bursaries,11641544,12456452
5,2,Local Governance,Programme,,6125473,5827497
6,003,Local Governance,Sub-Programme,Legislative Functions,3767535,3349182
7,043,Local Governance,Sub-Programme,Citizen Engagement,2357938,2478315
8,3,Integrated Development Planning,Programme,,6342899,5387125
9,006,Integrated Development Planning,Sub-Programme,Environmental Planning,893433,3677653


In [108]:
programme_totals = programme_budget_df[
    programme_budget_df["record_type"] == "Programme"
]

print("Number of programmes:", len(programme_totals))

print(
    "2024 programme total:",
    programme_totals["approved_budget_2024"].sum()
)

print(
    "2025 programme total:",
    programme_totals["budget_estimate_2025"].sum()
)

Number of programmes: 16
2024 programme total: 140728669
2025 programme total: 181079552


In [109]:
programme_budget_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

programme_budget_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
)

In [110]:
programme_budget_output = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_programme_subprogramme_budget_2024_2025.csv"
)

programme_budget_df.to_csv(
    programme_budget_output,
    sep="|",
    index=False
)

print(f"Saved: {programme_budget_output}")

Saved: ../data/processed/db-unza26-csc4792-kabwe_programme_subprogramme_budget_2024_2025.csv


In [111]:
check_programme_budget = pd.read_csv(
    programme_budget_output,
    sep="|"
)

print("Rows:", len(check_programme_budget))
print("Columns:", len(check_programme_budget.columns))
print("Duplicates:", check_programme_budget.duplicated().sum())

display(check_programme_budget)

Rows: 63
Columns: 8
Duplicates: 0


,programme_code,programme,record_type,sub_programme,approved_budget_2024,budget_estimate_2025,source_url,source_document
0,1,Constituency Development,Programme,NaN,61271284,72116301,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
1,779,Constituency Development,Sub-Programme,Community Projects,34924632,43925383,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
2,780,Constituency Development,Sub-Programme,Women and Youth Empowerment,11641544,12456452,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
3,781,Constituency Development,Sub-Programme,CDF Administration,3063564,3278014,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
4,782,Constituency Development,Sub-Programme,Secondary School and Skills Development Bursaries,11641544,12456452,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
5,2,Local Governance,Programme,NaN,6125473,5827497,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
6,3,Local Governance,Sub-Programme,Legislative Functions,3767535,3349182,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
7,43,Local Governance,Sub-Programme,Citizen Engagement,2357938,2478315,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
8,3,Integrated Development Planning,Programme,NaN,6342899,5387125,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
9,6,Integrated Development Planning,Sub-Programme,Environmental Planning,893433,3677653,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"


In [112]:
financial_url = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/"
    "2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"
)

financial_raw_path = "../data/raw/2025_kabwe_annual_financial_statement.pdf"

response = requests.get(
    financial_url,
    timeout=120,
    verify=False
)

response.raise_for_status()

with open(financial_raw_path, "wb") as f:
    f.write(response.content)

print("Downloaded:", financial_raw_path)
print("File size:", len(response.content), "bytes")

Downloaded: ../data/raw/2025_kabwe_annual_financial_statement.pdf
File size: 11368536 bytes


In [113]:
financial_doc = fitz.open(financial_raw_path)

print("Pages:", financial_doc.page_count)

Pages: 36


In [116]:
financial_keywords = [
    "LGEF",
    "Local Government Equalisation Fund",
    "revenue",
    "receipts",
    "expenditure",
    "CDF",
    "utilisation",
    "capital",
    "own source",
    "financial performance"
]

financial_pages = []

for page_number, page in enumerate(financial_doc, start=1):
    text = page.get_text()

    matches = [
        keyword
        for keyword in financial_keywords
        if keyword.lower() in text.lower()
    ]

    if matches:
        financial_pages.append({
            "page": page_number,
            "keywords_found": ", ".join(matches)
        })

financial_pages_df = pd.DataFrame(financial_pages)

display(financial_pages_df)

""


In [2]:
import pandas as pd
from pathlib import Path

rows = [
    [2025, "Kabwe Central", "Receipt", "CDF Funding", 6205128, "8(a)"],
    [2025, "Bwacha", "Receipt", "CDF Funding", 6205128, "8(a)"],

    [2025, "Kabwe Central", "Receipt", "Loan Repayments", 179669, "8(b)"],
    [2025, "Bwacha", "Receipt", "Loan Repayments", 477394, "8(b)"],

    [2025, "Kabwe Central", "Receipt", "Interest Earned", 21067, "8(c)"],
    [2025, "Bwacha", "Receipt", "Interest Earned", 6331, "8(c)"],

    [2025, "Kabwe Central", "Payment", "Infrastructure Development", 3703681, "8(d)"],
    [2025, "Bwacha", "Payment", "Infrastructure Development", 2649421, "8(d)"],

    [2025, "Kabwe Central", "Payment", "Rehabilitation Works", 557547, "8(e)"],
    [2025, "Bwacha", "Payment", "Rehabilitation Works", 0, "8(e)"],

    [2025, "Kabwe Central", "Payment", "Asset Acquisition", 5244633, "8(f)"],
    [2025, "Bwacha", "Payment", "Asset Acquisition", 5208500, "8(f)"],

    [2025, "All Constituencies", "Payment", "Rural Electrification", 0, "8(g)"],

    [2025, "Kabwe Central", "Payment", "Social Benefits - Grants", 2320000, "8(h)"],
    [2025, "Bwacha", "Payment", "Social Benefits - Grants", 2176650, "8(h)"],

    [2025, "All Constituencies", "Payment", "Loans", 0, "8(i)"],

    [2025, "Kabwe Central", "Payment", "Skills & Boarding School Bursaries", 4306021, "8(j)"],
    [2025, "Bwacha", "Payment", "Skills & Boarding School Bursaries", 4292889, "8(j)"],

    [2025, "Kabwe Central", "Payment", "Administrative Cost", 928316, "8(k)"],
    [2025, "Bwacha", "Payment", "Administrative Cost", 1067956, "8(k)"],

    [2025, "Kabwe Central", "Payment", "Disaster Contingency", 60038, "8(l)"],
    [2025, "Bwacha", "Payment", "Disaster Contingency", 235214, "8(l)"],

    [2025, "Kabwe Central", "Payment", "Other Payments - Fuel for Roads", 855602, "8(m)"],
    [2025, "Bwacha", "Payment", "Other Payments - Fuel for Roads", 855602, "8(m)"],
]

columns = [
    "year",
    "constituency",
    "transaction_type",
    "financial_category",
    "amount_kwacha",
    "source_note"
]

cdf_financial = pd.DataFrame(rows, columns=columns)

cdf_financial["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/"
    "2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"
)

cdf_financial["source_document"] = (
    "Kabwe Municipal Council BI-Annual Financial Statements 2025"
)

output_path = Path("../data/processed/"
                   "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv")

cdf_financial.to_csv(
    output_path,
    sep="|",
    index=False
)

print("CSV created successfully!")
print("File:", output_path)
print("Rows:", len(cdf_financial))
print("Columns:", len(cdf_financial.columns))

print("\nCDF receipts:", 
      cdf_financial.loc[
          cdf_financial["transaction_type"] == "Receipt",
          "amount_kwacha"
      ].sum())

print("CDF payments:",
      cdf_financial.loc[
          cdf_financial["transaction_type"] == "Payment",
          "amount_kwacha"
      ].sum())

cdf_financial

CSV created successfully!
File: ..\data\processed\db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv
Rows: 24
Columns: 8

CDF receipts: 13094717
CDF payments: 34462070


,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
5,2025,Bwacha,Receipt,Interest Earned,6331,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
6,2025,Kabwe Central,Payment,Infrastructure Development,3703681,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
7,2025,Bwacha,Payment,Infrastructure Development,2649421,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
8,2025,Kabwe Central,Payment,Rehabilitation Works,557547,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
9,2025,Bwacha,Payment,Rehabilitation Works,0,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


In [3]:
# Load the CDF financial statement CSV
cdf_financial_check = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    sep="|"
)

print("Shape:", cdf_financial_check.shape)
print("\nColumns:")
print(cdf_financial_check.columns.tolist())

print("\nMissing values:")
print(cdf_financial_check.isna().sum())

print("\nDuplicate rows:", cdf_financial_check.duplicated().sum())

print("\nTransaction types:")
print(cdf_financial_check["transaction_type"].value_counts())

print("\nFinancial categories:")
print(cdf_financial_check["financial_category"].value_counts())

print("\nPreview:")
display(cdf_financial_check)

Shape: (24, 8)

Columns:
['year', 'constituency', 'transaction_type', 'financial_category', 'amount_kwacha', 'source_note', 'source_url', 'source_document']

Missing values:
year                  0
constituency          0
transaction_type      0
financial_category    0
amount_kwacha         0
source_note           0
source_url            0
source_document       0
dtype: int64

Duplicate rows: 0

Transaction types:
transaction_type
Payment    18
Receipt     6
Name: count, dtype: int64

Financial categories:
financial_category
CDF Funding                           2
Loan Repayments                       2
Interest Earned                       2
Infrastructure Development            2
Rehabilitation Works                  2
Asset Acquisition                     2
Social Benefits - Grants              2
Skills & Boarding School Bursaries    2
Administrative Cost                   2
Disaster Contingency                  2
Other Payments - Fuel for Roads       2
Rural Electrification        

,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
5,2025,Bwacha,Receipt,Interest Earned,6331,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
6,2025,Kabwe Central,Payment,Infrastructure Development,3703681,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
7,2025,Bwacha,Payment,Infrastructure Development,2649421,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
8,2025,Kabwe Central,Payment,Rehabilitation Works,557547,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
9,2025,Bwacha,Payment,Rehabilitation Works,0,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


In [4]:
# CDF funding received by constituency in 2025

cdf_disbursements = pd.DataFrame([
    [2025, "Kabwe Central", "CDF Funding", 6205128, "8(a)"],
    [2025, "Bwacha", "CDF Funding", 6205128, "8(a)"],
], columns=[
    "year",
    "constituency",
    "funding_type",
    "amount_kwacha",
    "source_note"
])

cdf_disbursements["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/"
    "2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"
)

cdf_disbursements["source_document"] = (
    "Kabwe Municipal Council BI-Annual Financial Statements 2025"
)

output_path = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv"
)

cdf_disbursements.to_csv(
    output_path,
    sep="|",
    index=False
)

print("CSV created successfully!")
print("File:", output_path)
print("Rows:", len(cdf_disbursements))
print("Total CDF funding:", 
      f"K{cdf_disbursements['amount_kwacha'].sum():,.0f}")

display(cdf_disbursements)

CSV created successfully!
File: ../data/processed/db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv
Rows: 2
Total CDF funding: K12,410,256


,year,constituency,funding_type,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


In [5]:
# Verify the 2024 and 2025 CDF project datasets

cdf_2024 = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    sep="|"
)

cdf_2025 = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2025.csv",
    sep="|"
)

print("===== 2024 CDF PROJECTS =====")
print("Shape:", cdf_2024.shape)
print("Duplicates:", cdf_2024.duplicated().sum())
print("\nProjects by constituency:")
print(cdf_2024["constituency"].value_counts())

print("\nMissing values:")
print(cdf_2024.isna().sum())

print("\n===== 2025 CDF PROJECTS =====")
print("Shape:", cdf_2025.shape)
print("Duplicates:", cdf_2025.duplicated().sum())
print("\nProjects by sector:")
print(cdf_2025["sector"].value_counts())

print("\nMissing values:")
print(cdf_2025.isna().sum())

===== 2024 CDF PROJECTS =====
Shape: (79, 15)
Duplicates: 0

Projects by constituency:
constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

Missing values:
project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            1
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                 79
source_url              0
source_document         0
dtype: int64

===== 2025 CDF PROJECTS =====
Shape: (33, 9)
Duplicates: 0

Projects by sector:
sector
Education               16
Health                   8
Commerce and Trade       4
Water and Sanitation     3
Transport                2
Name: count, dtype: int64

Missing values:
no                 0
year               0
constituency       0
project            0
ward               0
sector             0
comment            0
s

In [7]:
from pathlib import Path

source_inventory = """id,source_name,source_type,url,description
S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and digital source directory.
S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Recieved.pdf,2024 CDF community project submissions for Bwacha Constituency.
S003,2024 Kabwe Central CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central-received.pdf,2024 CDF community project submissions for Kabwe Central Constituency.
S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf,2025 Output Based Budget containing CDF allocations and CDF output indicators.
S005,Kabwe District Integrated Development Plan 2023-2028,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-IDP_Final-Version-1.pdf,Approved Kabwe District Integrated Development Plan for 2023-2028.
S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/Proposed-2025-CDF-projects.pdf,Proposed 2025 CDF projects for Kabwe Central Constituency.
S007,2025 Kabwe Municipal Council BI-Annual Financial Statements,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf,2025 BI-Annual Financial Statements containing CDF funding loan repayments interest expenditure and constituency-level CDF financial information.
"""

source_path = Path("../sources/source_inventory.csv")

source_path.write_text(
    source_inventory,
    encoding="utf-8"
)

print("Source inventory updated successfully.")
print("File:", source_path)

Source inventory updated successfully.
File: ..\sources\source_inventory.csv


In [8]:
sources = pd.read_csv(
    "../sources/source_inventory.csv"
)

print("Number of sources:", len(sources))
print("Columns:", sources.columns.tolist())

display(sources)

Number of sources: 7
Columns: ['id', 'source_name', 'source_type', 'url', 'description']


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and d...
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 CDF community project submissions for Bwa...
2,S003,2024 Kabwe Central CDF Community Projects Subm...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 CDF community project submissions for Kab...
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2025 Output Based Budget containing CDF alloca...
4,S005,Kabwe District Integrated Development Plan 202...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,Approved Kabwe District Integrated Development...
5,S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF projects for Kabwe Central C...
6,S007,2025 Kabwe Municipal Council BI-Annual Financi...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2025 BI-Annual Financial Statements containing...


In [9]:
sources = pd.read_csv("../sources/source_inventory.csv")

print("Number of sources:", len(sources))
print("Columns:", sources.columns.tolist())

print("\nDuplicate IDs:", sources["id"].duplicated().sum())
print("Duplicate URLs:", sources["url"].duplicated().sum())

display(sources)

Number of sources: 7
Columns: ['id', 'source_name', 'source_type', 'url', 'description']

Duplicate IDs: 0
Duplicate URLs: 0


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and d...
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 CDF community project submissions for Bwa...
2,S003,2024 Kabwe Central CDF Community Projects Subm...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 CDF community project submissions for Kab...
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2025 Output Based Budget containing CDF alloca...
4,S005,Kabwe District Integrated Development Plan 202...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,Approved Kabwe District Integrated Development...
5,S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF projects for Kabwe Central C...
6,S007,2025 Kabwe Municipal Council BI-Annual Financi...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2025 BI-Annual Financial Statements containing...


In [10]:
from pathlib import Path

cdf_files = [
    "../data/processed/db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",
    "../data/processed/db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv",
    "../data/processed/db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    "../data/processed/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv",
    "../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    "../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
]

print("CDF DATASET FILE CHECK")
print("=" * 50)

for file in cdf_files:
    path = Path(file)

    if path.exists():
        df = pd.read_csv(path, sep="|")
        print(f"✓ {path.name}")
        print(f"  Rows: {len(df)}")
        print(f"  Columns: {len(df.columns)}")
    else:
        print(f"✗ MISSING: {path.name}")

CDF DATASET FILE CHECK
✓ db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv
  Rows: 5
  Columns: 7
✓ db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv
  Rows: 2
  Columns: 7
✓ db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv
  Rows: 24
  Columns: 8
✓ db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv
  Rows: 12
  Columns: 10
✓ db-unza26-csc4792-kabwe_cdf_projects_2024.csv
  Rows: 79
  Columns: 15
✓ db-unza26-csc4792-kabwe_cdf_projects_2025.csv
  Rows: 33
  Columns: 9


In [11]:
cdf_financial = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    sep="|"
)

print("Shape:", cdf_financial.shape)

print("\nMissing values:")
print(cdf_financial.isna().sum())

print("\nDuplicate rows:", cdf_financial.duplicated().sum())

print("\nTransaction types:")
print(cdf_financial["transaction_type"].value_counts())

print("\nFinancial categories:")
print(cdf_financial["financial_category"].value_counts())

display(cdf_financial)

Shape: (24, 8)

Missing values:
year                  0
constituency          0
transaction_type      0
financial_category    0
amount_kwacha         0
source_note           0
source_url            0
source_document       0
dtype: int64

Duplicate rows: 0

Transaction types:
transaction_type
Payment    18
Receipt     6
Name: count, dtype: int64

Financial categories:
financial_category
CDF Funding                           2
Loan Repayments                       2
Interest Earned                       2
Infrastructure Development            2
Rehabilitation Works                  2
Asset Acquisition                     2
Social Benefits - Grants              2
Skills & Boarding School Bursaries    2
Administrative Cost                   2
Disaster Contingency                  2
Other Payments - Fuel for Roads       2
Rural Electrification                 1
Loans                                 1
Name: count, dtype: int64


,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
5,2025,Bwacha,Receipt,Interest Earned,6331,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
6,2025,Kabwe Central,Payment,Infrastructure Development,3703681,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
7,2025,Bwacha,Payment,Infrastructure Development,2649421,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
8,2025,Kabwe Central,Payment,Rehabilitation Works,557547,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
9,2025,Bwacha,Payment,Rehabilitation Works,0,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


In [12]:
receipts = cdf_financial[
    cdf_financial["transaction_type"] == "Receipt"
]

payments = cdf_financial[
    cdf_financial["transaction_type"] == "Payment"
]

print("Total CDF receipts:",
      f"K{receipts['amount_kwacha'].sum():,.0f}")

print("Total CDF payments:",
      f"K{payments['amount_kwacha'].sum():,.0f}")

Total CDF receipts: K13,094,717
Total CDF payments: K34,462,070


In [13]:
from pathlib import Path

docs_path = Path("../docs")
docs_path.mkdir(exist_ok=True)

print("Documentation folder ready:", docs_path)

Documentation folder ready: ..\docs


In [14]:
data_dictionary = [
    # CDF Projects 2024
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_number",
     "Unique project number assigned in the source document.", "Identifier"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "year",
     "Year of the CDF project submission.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "constituency",
     "CDF constituency where the project is proposed or recorded.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_name",
     "Name of the proposed CDF project.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_description",
     "Description of the proposed project.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_type",
     "Type or category of the CDF project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "ward",
     "Ward associated with the project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_site",
     "Location or site where the project is proposed.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "application_amount",
     "Amount requested in the project application.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "engineers_estimate",
     "Engineer estimated cost of the project.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "approved_amount",
     "Amount approved for the project.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "contract_amount",
     "Contracted amount for the project.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "status",
     "Project status recorded in the source.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Projects 2025
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "no",
     "Project number in the 2025 proposed projects document.", "Identifier"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "year",
     "Year of the CDF project proposal.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "constituency",
     "CDF constituency associated with the project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "project",
     "Name or description of the proposed project.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "ward",
     "Ward associated with the project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "sector",
     "Sector of the proposed CDF project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "comment",
     "Comment or approval information recorded in the source.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Budget
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "council",
     "Name of the local authority.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "fund",
     "CDF funding area represented by the budget record.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "budget_category",
     "Specific CDF budget category.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "budget_2024",
     "Budgeted amount for 2024.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "budget_2025",
     "Budgeted amount for 2025.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Disbursements
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "year",
     "Year in which the CDF funding was recorded.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "constituency",
     "Constituency receiving the CDF funding.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "funding_type",
     "Type of CDF funding received.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "amount_kwacha",
     "Amount of CDF funding recorded in Zambian Kwacha.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "source_note",
     "Reference to the relevant note in the financial statement.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Financial Statement
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "year",
     "Financial reporting year.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "constituency",
     "Constituency associated with the CDF transaction.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "transaction_type",
     "Whether the record represents a CDF receipt or payment.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "financial_category",
     "CDF financial category under which the transaction was recorded.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "amount_kwacha",
     "Amount of the transaction in Zambian Kwacha.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "source_note",
     "Reference to the relevant CDF note in the financial statement.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Output Indicators
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "year",
     "Budget year associated with the output indicator.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "output",
     "CDF output or activity being measured.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "output_indicator",
     "Indicator used to measure the CDF output.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "target_2023",
     "Target value for 2023.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "actual_2023",
     "Actual value reported for 2023.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "target_2024",
     "Target value for 2024.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "actual_2024",
     "Actual value reported for 2024.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "target_2025",
     "Target value for 2025.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],
]

dictionary_df = pd.DataFrame(
    data_dictionary,
    columns=[
        "dataset",
        "column_name",
        "description",
        "data_type"
    ]
)

dictionary_path = Path("../docs/cdf_data_dictionary.csv")

dictionary_df.to_csv(
    dictionary_path,
    sep="|",
    index=False
)

print("CDF data dictionary created successfully.")
print("Rows:", len(dictionary_df))
print("File:", dictionary_path)

display(dictionary_df)

CDF data dictionary created successfully.
Rows: 56
File: ..\docs\cdf_data_dictionary.csv


,dataset,column_name,description,data_type
0,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_number,Unique project number assigned in the source d...,Identifier
1,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,year,Year of the CDF project submission.,Year
2,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,constituency,CDF constituency where the project is proposed...,Categorical
3,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_name,Name of the proposed CDF project.,Text
4,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_description,Description of the proposed project.,Text
5,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_type,Type or category of the CDF project.,Categorical
6,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,ward,Ward associated with the project.,Categorical
7,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_site,Location or site where the project is proposed.,Text
8,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,application_amount,Amount requested in the project application.,Numeric - ZMW
9,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,engineers_estimate,Engineer estimated cost of the project.,Numeric - ZMW


In [15]:
cdf_dictionary = pd.read_csv(
    "../docs/cdf_data_dictionary.csv",
    sep="|"
)

print("Shape:", cdf_dictionary.shape)
print("Duplicates:", cdf_dictionary.duplicated().sum())
print("Missing values:")
print(cdf_dictionary.isna().sum())

display(cdf_dictionary.head(10))

Shape: (56, 4)
Duplicates: 0
Missing values:
dataset        0
column_name    0
description    0
data_type      0
dtype: int64


,dataset,column_name,description,data_type
0,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_number,Unique project number assigned in the source d...,Identifier
1,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,year,Year of the CDF project submission.,Year
2,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,constituency,CDF constituency where the project is proposed...,Categorical
3,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_name,Name of the proposed CDF project.,Text
4,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_description,Description of the proposed project.,Text
5,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_type,Type or category of the CDF project.,Categorical
6,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,ward,Ward associated with the project.,Categorical
7,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_site,Location or site where the project is proposed.,Text
8,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,application_amount,Amount requested in the project application.,Numeric - ZMW
9,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,engineers_estimate,Engineer estimated cost of the project.,Numeric - ZMW


In [17]:
from pathlib import Path

docs_path = Path("../docs")
docs_path.mkdir(exist_ok=True)

readme_lines = [
    "# Kabwe Municipal Council CDF Dataset",
    "",
    "## 1. Dataset Overview",
    "",
    "This dataset contains curated data extracted from publicly available digital footprints of Kabwe Municipal Council, with a focus on Constituency Development Fund (CDF) activities.",
    "",
    "The dataset was prepared for the CSC4792 Data Mining and Warehousing practical assignment at the University of Zambia.",
    "",
    "The CDF datasets cover:",
    "",
    "- CDF project records",
    "- CDF budget allocations",
    "- CDF funding and disbursements",
    "- CDF financial receipts and payments",
    "- CDF output indicators",
    "- CDF project proposals for different years",
    "",
    "The datasets are maintained as separate CSV files because each dataset represents a different level of detail.",
    "",
    "## 2. Source",
    "",
    "The primary source is the official Kabwe Municipal Council website:",
    "",
    "https://www.kabwecouncil.gov.zm",
    "",
    "The source documents include official council budget documents, financial statements and CDF project documents published by the council.",
    "",
    "A complete list of sources is available in:",
    "",
    "sources/source_inventory.csv",
    "",
    "## 3. CDF Dataset Files",
    "",
    "### 3.1 CDF Projects 2024",
    "",
    "File: data/processed/db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    "",
    "Contains CDF community project records from Bwacha and Kabwe Central constituencies.",
    "",
    "Records: 79",
    "",
    "The dataset includes project names, descriptions, wards, project sites, financial estimates, approved amounts, contract amounts and project status where these values were available in the source documents.",
    "",
    "### 3.2 CDF Projects 2025",
    "",
    "File: data/processed/db-unza26-csc4792-kabwe_cdf_projects_2025.csv",
    "",
    "Contains proposed 2025 CDF projects for Kabwe Central Constituency.",
    "",
    "Records: 33",
    "",
    "The dataset includes project number, project description, ward, sector and comments from the source document.",
    "",
    "### 3.3 CDF Budget 2024-2025",
    "",
    "File: data/processed/db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",
    "",
    "Contains CDF budget allocations for 2024 and 2025.",
    "",
    "Budget categories include Constituency Development, Community Projects, Women and Youth Empowerment, CDF Administration, and Secondary School and Skills Development Bursaries.",
    "",
    "Records: 5",
    "",
    "### 3.4 CDF Disbursements 2025",
    "",
    "File: data/processed/db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv",
    "",
    "Contains constituency-level CDF funding recorded in the 2025 financial statement.",
    "",
    "The records cover Kabwe Central and Bwacha constituencies.",
    "",
    "Records: 2",
    "",
    "### 3.5 CDF Financial Statement 2025",
    "",
    "File: data/processed/db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    "",
    "Contains CDF receipts and payments extracted from the 2025 BI-Annual Financial Statements.",
    "",
    "The dataset includes CDF funding, loan repayments, interest earned, infrastructure development, rehabilitation works, asset acquisition, social benefits, bursaries, administration costs, disaster contingency and other payments.",
    "",
    "Records: 24",
    "",
    "Total CDF receipts: K13,094,717",
    "",
    "Total CDF payments: K34,462,069",
    "",
    "These totals were checked against the reported CDF financial statement totals.",
    "",
    "### 3.6 CDF Output Indicators 2025",
    "",
    "File: data/processed/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv",
    "",
    "Contains CDF output indicators and reported targets and actual values for 2023-2025.",
    "",
    "Examples include desks provided, maternity wings, ambulances, roads graded, youth groups, women groups, business entities receiving loans, CDFC meetings, project monitoring visits, projects branded, skills beneficiaries and secondary boarding beneficiaries.",
    "",
    "Records: 12",
    "",
    "## 4. Data Format",
    "",
    "All curated datasets are stored as CSV files.",
    "",
    "The pipe character (|) is used as the field delimiter as required by the assignment.",
    "",
    "Example:",
    "",
    "year|constituency|amount_kwacha",
    "2025|Kabwe Central|6205128",
    "",
    "When loading the files with pandas, use:",
    "",
    'pd.read_csv("file.csv", sep="|")',
    "",
    "## 5. Data Preparation",
    "",
    "The data preparation process followed these stages:",
    "",
    "1. Source identification",
    "2. Document retrieval",
    "3. Data extraction",
    "4. Manual verification where required",
    "5. Data cleaning",
    "6. Standardisation of fields",
    "7. Missing-value handling",
    "8. Duplicate checking",
    "9. Validation against source totals",
    "10. Export to pipe-delimited CSV files",
    "",
    "## 6. Missing Values",
    "",
    "Missing values from the original source documents were not automatically replaced with zero.",
    "",
    "A blank value can mean that the source document did not provide a value. Therefore, missing values are preserved where the source did not report a value.",
    "",
    "Some 2024 project financial and status fields were blank in the original source document.",
    "",
    "## 7. Financial Values",
    "",
    "Financial amounts are represented in Zambian Kwacha (ZMW).",
    "",
    "Numeric financial columns are stored as numeric values without currency symbols or thousands separators to make them suitable for data analysis.",
    "",
    "For example, 6205128 represents K6,205,128.",
    "",
    "## 8. Data Quality Checks",
    "",
    "The curated datasets were checked for:",
    "",
    "- Correct CSV structure",
    "- Pipe delimiter usage",
    "- Duplicate records",
    "- Missing values",
    "- Numeric financial fields",
    "- Expected record counts",
    "- Financial statement totals",
    "- Consistency with source documents",
    "",
    "## 9. Dataset Grain",
    "",
    "The CDF datasets should not be blindly merged into one table because they represent different types of records.",
    "",
    "Project datasets have one record per project.",
    "",
    "Budget data has one record per budget category.",
    "",
    "Disbursement data has one record per constituency funding record.",
    "",
    "Financial statement data has one record per financial transaction category and constituency.",
    "",
    "Output indicator data has one record per output indicator.",
    "",
    "Keeping these datasets separate reduces duplication and preserves the meaning of each record.",
    "",
    "## 10. Data Dictionary",
    "",
    "A detailed description of every column is available in:",
    "",
    "docs/cdf_data_dictionary.csv",
    "",
    "## 11. Reproducibility",
    "",
    "The project contains scripts and a Jupyter Notebook used during the extraction, cleaning and validation process.",
    "",
    "Project structure:",
    "",
    "CSC4792-Kabwe-Dataset/",
    "├── data/",
    "│   ├── raw/",
    "│   └── processed/",
    "├── notebooks/",
    "│   └── kabwe_council_dataset.ipynb",
    "├── scripts/",
    "│   ├── scrape.py",
    "│   ├── extract.py",
    "│   └── clean.py",
    "├── sources/",
    "│   └── source_inventory.csv",
    "├── docs/",
    "│   ├── cdf_data_dictionary.csv",
    "│   └── CDF_README.md",
    "├── requirements.txt",
    "└── README.md",
    "",
    "## 12. Intended Use",
    "",
    "The curated CDF datasets can be used for exploratory data analysis, budget analysis, project distribution analysis, sector analysis, constituency comparisons, financial expenditure analysis, project status analysis, data visualisation, classification, clustering and data warehouse modelling.",
    "",
    "The datasets should be interpreted together with their original source documents.",
    "",
    "## 13. Source Attribution",
    "",
    "Data was extracted from publicly available Kabwe Municipal Council documents.",
    "",
    "The original source URLs and document names are retained in the dataset metadata columns and in the source inventory.",
]

readme_path = Path("../docs/CDF_README.md")

readme_path.write_text(
    "\n".join(readme_lines),
    encoding="utf-8"
)

print("CDF README created successfully.")
print("File:", readme_path)
print("Size:", readme_path.stat().st_size, "bytes")

CDF README created successfully.
File: ..\docs\CDF_README.md
Size: 7044 bytes


In [18]:
from pathlib import Path

readme_path = Path("../docs/CDF_README.md")

print("Exists:", readme_path.exists())
print("Size:", readme_path.stat().st_size, "bytes")

Exists: True
Size: 7044 bytes


In [19]:
from pathlib import Path

extracted_path = Path("../data/extracted")
extracted_path.mkdir(exist_ok=True)

print("Extraction folder ready:", extracted_path)

Extraction folder ready: ..\data\extracted


In [20]:
from pathlib import Path
import pandas as pd

processed_dir = Path("../data/processed")

cdf_files = {
    "CDF Budget 2024-2025":
        "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",

    "CDF Disbursements 2025":
        "db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv",

    "CDF Financial Statement 2025":
        "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",

    "CDF Output Indicators 2025":
        "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv",

    "CDF Projects 2024":
        "db-unza26-csc4792-kabwe_cdf_projects_2024.csv",

    "CDF Projects 2025":
        "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
}


print("CDF DATA QUALITY CONTROL")
print("=" * 70)

results = []


for dataset_name, filename in cdf_files.items():

    path = processed_dir / filename

    print(f"\nChecking: {filename}")
    print("-" * 70)

    if not path.exists():
        print("❌ FILE NOT FOUND")

        results.append({
            "dataset": dataset_name,
            "file": filename,
            "exists": False,
            "rows": None,
            "columns": None,
            "duplicates": None,
            "missing_values": None
        })

        continue

    df = pd.read_csv(path, sep="|")

    duplicate_count = df.duplicated().sum()
    missing_count = df.isna().sum().sum()

    print("✓ File exists")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicates:", duplicate_count)
    print("Missing values:", missing_count)

    print("Columns:")
    print(df.columns.tolist())

    results.append({
        "dataset": dataset_name,
        "file": filename,
        "exists": True,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicates": duplicate_count,
        "missing_values": missing_count
    })


qc_results = pd.DataFrame(results)

print("\n")
print("=" * 70)
print("QUALITY CONTROL SUMMARY")
print("=" * 70)

display(qc_results)

CDF DATA QUALITY CONTROL

Checking: db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv
----------------------------------------------------------------------
✓ File exists
Rows: 5
Columns: 7
Duplicates: 0
Missing values: 0
Columns:
['council', 'fund', 'budget_category', 'budget_2024', 'budget_2025', 'source_url', 'source_document']

Checking: db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv
----------------------------------------------------------------------
✓ File exists
Rows: 2
Columns: 7
Duplicates: 0
Missing values: 0
Columns:
['year', 'constituency', 'funding_type', 'amount_kwacha', 'source_note', 'source_url', 'source_document']

Checking: db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv
----------------------------------------------------------------------
✓ File exists
Rows: 24
Columns: 8
Duplicates: 0
Missing values: 0
Columns:
['year', 'constituency', 'transaction_type', 'financial_category', 'amount_kwacha', 'source_note', 'source_url', 'source_document']

Checking

,dataset,file,exists,rows,columns,duplicates,missing_values
0,CDF Budget 2024-2025,db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv,True,5,7,0,0
1,CDF Disbursements 2025,db-unza26-csc4792-kabwe_cdf_disbursements_2025...,True,2,7,0,0
2,CDF Financial Statement 2025,db-unza26-csc4792-kabwe_cdf_financial_statemen...,True,24,8,0,0
3,CDF Output Indicators 2025,db-unza26-csc4792-kabwe_cdf_output_indicators_...,True,12,10,0,4
4,CDF Projects 2024,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,True,79,15,0,432
5,CDF Projects 2025,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,True,33,9,0,0


In [21]:
expected_rows = {
    "CDF Budget 2024-2025": 5,
    "CDF Disbursements 2025": 2,
    "CDF Financial Statement 2025": 24,
    "CDF Output Indicators 2025": 12,
    "CDF Projects 2024": 79,
    "CDF Projects 2025": 33
}

print("ROW COUNT VALIDATION")
print("=" * 50)

for dataset, expected in expected_rows.items():

    actual = qc_results.loc[
        qc_results["dataset"] == dataset,
        "rows"
    ].iloc[0]

    if actual == expected:
        print(f"✓ {dataset}: {actual} rows")
    else:
        print(
            f"❌ {dataset}: expected {expected}, "
            f"found {actual}"
        )

ROW COUNT VALIDATION
✓ CDF Budget 2024-2025: 5 rows
✓ CDF Disbursements 2025: 2 rows
✓ CDF Financial Statement 2025: 24 rows
✓ CDF Output Indicators 2025: 12 rows
✓ CDF Projects 2024: 79 rows
✓ CDF Projects 2025: 33 rows


In [22]:
print("PIPE DELIMITER CHECK")
print("=" * 50)

for dataset_name, filename in cdf_files.items():

    path = processed_dir / filename

    if not path.exists():
        print(f"❌ {dataset_name}: file missing")
        continue

    with open(path, "r", encoding="utf-8") as file:
        first_line = file.readline().strip()

    if "|" in first_line:
        print(f"✓ {dataset_name}: pipe delimiter detected")
    else:
        print(f"❌ {dataset_name}: pipe delimiter NOT detected")

PIPE DELIMITER CHECK
✓ CDF Budget 2024-2025: pipe delimiter detected
✓ CDF Disbursements 2025: pipe delimiter detected
✓ CDF Financial Statement 2025: pipe delimiter detected
✓ CDF Output Indicators 2025: pipe delimiter detected
✓ CDF Projects 2024: pipe delimiter detected
✓ CDF Projects 2025: pipe delimiter detected


In [26]:
import pandas as pd

financial_df = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    sep="|"
)

print("Rows:", len(financial_df))
print("Columns:", financial_df.columns.tolist())

display(financial_df.head())

Rows: 24
Columns: ['year', 'constituency', 'transaction_type', 'financial_category', 'amount_kwacha', 'source_note', 'source_url', 'source_document']


,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


In [27]:
import pandas as pd

financial_df = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    sep="|"
)

print("Rows:", len(financial_df))
print("Columns:", financial_df.columns.tolist())

display(financial_df.head())

Rows: 24
Columns: ['year', 'constituency', 'transaction_type', 'financial_category', 'amount_kwacha', 'source_note', 'source_url', 'source_document']


,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


In [28]:
payments = financial_df[
    financial_df["transaction_type"].str.lower() == "payment"
]

print("Payment rows:")
display(
    payments[
        ["constituency", "financial_category", "amount_kwacha"]
    ]
)

payments_total = payments["amount_kwacha"].sum()

print(f"\nCalculated payment total: K{payments_total:,.2f}")
print("Expected payment total:   K34,462,069.00")

Payment rows:


,constituency,financial_category,amount_kwacha
6,Kabwe Central,Infrastructure Development,3703681
7,Bwacha,Infrastructure Development,2649421
8,Kabwe Central,Rehabilitation Works,557547
9,Bwacha,Rehabilitation Works,0
10,Kabwe Central,Asset Acquisition,5244633
11,Bwacha,Asset Acquisition,5208500
12,All Constituencies,Rural Electrification,0
13,Kabwe Central,Social Benefits - Grants,2320000
14,Bwacha,Social Benefits - Grants,2176650
15,All Constituencies,Loans,0



Calculated payment total: K34,462,070.00
Expected payment total:   K34,462,069.00


In [29]:
# Document the K1 source discrepancy

expected_payments = 34_462_069
calculated_payments = payments_total

difference = calculated_payments - expected_payments

print(f"Reported payment total:    K{expected_payments:,.0f}")
print(f"Calculated payment total:  K{calculated_payments:,.0f}")
print(f"Difference:                K{difference:,.0f}")

if difference == 1:
    print("\n⚠ Source discrepancy identified:")
    print(
        "The Disaster Contingency constituency amounts sum to K295,252, "
        "while the statement reports K295,251."
    )
    print("The source values are preserved without adjustment.")
else:
    print("\n✓ No payment discrepancy found.")

Reported payment total:    K34,462,069
Calculated payment total:  K34,462,070
Difference:                K1

⚠ Source discrepancy identified:
The Disaster Contingency constituency amounts sum to K295,252, while the statement reports K295,251.
The source values are preserved without adjustment.


In [30]:
# Check 2025 CDF projects by sector

projects_2025 = pd.read_csv(
    "../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2025.csv",
    sep="|"
)

sector_counts = projects_2025["sector"].value_counts()

print("2025 CDF Projects by Sector:")
display(sector_counts)

print(f"\nTotal projects: {len(projects_2025)}")

2025 CDF Projects by Sector:


sector
Education               16
Health                   8
Commerce and Trade       4
Water and Sanitation     3
Transport                2
Name: count, dtype: int64


Total projects: 33


In [31]:
# Verify that all CDF datasets use the pipe (|) delimiter

from pathlib import Path

processed_dir = Path("../data/processed")

cdf_files = [
    "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
]

for filename in cdf_files:
    file_path = processed_dir / filename

    with open(file_path, "r", encoding="utf-8") as file:
        first_line = file.readline().strip()

    pipe_count = first_line.count("|")
    comma_count = first_line.count(",")

    print(f"\n{filename}")
    print(f"  Pipe separators: {pipe_count}")
    print(f"  Comma separators: {comma_count}")

    assert pipe_count > 0, f"Pipe delimiter missing in {filename}"

print("\n✓ All CDF datasets use the required pipe (|) delimiter.")


db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv
  Pipe separators: 6
  Comma separators: 0

db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv
  Pipe separators: 6
  Comma separators: 0

db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv
  Pipe separators: 7
  Comma separators: 0

db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv
  Pipe separators: 9
  Comma separators: 0

db-unza26-csc4792-kabwe_cdf_projects_2024.csv
  Pipe separators: 14
  Comma separators: 0

db-unza26-csc4792-kabwe_cdf_projects_2025.csv
  Pipe separators: 8
  Comma separators: 0

✓ All CDF datasets use the required pipe (|) delimiter.


In [32]:
# Verify expected row counts for all CDF datasets

expected_rows = {
    "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv": 5,
    "db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv": 2,
    "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv": 24,
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv": 12,
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv": 79,
    "db-unza26-csc4792-kabwe_cdf_projects_2025.csv": 33
}

for filename, expected in expected_rows.items():
    file_path = processed_dir / filename
    df = pd.read_csv(file_path, sep="|")
    
    actual = len(df)
    
    print(f"{filename}: {actual} rows (expected {expected})")
    
    assert actual == expected, (
        f"Row count mismatch in {filename}: "
        f"expected {expected}, got {actual}"
    )

print("\n✓ All dataset row counts are correct.")

db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv: 5 rows (expected 5)
db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv: 2 rows (expected 2)
db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv: 24 rows (expected 24)
db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv: 12 rows (expected 12)
db-unza26-csc4792-kabwe_cdf_projects_2024.csv: 79 rows (expected 79)
db-unza26-csc4792-kabwe_cdf_projects_2025.csv: 33 rows (expected 33)

✓ All dataset row counts are correct.


In [33]:
# Check for duplicate rows in all CDF datasets

for filename in cdf_files:
    file_path = processed_dir / filename
    df = pd.read_csv(file_path, sep="|")
    
    duplicate_count = df.duplicated().sum()
    
    print(f"{filename}: {duplicate_count} duplicate rows")
    
    assert duplicate_count == 0, (
        f"Duplicates found in {filename}: {duplicate_count}"
    )

print("\n✓ No duplicate records found in any CDF dataset.")

db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv: 0 duplicate rows
db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv: 0 duplicate rows
db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv: 0 duplicate rows
db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv: 0 duplicate rows
db-unza26-csc4792-kabwe_cdf_projects_2024.csv: 0 duplicate rows
db-unza26-csc4792-kabwe_cdf_projects_2025.csv: 0 duplicate rows

✓ No duplicate records found in any CDF dataset.
